In [ ]:
!pip install -q timm grad-cam albumentations thop scikit-learn squarify

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║ CELL 2 — Imports & Global Setup                   ║
# ╚══════════════════════════════════════════════════════╝


# ── Standard Library ──────────────────────────────────
import os, random, warnings, gc, time, datetime
import glob, math, json, hashlib
import multiprocessing as mp
from functools import partial

# ── Data & Visualisasi ────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from matplotlib import rcParams
from matplotlib.ticker import MaxNLocator
import seaborn as sns
import cv2
import ctypes
from PIL import Image
from tqdm import tqdm

# ── Excel Export ──────────────────────────────────────
from openpyxl import Workbook
from openpyxl.styles import (Font, PatternFill, Alignment,
                              Border, Side, GradientFill)
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule

# ── Metrics & Sklearn ─────────────────────────────────
from sklearn.metrics import (
    classification_report, confusion_matrix,
    cohen_kappa_score, f1_score,
    roc_curve, auc, precision_recall_curve,
    average_precision_score
)
from sklearn.preprocessing import label_binarize

# ── PyTorch ───────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import OneCycleLR

# ── Model & Augmentasi ────────────────────────────────
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── GradCAM ───────────────────────────────────────────
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# ── Profiling ─────────────────────────────────────────
from thop import profile, clever_format
from IPython.display import display, HTML

try:
    import squarify
    HAS_SQUARIFY = True
except ImportError:
    HAS_SQUARIFY = False

# ── Global Config ─────────────────────────────────────
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "font.size": 10,
                     "axes.titlesize": 12, "axes.labelsize": 10})

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]
PALETTE     = ["#2196F3", "#4CAF50", "#FF9800", "#E91E63", "#9C27B0"]
OUTPUT_DIR  = "/kaggle/working/"

# ── Seed ──────────────────────────────────────────────
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark     = True
    os.environ["PYTHONHASHSEED"]       = str(seed)

set_seed(42)

# ── Logger ────────────────────────────────────────────
class Logger:
    BOLD    = "\033[1m";  RESET   = "\033[0m"
    GREEN   = "\033[92m"; CYAN    = "\033[96m"
    YELLOW  = "\033[93m"; RED     = "\033[91m"
    MAGENTA = "\033[95m"

    def __init__(self, run_name=""):
        self.run_name = run_name
        self._t0      = time.time()

    def _ts(self):
        return now_wib().strftime("%H:%M:%S")

    def _elapsed(self):
        s = int(time.time() - self._t0)
        return f"{s//60:02d}m{s%60:02d}s"

    def section(self, title):
        print(f"\n{self.BOLD}{self.CYAN}{'═'*65}{self.RESET}")
        print(f"{self.BOLD}{self.CYAN}  [{self._ts()}] {title}{self.RESET}")
        print(f"{self.BOLD}{self.CYAN}{'═'*65}{self.RESET}")

    def info(self, msg):
        print(f"{self.GREEN}  ✅ [{self._ts()}] {msg}{self.RESET}")

    def metric(self, epoch, total, tr_loss, va_loss, tr_acc, va_acc,
                tr_f1, va_f1, tr_qwk, va_qwk, lr, tag="", epoch_sec=0.0):
        bar = "█" * int(epoch / total * 20)
        print(f"  Ep {epoch:02d}/{total} [{bar:<20}] "
              f"Loss {tr_loss:.4f}/{va_loss:.4f} "
              f"Acc {tr_acc:.4f}/{va_acc:.4f} "
              f"F1 {tr_f1:.4f}/{va_f1:.4f} "
              f"QWK {tr_qwk:.4f}/{va_qwk:.4f} "
              f"LR {lr:.2e} ⏱{epoch_sec:.1f}s {tag}")

    def best(self, epoch, metric_name, val, extra=""):
        extra_str = f" ({extra})" if extra else ""
        print(f"{self.YELLOW}  🏆 [{self._ts()}] New Best! Epoch {epoch} → {metric_name}={val:.4f}{extra_str}{self.RESET}")

    def warn(self, msg):
        print(f"{self.RED}  ⚠️  [{self._ts()}] {msg}{self.RESET}")

    def done(self, msg):
        print(f"{self.MAGENTA}  🎉 [{self._ts()}] DONE in {self._elapsed()} — {msg}{self.RESET}")

def now_wib():
    return datetime.datetime.utcnow() + datetime.timedelta(hours=7)

LOG = Logger("global")

# ── Info ──────────────────────────────────────────────
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i} : {torch.cuda.get_device_name(i)}")
print(f"Device  : {DEVICE}")
print(f"squarify: {'✅' if HAS_SQUARIFY else '⚠️ tidak tersedia'}")
LOG.info("Semua import berhasil → Lanjut ke Cell 3")

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║ CELL 3 — Konfigurasi Eksperimen (Multi-Model)        ║
# ╚══════════════════════════════════════════════════════╝

BASE = "/kaggle/input/datasets/mariaherrerot/aptos2019"

PATHS = {
    "train_csv": f"{BASE}/train_1.csv",
    "train_dir": f"{BASE}/train_images/train_images",

    "val_csv": f"{BASE}/valid.csv",
    "val_dir": f"{BASE}/val_images/val_images",

    "test_csv": f"{BASE}/test.csv",
    "test_dir": f"{BASE}/test_images/test_images",
}


# ──────────────────────────────────────────────────────
# Dict per-model:
# "model_key": (timm_model_name, img_size, batch_size)
# ──────────────────────────────────────────────────────

MODEL_CONFIGS = {
    "mobilenetv3_small_050": ("mobilenetv3_small_050", 300, 64),
    "lcnet_050": ("lcnet_050", 300, 64),
    "mobilenetv2_050": ("mobilenetv2_050", 300, 64),
    "mobilevit_xxs": ("mobilevit_xxs", 300, 64),
    "mobilenetv3_small_075": ("mobilenetv3_small_075", 300, 64),
    "lcnet_075": ("lcnet_075", 300, 64),
    "lcnet_100": ("lcnet_100", 300, 64),
    "regnetx_002": ("regnetx_002", 300, 64),
    "mixnet_s": ("mixnet_s", 300, 64),
    "semnasnet_100": ("semnasnet_100", 300, 64),
    "regnety_002": ("regnety_002", 300, 64),
    "xcit_nano_12_p8_224": ("xcit_nano_12_p8_224", 224, 64),
    "xcit_nano_12_p8_384": ("xcit_nano_12_p8_384", 384, 32),
    "xcit_nano_12_p16_224": ("xcit_nano_12_p16_224", 224, 64),
    "xcit_nano_12_p16_384": ("xcit_nano_12_p16_384", 384, 32),
    "mnasnet_100": ("mnasnet_100", 300, 64),
    "fastvit_t8": ("fastvit_t8", 300, 64),
    "efficientnet_lite0": ("efficientnet_lite0", 300, 64),
    "convnextv2_atto": ("convnextv2_atto", 300, 64),
    "mixnet_m": ("mixnet_m", 300, 64),
    "mobileone_s1": ("mobileone_s1", 300, 64),
    "regnety_004": ("regnety_004", 300, 64),
    "tf_efficientnet_lite1": ("tf_efficientnet_lite1", 300, 64),
    "efficientnet_es": ("efficientnet_es", 300, 64),
    "mobileone_s0": ("mobileone_s0", 300, 64),
    "pit_ti_distilled_224": ("pit_ti_distilled_224", 224, 64),
    "regnetx_004": ("regnetx_004", 300, 64),
    "convnextv2_femto": ("convnextv2_femto", 300, 32),
    "ghostnetv2_100": ("ghostnetv2_100", 300, 64),
    "resnet10t": ("resnet10t", 300, 64),
    "mobilevit_s": ("mobilevit_s", 300, 64),
    # ── MobileNet ──────────────────────────────────────
    "mobilenetv2_100": ("mobilenetv2_100", 300, 64),
    "mobilenetv2": ("mobilenetv2_100", 300, 64),
    "mobilenetv3-s": ("mobilenetv3_small_100", 300, 64),
    "mobilenetv3-small": ("mobilenetv3_small_100", 300, 64),
    "mobilenetv3_small_100": ("mobilenetv3_small_100", 300, 64),
    "mobilenetv3-l": ("mobilenetv3_large_100", 300, 64),
    "mobilenetv3-large": ("mobilenetv3_large_100", 300, 64),
    "mobilenetv3_large_100": ("mobilenetv3_large_100", 300, 64),

    # ── Inception Family ──────────────────────────────
    "inception_v3": ("inception_v3", 300, 32),
    "inception-v3": ("inception_v3", 300, 32),
    "inception_resnet_v2": ("inception_resnet_v2", 300, 16),
    "inception-resnet-v2": ("inception_resnet_v2", 300, 16),
    "inception resnetv2": ("inception_resnet_v2", 300, 16),
    "xception": ("xception", 300, 32),

    # ── DenseNet / ResNet / VGG ──────────────────────
    "densenet121": ("densenet121", 300, 32),
    "densenet169": ("densenet169", 300, 32),
    "densenet-169": ("densenet169", 300, 32),
    "resnet50": ("resnet50", 300, 32),
    "vgg16": ("vgg16", 300, 32),
    "vgg19": ("vgg19", 300, 32),

    # ── EfficientNet / EfficientNetV2 ─────────────────
    "efficientnet_b0": ("efficientnet_b0", 300, 64),
    "efficientnet_b1": ("efficientnet_b1", 300, 32),
    "efficientnet_b3": ("efficientnet_b3", 300, 32),
    "tf_efficientnetv2_s": ("tf_efficientnetv2_s", 300, 32),
    "tf_efficientnetv2_m": ("tf_efficientnetv2_m", 300, 16),
    "tf_efficientnetv2_l": ("tf_efficientnetv2_l", 300, 8),
    "efficientnetv2_s": ("tf_efficientnetv2_s", 300, 32),
    "efficientnetv2_m": ("tf_efficientnetv2_m", 300, 16),
    "efficientnetv2_l": ("tf_efficientnetv2_l", 300, 8),
    "efficientnetv2-s": ("tf_efficientnetv2_s", 300, 32),
    "efficientnetv2-m": ("tf_efficientnetv2_m", 300, 16),
    "efficientnetv2-l": ("tf_efficientnetv2_l", 300, 8),

    # ── RegNet ────────────────────────────────────────
    "regnetx_040": ("regnetx_040", 300, 32),
    "regnetx-50": ("regnetx_040", 300, 32),

    # ── Swin Transformer ──────────────────────────────
    "swin_tiny": ("swin_tiny_patch4_window7_224", 224, 64),
    "swin-tiny": ("swin_tiny_patch4_window7_224", 224, 64),

    # ── Vision Transformers & Modern CNNs ─────────────
    "maxvit_nano_rw_256": ("maxvit_nano_rw_256", 256, 64),
    "repvit_m1": ("repvit_m1", 300, 32),
    "edgenext_small": ("edgenext_small", 384, 64),
    "edgenext_x_small": ("edgenext_x_small", 384, 64),
    "edgenext-xs": ("edgenext_x_small", 384, 64),
    "edgenext-xxs": ("edgenext_xx_small", 384, 64),
    "edgenext_xx_small": ("edgenext_xx_small", 384, 64),

    # ── Tambahan Ringan (≤ 6M Parameter) ──────────────
    "convnext_atto": ("convnext_atto", 300, 64),
    "convnext-atto": ("convnext_atto", 300, 64),
    "convnext_femto": ("convnext_femto", 300, 32),
    "convnext-femto": ("convnext_femto", 300, 32),
    "ghostnet_100": ("ghostnet_100", 300, 64),
    "ghostnet": ("ghostnet_100", 300, 64),
    "mobilevitv2_100": ("mobilevitv2_100", 300, 32),
    "mobilevitv2": ("mobilevitv2_100", 300, 32),
    "mobilevitv2_075": ("mobilevitv2_075", 300, 64),
    "mobilevitv2-075": ("mobilevitv2_075", 300, 64),
    "efficientvit_b0": ("efficientvit_b0", 300, 64),
    "efficientvit-b0": ("efficientvit_b0", 300, 64),
    "mobilevit_xs": ("mobilevit_xs", 300, 64),
    "mobilevit-xs": ("mobilevit_xs", 300, 64),
}

# ──────────────────────────────────────────────────────
# Model yang biasanya tidak support drop_path_rate
# ──────────────────────────────────────────────────────
MODELS_SKIP_DROP_PATH = {
    # CNN klasik
    "vgg16",
    "vgg19",

    # Residual klasik
    "resnet50",

    # DenseNet
    "densenet121",
    "densenet169",
    "densenet-169",

    # MobileNet & ShuffleNet
    "mobilenetv2_100",
    "mobilenetv2",
    "mobilenetv3_small_100",
    "mobilenetv3-small",
    "mobilenetv3-s",
    "mobilenetv3_large_100",
    "mobilenetv3-large",
    "mobilenetv3-l",
    "shufflenetv2",
    "ghostnet_100",
    "ghostnet",

    # Inception family
    "inception_v3",
    "inception-v3",
    "inception_resnet_v2",
    "inception-resnet-v2",
    "inception resnetv2",

    # Xception
    "xception",

    # RegNet
    "regnetx_040",
    "regnetx-50",

    # MobileViT (Tidak support drop_path_rate di timm)
    "mobilevitv2_100",
    "mobilevitv2",
    "mobilevitv2_075",
    "mobilevitv2-075",
    "mobilevit_xs",
    "mobilevit-xs",
}


# ╔══════════════════════════════════════════════════════╗
# ║ CONFIG FACTORY                                      ║
# ╚══════════════════════════════════════════════════════╝

def make_cfg(
    model_key="convnextv2_atto",
    use_clahe=True,
    use_aug=True,
    epochs=50,
):

    # ── Validasi model_key ────────────────────────────
    assert model_key in MODEL_CONFIGS, (
        f"model_key '{model_key}' tidak ditemukan!\n"
        f"Pilihan:\n{list(MODEL_CONFIGS.keys())}"
    )

    model_name, img_size, batch_size = MODEL_CONFIGS[model_key]

    # ── Naming tag ────────────────────────────────────
    aug_tag = "AUG" if use_aug else "noAUG"

    clahe_tag = (
        "CLAHE" if use_clahe else "noCLAHE"
    )

    # ── Short model naming ────────────────────────────
    short_name = (
        model_key
        .replace("tf_efficientnetv2_", "V2")
        .replace("efficientnetv2_", "V2")
        .replace("efficientnetv2-", "V2")
        .replace("efficientnet_", "EN")
        .replace("mobilenetv3_small_100", "MNV3S")
        .replace("mobilenetv3-small", "MNV3S")
        .replace("mobilenetv3-s", "MNV3S")
        .replace("mobilenetv3_large_100", "MNV3L")
        .replace("mobilenetv3-large", "MNV3L")
        .replace("mobilenetv3-l", "MNV3L")
        .replace("mobilenetv2_100", "MNV2")
        .replace("mobilenetv2", "MNV2")
        .replace("inception_resnet_v2", "IRV2")
        .replace("inception-resnet-v2", "IRV2")
        .replace("inception resnetv2", "IRV2")
        .replace("inception_v3", "IV3")
        .replace("inception-v3", "IV3")
        .replace("densenet121", "DN121")
        .replace("densenet169", "DN169")
        .replace("densenet-169", "DN169")
        .replace("resnet50", "RN50")
        .replace("xception", "XCP")
        .replace("maxvit_nano_rw_256", "maxvitN")
        .replace("repvit_m1", "repvitM1")
        .replace("regnetx_040", "RegX50")
        .replace("regnetx-50", "RegX50")
        .replace("swin_tiny_patch4_window7_224", "SwinT")
        .replace("swin_tiny", "SwinT")
        .replace("swin-tiny", "SwinT")
        .replace("convnext_atto", "CNXAtto")
        .replace("convnext-atto", "CNXAtto")
        .replace("convnext_femto", "CNXFemto")
        .replace("convnext-femto", "CNXFemto")
        .replace("ghostnet_100", "Ghost")
        .replace("ghostnet", "Ghost")
        .replace("shufflenetv2", "Shuffle")
        .replace("mobilevitv2_100", "MVITV2")
        .replace("mobilevitv2", "MVITV2")
        .replace("mobilevitv2_075", "MVITV2_075")
        .replace("mobilevitv2-075", "MVITV2_075")
        .replace("efficientvit_b0", "EVITB0")
        .replace("efficientvit-b0", "EVITB0")
        .replace("mobilevit_xs", "MVITXS")
        .replace("mobilevit-xs", "MVITXS")
        .upper()
    )

    return {

        # ── Dataset paths ─────────────────────────────
        **PATHS,

        # ── Model identity ────────────────────────────
        "model_name": model_name,
        "model_key": model_key,

        "run_name":
            f"{short_name}_{clahe_tag}_{aug_tag}",

        # ── Experiment flags ──────────────────────────
        "use_clahe": use_clahe,
        "use_aug": use_aug,

        # ── EMA & Multi-Sample Dropout ───────────────
        "use_ema": True,
        "ema_decay": 0.99,
        "use_msd": True,
        "use_msag": True,
        "msag_type": "full", # Pilihan: "full" (MSAG Biasa/Standard) atau "lite" (MSAG Lite)    # [FIX] Aktifkan MSAG (sebelumnya False)
        "use_crop": True,

        # ── Input / Dataset ───────────────────────────
        "img_size": img_size,
        "batch_size": batch_size,
        "num_classes": 5,
        "num_workers": 4,
        "cache_to_ram": True,

        # ── Training ──────────────────────────────────
        "epochs": epochs,
        "freeze_epochs": 3,
        "patience": 15,
        "seed": 42,

        # ── Resume Checkpoint ─────────────────────────
        "resume_path": None,

        # ── Optimizer ─────────────────────────────────
        "lr": 1e-4,
        "lr_min": 1e-6,
        "weight_decay": 3e-4,
        "backbone_lr_mult": 0.3,

        # ── Regularization ────────────────────────────
        "drop_rate": 0.3,
        "drop_path_rate": 0.15,
        "ordinal_weight": 0.15,       # MODIFIED (from 0.05)
        "grad_clip": 1.0,

        # ── Loss ──────────────────────────────────────
        "focal_gamma": 2.5,
        "loss_beta": 0.999,
        "label_smoothing": 0.1,       # MODIFIED (from 0.05)
        "focal_loss_weight": 0.2,     # MODIFIED (from 0.05)
        "ce_loss_weight": 0.8,        # MODIFIED (from 1.0)

        # ── CLAHE ─────────────────────────────────────
        "clahe_mode":      "combined",
        "clahe_clip":      2.0,
        "clahe_grid":      8,
        "clahe_alpha":     0.7,
        "clahe_green_alpha":  0.5,
        "clahe_green_output": "merge_rgb",
        "clahe_green_weight": 0.4,

        # ── Test-Time Augmentation (TTA) ──────────────
        "use_tta": True,             # MODIFIED (from True)
        "n_tta": 2,

        # ── GradCAM Parameters ────────────────────────
        "gradcam_grid_per_class":  2,
        "gradcam_comparison_samples": 6,
        "gradcam_mean_per_class":  8,
        "gradcam_misclassified_samples": 8,
        "gradcam_split":           "test",
        "gradcam_aug_smooth":      False,

        # ── Scheduler ─────────────────────────────────
        "max_lr": 2e-4,
        "pct_start": 0.2,
        "div_factor": 25.0,
        "final_div_factor": 1e4,

        "mixup_alpha": 0.2,           # MODIFIED (from 0.05)
        "steps_per_epoch": None,
    }

# ══════════════════════════════════════════════════════
# ▶▶ UBAH HANYA BARIS INI SAAT GANTI ARSITEKTUR ◀◀
# ══════════════════════════════════════════════════════

ACTIVE_MODEL = "convnextv2_atto"


# ╔══════════════════════════════════════════════════════╗
# ║ Preview Config                                       ║
# ╚══════════════════════════════════════════════════════╝

_t = make_cfg(model_key=ACTIVE_MODEL)

print(f"\n{'═'*60}")
print(f"  ACTIVE MODEL : {ACTIVE_MODEL}")
print(f"{'═'*60}")

for k in [
    "run_name",
    "model_name",
    "img_size",
    "batch_size",
    "use_clahe",
    "use_aug",
    "focal_gamma",
    "clahe_grid",
    "patience",
    "epochs",
    "cache_to_ram",
    "use_msag",
    "use_crop",
]:
    print(f"  {k:<20}: {_t[k]}")

print(f"{'═'*60}")

# ── Info drop_path support ────────────────────────────
if ACTIVE_MODEL in MODELS_SKIP_DROP_PATH:
    print("\n  drop_path_rate : SKIPPED")
else:
    print(f"\n  drop_path_rate : {_t['drop_path_rate']}")

del _t


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║ CELL 4 — CLAHE Preprocessing                                     ║
# ╚══════════════════════════════════════════════════════════════════╝

# ──────────────────────────────────────────────────────
# Circular Auto-Cropping
# ──────────────────────────────────────────────────────
def crop_image_from_gray(img, tol=7):
    """
    Memotong bingkai hitam otomatis pada gambar fundus retina.
    Dioptimasi menggunakan downsampling cepat untuk mendeteksi koordinat crop pada citra resolusi tinggi.
    """
    h, w = img.shape[:2]
    # Jika gambar kecil, langsung proses untuk menghindari overhead resize
    if h < 512 or w < 512:
        if img.ndim == 2:
            mask = img > tol
            rows = np.any(mask, axis=1)
            cols = np.any(mask, axis=0)
            if not np.any(rows) or not np.any(cols):
                return img
            ymin, ymax = np.where(rows)[0][[0, -1]]
            xmin, xmax = np.where(cols)[0][[0, -1]]
            return img[ymin:ymax+1, xmin:xmax+1]
        else:
            gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            mask = gray_img > tol
            rows = np.any(mask, axis=1)
            cols = np.any(mask, axis=0)
            if not np.any(rows) or not np.any(cols):
                return img
            ymin, ymax = np.where(rows)[0][[0, -1]]
            xmin, xmax = np.where(cols)[0][[0, -1]]
            return img[ymin:ymax+1, xmin:xmax+1]
            
    # Deteksi koordinat crop pada versi downsample (max 384px)
    max_dim = 384
    scale_y = h / max_dim
    scale_x = w / max_dim
    small_w = int(w / scale_x)
    small_h = int(h / scale_y)
    
    small_img = cv2.resize(img, (small_w, small_h), interpolation=cv2.INTER_AREA)
    
    if small_img.ndim == 2:
        gray_small = small_img
    else:
        gray_small = cv2.cvtColor(small_img, cv2.COLOR_BGR2GRAY)
        
    mask = gray_small > tol
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    if not np.any(rows) or not np.any(cols):
        return img
        
    ymin_s, ymax_s = np.where(rows)[0][[0, -1]]
    xmin_s, xmax_s = np.where(cols)[0][[0, -1]]
    
    # Proyeksikan kembali ke koordinat resolusi tinggi asli
    ymin = int(ymin_s * scale_y)
    ymax = min(h - 1, int((ymax_s + 1) * scale_y))
    xmin = int(xmin_s * scale_x)
    xmax = min(w - 1, int((xmax_s + 1) * scale_x))
    
    return img[ymin:ymax+1, xmin:xmax+1]

# ──────────────────────────────────────────────────────
# CLAHE LAB (BGR Optimized)
# ──────────────────────────────────────────────────────
def apply_clahe_bgr(img_bgr, clip_limit=2.0, tile_grid=8, alpha=0.7):
    """
    CLAHE di ruang warna LAB — channel L (luminance) dengan output BGR.
    """
    lab     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe   = cv2.createCLAHE(clipLimit=clip_limit,
                               tileGridSize=(tile_grid, tile_grid))
    l_clahe = clahe.apply(l)

    # Blending: alpha% CLAHE + (1-alpha)% original
    l_blend = cv2.addWeighted(l_clahe, alpha, l, 1 - alpha, 0)

    lab_out = cv2.merge([l_blend, a, b])
    img_bgr_out = cv2.cvtColor(lab_out, cv2.COLOR_LAB2BGR)
    return img_bgr_out


def apply_clahe(img_bgr, clip_limit=2.0, tile_grid=8, alpha=0.7):
    """
    CLAHE di ruang warna LAB — channel L (luminance) dengan output RGB.
    """
    return cv2.cvtColor(apply_clahe_bgr(img_bgr, clip_limit, tile_grid, alpha), cv2.COLOR_BGR2RGB)


# ──────────────────────────────────────────────────────
# CLAHE Green Channel (BGR Optimized)
# ──────────────────────────────────────────────────────
def apply_clahe_green_bgr(img_bgr, clip_limit=2.0, tile_grid=8, alpha=0.5,
                          mode="merge_rgb"):
    """
    CLAHE khusus Green Channel dengan output BGR.
    """
    # Green channel di BGR adalah index 1
    b, g, r = cv2.split(img_bgr)

    # CLAHE di green channel
    clahe   = cv2.createCLAHE(clipLimit=clip_limit,
                               tileGridSize=(tile_grid, tile_grid))
    g_clahe = clahe.apply(g)

    # Blending green asli + green CLAHE
    g_blend = cv2.addWeighted(g_clahe, alpha, g, 1 - alpha, 0)

    if mode == "merge_rgb":
        img_out = cv2.merge([b, g_blend, r])
    elif mode == "grayscale":
        img_out = g_blend
    elif mode == "stack_rgb":
        # Stack green ke 3 channel (B=G=R)
        img_out = cv2.merge([g_blend, g_blend, g_blend])
    else:
        raise ValueError(f"mode tidak dikenal: '{mode}'. Pilih: merge_rgb / grayscale / stack_rgb")

    return img_out


def apply_clahe_green(img_bgr, clip_limit=2.0, tile_grid=8, alpha=0.5,
                      mode="merge_rgb"):
    """
    CLAHE khusus Green Channel dengan output RGB.
    """
    img_bgr_out = apply_clahe_green_bgr(img_bgr, clip_limit, tile_grid, alpha, mode)
    if mode == "grayscale":
        return img_bgr_out
    return cv2.cvtColor(img_bgr_out, cv2.COLOR_BGR2RGB)


# ──────────────────────────────────────────────────────
# CLAHE Combined (LAB + Green - BGR Optimized)
# ──────────────────────────────────────────────────────
def apply_clahe_combined_bgr(img_bgr, clip_limit=2.0, tile_grid=8,
                             alpha_lab=0.7, alpha_green=0.5,
                             green_weight=0.4):
    """
    Gabungan CLAHE LAB + CLAHE Green Channel (Optimasi BGR murni).
    Sangat cepat karena menghindari duplikasi konversi warna dan alokasi float32.
    """
    # Split BGR dari gambar original
    b_orig, g_orig, r_orig = cv2.split(img_bgr)

    # 1. Green Channel CLAHE (langsung di-apply ke g_orig)
    clahe_g = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_grid, tile_grid))
    g_clahe = clahe_g.apply(g_orig)
    g_green = cv2.addWeighted(g_clahe, alpha_green, g_orig, 1 - alpha_green, 0)

    # 2. LAB CLAHE
    lab     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe_l = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_grid, tile_grid))
    l_clahe = clahe_l.apply(l)
    l_blend = cv2.addWeighted(l_clahe, alpha_lab, l, 1 - alpha_lab, 0)
    lab_out = cv2.merge([l_blend, a, b])
    
    result_lab_bgr = cv2.cvtColor(lab_out, cv2.COLOR_LAB2BGR)
    b_lab, g_lab, r_lab = cv2.split(result_lab_bgr)

    # 3. Blending cepat tiap channel dengan cv2.addWeighted (uint8 murni)
    b_combined = cv2.addWeighted(b_lab, 1.0 - green_weight, b_orig, green_weight, 0)
    g_combined = cv2.addWeighted(g_lab, 1.0 - green_weight, g_green, green_weight, 0)
    r_combined = cv2.addWeighted(r_lab, 1.0 - green_weight, r_orig, green_weight, 0)

    return cv2.merge([b_combined, g_combined, r_combined])


def apply_clahe_combined(img_bgr, clip_limit=2.0, tile_grid=8,
                         alpha_lab=0.7, alpha_green=0.5,
                         green_weight=0.4):
    """
    Gabungan CLAHE LAB + CLAHE Green Channel dengan output RGB.
    """
    combined_bgr = apply_clahe_combined_bgr(img_bgr, clip_limit, tile_grid,
                                             alpha_lab, alpha_green, green_weight)
    return cv2.cvtColor(combined_bgr, cv2.COLOR_BGR2RGB)


# ──────────────────────────────────────────────────────
# Load Image (entry point dari Dataset)
# ──────────────────────────────────────────────────────
# ──────────────────────────────────────────────────────
# Path Resolver Dinamis (Menangani data split yang diacak lintas folder)
# ──────────────────────────────────────────────────────
def resolve_image_path(path):
    import os
    if os.path.exists(path):
        return path
    fname = os.path.basename(path)
    base_input = "/kaggle/input/datasets/mariaherrerot/aptos2019"
    for folder in ["train_images/train_images", "val_images/val_images", "test_images/test_images"]:
        test_path = os.path.join(base_input, folder, fname)
        if os.path.exists(test_path):
            return test_path
    return path


def load_image(path, use_clahe=False, cfg=None):
    """
    Load gambar dari disk dan terapkan preprocessing sesuai cfg.
    """
    path = resolve_image_path(path)
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        raise FileNotFoundError(f"Gambar tidak ditemukan: {path}")
    if cfg is not None and cfg.get("use_crop", True):
        img_bgr = crop_image_from_gray(img_bgr)

    if use_clahe and cfg is not None:
        mode = cfg.get("clahe_mode", "lab")

        if mode == "lab":
            img = apply_clahe(
                img_bgr,
                clip_limit = cfg["clahe_clip"],
                tile_grid  = cfg["clahe_grid"],
                alpha      = cfg["clahe_alpha"],
            )

        elif mode == "green":
            img = apply_clahe_green(
                img_bgr,
                clip_limit = cfg["clahe_clip"],
                tile_grid  = cfg["clahe_grid"],
                alpha      = cfg.get("clahe_green_alpha", 0.5),
                mode       = cfg.get("clahe_green_output", "merge_rgb"),
            )

        elif mode == "combined":
            img = apply_clahe_combined(
                img_bgr,
                clip_limit   = cfg["clahe_clip"],
                tile_grid    = cfg["clahe_grid"],
                alpha_lab    = cfg["clahe_alpha"],
                alpha_green  = cfg.get("clahe_green_alpha", 0.5),
                green_weight = cfg.get("clahe_green_weight", 0.4),
            )

        else:
            raise ValueError(f"clahe_mode tidak dikenal: '{mode}'. Pilih: lab / green / combined")
    else:
        img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    return img


# ──────────────────────────────────────────────────────
# Visualisasi Perbandingan CLAHE
# ──────────────────────────────────────────────────────
def visualize_clahe(cfg, n_samples=3):
    """
    Tampilkan perbandingan visual dengan Auto-Cropping aktif:
    Original | CLAHE LAB | CLAHE Green | CLAHE Combined
    untuk n_samples gambar acak dari training set.
    """
    df      = pd.read_csv(cfg["train_csv"]).sample(n=n_samples, random_state=42)
    fig, axes = plt.subplots(n_samples, 4,
                             figsize=(16, 4 * n_samples))
    if n_samples == 1:
        axes = [axes]

    titles = ["Original", "CLAHE LAB", "CLAHE Green\n(merge_rgb)", "CLAHE Combined"]

    for row_idx, (_, sample) in enumerate(df.iterrows()):
        path    = os.path.join(cfg["train_dir"], sample["id_code"] + ".png")
        img_bgr = cv2.imread(path)

        if img_bgr is None:
            continue

        # Terapkan Auto-Cropping
        if cfg.get("use_crop", True):
            img_bgr = crop_image_from_gray(img_bgr)

        imgs = [
            cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB),
            apply_clahe(img_bgr,
                        clip_limit=cfg["clahe_clip"],
                        tile_grid=cfg["clahe_grid"],
                        alpha=cfg["clahe_alpha"]),
            apply_clahe_green(img_bgr,
                              clip_limit=cfg["clahe_clip"],
                              tile_grid=cfg["clahe_grid"],
                              alpha=cfg.get("clahe_green_alpha", 0.5),
                              mode="merge_rgb"),
            apply_clahe_combined(img_bgr,
                                 clip_limit=cfg["clahe_clip"],
                                 tile_grid=cfg["clahe_grid"],
                                 alpha_lab=cfg["clahe_alpha"],
                                 alpha_green=cfg.get("clahe_green_alpha", 0.5),
                                 green_weight=cfg.get("clahe_green_weight", 0.4)),
        ]

        for col_idx, (img, title) in enumerate(zip(imgs, titles)):
            ax = axes[row_idx][col_idx]
            ax.imshow(img)
            ax.set_title(
                f"{title}\ngrade {int(sample['diagnosis'])}",
                fontsize=9
            )
            ax.axis("off")

    plt.suptitle(
        f"Perbandingan CLAHE — clip={cfg['clahe_clip']} "
        f"grid={cfg['clahe_grid']} alpha={cfg['clahe_alpha']}",
        fontsize=12, fontweight="bold"
    )
    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, "clahe_comparison.png")
    plt.savefig(p, dpi=150)
    plt.show()
    LOG.info(f"Visualisasi CLAHE disimpan → {os.path.basename(p)}")


# ── Jalankan visualisasi ──────────────────────────────
_tmp_cfg_vis = make_cfg(model_key=ACTIVE_MODEL)
visualize_clahe(_tmp_cfg_vis, n_samples=3)
del _tmp_cfg_vis

LOG.info("Cell 4 CLAHE OK → Lanjut ke Cell 5")

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║ CELL 5 — Dataset Class & Augmentasi                  ║
# ╚══════════════════════════════════════════════════════╝
def get_transforms(img_size, mode="train", use_aug=True):
    mean = [0.485, 0.456, 0.406]; std = [0.229, 0.224, 0.225]
    if mode == "train" and use_aug:
        # Augmentasi aman: Fokus pada rotasi/flip (seperti paper) + penyesuaian cahaya ringan
        aug_list = [
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.08, scale_limit=0.15, rotate_limit=30,
                border_mode=cv2.BORDER_REFLECT_101, p=0.5),
            A.RandomBrightnessContrast(
                brightness_limit=0.15, contrast_limit=0.15, p=0.4),
            A.HueSaturationValue(
                hue_shift_limit=5, sat_shift_limit=15, val_shift_limit=8, p=0.3),
            A.GaussianBlur(blur_limit=(3, 3), p=0.1),  # blur minimal
            # GridDistortion, GaussNoise, CoarseDropout → hapus dari train biasa
        ]
        aug_list += [A.Normalize(mean=mean, std=std), ToTensorV2()]
        return A.Compose(aug_list)
        
    else:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ])

class RetinopathyDataset(Dataset):
    def __init__(self, df, img_dir, cfg, transform=None, cache_dir=None, cache_to_ram=False):
        self.df             = df.reset_index(drop=True)
        self.img_dir        = img_dir
        self.cfg            = cfg
        self.transform      = transform
        self.cache_dir      = cache_dir   # None = tidak pakai cache
        self.cache_to_ram   = cache_to_ram

        # Pre-load cache ke RAM jika diaktifkan
        self.ram_cache = {}
        if self.cache_to_ram:
            self._preload_images()

    def _preload_images(self):
        unique_fnames = self.df["id_code"].unique()
        print(f"\n  [RAM CACHE] Pre-loading {len(unique_fnames)} unique images to RAM...")
        
        for fname in tqdm(unique_fnames, desc="RAM Caching", leave=False):
            orig_path = os.path.join(self.img_dir, fname + ".png")
            if self.cache_dir:
                cache_path = get_cache_path(orig_path, self.cache_dir)
                if os.path.exists(cache_path):
                    img_bgr = cv2.imread(cache_path)
                    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                else:
                    img = load_image(orig_path, self.cfg["use_clahe"], self.cfg)
            else:
                img = load_image(orig_path, self.cfg["use_clahe"], self.cfg)
            self.ram_cache[fname] = img
        print(f"  [RAM CACHE] Pre-load selesai. {len(self.ram_cache)} gambar disimpan di RAM.")

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        fname = row["id_code"]

        if self.cache_to_ram and fname in self.ram_cache:
            img = self.ram_cache[fname]
        else:
            path = os.path.join(self.img_dir, fname + ".png")
            if self.cache_dir:
                # ── Baca dari cache (sudah di-resize ke /kaggle/working) ─────────────
                cache_path = get_cache_path(path, self.cache_dir)
                if os.path.exists(cache_path):
                    img_bgr = cv2.imread(cache_path)
                    img     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                else:
                    # Fallback: proses real-time kalau cache tidak ada
                    img = load_image(path, self.cfg["use_clahe"], cfg=self.cfg)
            else:
                # ── Tidak pakai cache → load biasa ───────────────
                img = load_image(path, self.cfg["use_clahe"], self.cfg)

        label = int(row["diagnosis"])
        
        if self.transform:
            img = self.transform(image=img)["image"]
            
        return img, label


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║ CELL 5.5 — Pre-compute CLAHE ke Disk (Super-Fast)    ║
# ╚══════════════════════════════════════════════════════╝

# ──────────────────────────────────────────────────────
# Helper — dijalankan di setiap worker process
# ──────────────────────────────────────────────────────
def _process_single_image(path, cache_dir, cfg):
    """
    Proses 1 gambar secara sangat cepat dan aman:
    baca → downsample awal (jika besar) → crop → resize → CLAHE → tulis atomik.
    """
    path = resolve_image_path(path)
    out_path = get_cache_path(path, cache_dir)

    # Skip jika sudah selesai diproses sebelumnya & valid (Deep Verification)
    if os.path.exists(out_path):
        try:
            from PIL import Image as PILImage
            with PILImage.open(out_path) as im:
                im.load()  # deteksi file truncated/corrupt
            img = cv2.imread(out_path)
            if img is not None and img.size > 0:
                return "skipped"
        except Exception:
            # File corrupt -> hapus agar diproses ulang
            try:
                os.remove(out_path)
            except Exception:
                pass

    # Jalur penulisan sementara untuk mencegah file korup jika proses terputus
    tmp_path = out_path + ".tmp.jpg"

    try:
        img_bgr = cv2.imread(path)
        if img_bgr is None:
            return "failed"

        h, w = img_bgr.shape[:2]
        
        # Optimasi: Jika gambar raksasa, ciutkan dulu ke 1024px agar proses crop & resize melesat cepat
        max_dim = max(h, w)
        if max_dim > 1024:
            scale = 1024.0 / max_dim
            img_bgr = cv2.resize(img_bgr, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)

        # Potong bingkai hitam
        if cfg.get("use_crop", True):
            img_bgr = crop_image_from_gray(img_bgr)

        # Ubah ukuran ke target akhir
        img_size = cfg["img_size"]
        img_bgr  = cv2.resize(img_bgr, (img_size, img_size), interpolation=cv2.INTER_LINEAR)

        mode = cfg["clahe_mode"] if cfg["use_clahe"] else "NO_CLAHE"

        if not cfg["use_clahe"]:
            img_save = img_bgr
        elif mode == "lab":
            img_save = apply_clahe_bgr(
                img_bgr,
                clip_limit = cfg["clahe_clip"],
                tile_grid  = cfg["clahe_grid"],
                alpha      = cfg["clahe_alpha"],
            )
        elif mode == "blur_lab":
            img_save = apply_clahe_blur_lab_bgr(
                img_bgr,
                clip_limit  = cfg["clahe_clip"],
                tile_grid   = cfg["clahe_grid"],
                alpha       = cfg["clahe_alpha"],
                beta        = cfg.get("clahe_beta", 0.3),
                blur_kernel = cfg.get("clahe_blur_kernel", 5),
            )
        elif mode == "green":
            img_save = apply_clahe_green_bgr(
                img_bgr,
                clip_limit = cfg["clahe_clip"],
                tile_grid  = cfg["clahe_grid"],
                alpha      = cfg["clahe_green_alpha"],
                mode       = cfg["clahe_green_output"],
            )
        elif mode == "combined":
            img_save = apply_clahe_combined_bgr(
                img_bgr,
                clip_limit   = cfg["clahe_clip"],
                tile_grid    = cfg["clahe_grid"],
                alpha_lab    = cfg["clahe_alpha"],
                alpha_green  = cfg["clahe_green_alpha"],
                green_weight = cfg["clahe_green_weight"],
            )
        else:
            return "failed"

        if len(img_save.shape) == 2:
            img_save = cv2.cvtColor(img_save, cv2.COLOR_GRAY2BGR)

        # Simpan secara atomik (tulis ke .tmp lalu rename)
        cv2.imwrite(tmp_path, img_save, [cv2.IMWRITE_JPEG_QUALITY, 95])
        os.replace(tmp_path, out_path)
        return "ok"

    except Exception as e:
        import traceback
        print(f"\nError pada {path}: {e}")
        traceback.print_exc()
        # Bersihkan file sampah jika gagal di tengah jalan
        if os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except Exception:
                pass
        return "failed"


# ──────────────────────────────────────────────────────
# Cache Dir & Path Helper
# ──────────────────────────────────────────────────────
def get_clahe_cache_dir(cfg):
    """
    Folder cache unik per kombinasi parameter CLAHE & resolusi gambar.
    Ganti parameter atau resolusi → folder baru otomatis.
    """
    # Menyertakan format=jpg serta blur_kernel & beta ke dalam hash
    param_str = (
        f"use_clahe={cfg['use_clahe']}"
        f"_mode={cfg['clahe_mode']}"
        f"_clip={cfg['clahe_clip']}"
        f"_grid={cfg['clahe_grid']}"
        f"_alpha={cfg['clahe_alpha']}"
        f"_beta={cfg.get('clahe_beta', 0.3)}"
        f"_blur={cfg.get('clahe_blur_kernel', 5)}"
        f"_galpha={cfg['clahe_green_alpha']}"
        f"_gweight={cfg['clahe_green_weight']}"
        f"_gout={cfg['clahe_green_output']}"
        f"_size={cfg['img_size']}"
        f"_format=jpg"
    )
    tag       = hashlib.md5(param_str.encode()).hexdigest()[:8]
    cache_dir = os.path.join(OUTPUT_DIR, f"clahe_cache_{tag}")
    os.makedirs(cache_dir, exist_ok=True)
    return cache_dir


def get_cache_path(original_path, cache_dir):
    """
    Konversi path gambar asli → path cache.
    Menggunakan resolve_image_path secara dinamis agar cache match dengan folder aslinya.
    """
    original_path = resolve_image_path(original_path)
    fname_hash = hashlib.md5(original_path.encode()).hexdigest() + ".jpg"
    return os.path.join(cache_dir, fname_hash)


# ──────────────────────────────────────────────────────
# Pre-compute CLAHE — Multiprocessing
# ──────────────────────────────────────────────────────
def precompute_clahe(cfg, force=False, num_workers=None):
    """
    Proses CLAHE/Resize sekali untuk semua gambar, simpan ke disk.
    Pakai multiprocessing untuk memproses banyak gambar paralel.
    """
    cache_dir = get_clahe_cache_dir(cfg)
    flag_file = os.path.join(cache_dir, "DONE.flag")

    # Kalau sudah selesai sebelumnya & valid → skip
    if os.path.exists(flag_file) and not force:
        LOG.info(f"Cache sudah ada → {cache_dir}")
        LOG.info("Gunakan force=True untuk proses ulang.")
        return cache_dir

    # Auto-detect jumlah worker
    cpu_count = mp.cpu_count()
    if num_workers is None:
        num_workers = max(1, cpu_count - 1)

    mode = cfg["clahe_mode"] if cfg["use_clahe"] else "NO_CLAHE"
    LOG.info(f"Pre-compute gambar [{mode.upper()}] (Pre-resized to {cfg['img_size']}x{cfg['img_size']})")
    LOG.info(f"Workers : {num_workers} / {cpu_count} CPU cores")
    LOG.info(f"Cache   : {cache_dir}")

    # Kumpulkan semua path gambar dari split data
    all_paths = []
    for split in ["train", "val", "test"]:
        csv_key = f"{split}_csv"
        dir_key = f"{split}_dir"
        if csv_key not in cfg or dir_key not in cfg:
            continue
        df      = pd.read_csv(cfg[csv_key])
        img_dir = cfg[dir_key]
        for fname in df["id_code"]:
            p = os.path.join(img_dir, fname + ".png")
            all_paths.append(p)

    # Hapus duplikat
    all_paths = list(set(all_paths))
    LOG.info(f"Total gambar: {len(all_paths)}")

    # Hitung yang sudah ada (menggunakan Deep Verification)
    existing_files = set(os.listdir(cache_dir))
    already_done = 0
    remaining_paths = []
    
    # Pre-verify file cache untuk mendeteksi file corrupt
    for p in tqdm(all_paths, desc="Verifikasi cache di disk", leave=False):
        cache_path = get_cache_path(p, cache_dir)
        fname_hash = os.path.basename(cache_path)
        if fname_hash in existing_files:
            # Deep verification untuk file cache
            try:
                from PIL import Image as PILImage
                with PILImage.open(cache_path) as im:
                    im.load()
                img = cv2.imread(cache_path)
                if img is not None and img.size > 0:
                    already_done += 1
                    continue
            except Exception:
                try:
                    os.remove(cache_path)
                except Exception:
                    pass
        remaining_paths.append(p)
        
    remaining = len(remaining_paths)

    if remaining == 0:
        LOG.info("Semua gambar sudah ada di cache.")
        with open(flag_file, "w") as f:
            f.write(f"mode={mode} | total={len(all_paths)} | skipped={already_done}")
        return cache_dir

    LOG.info(f"Sudah ada : {already_done} gambar (di-skip)")
    LOG.info(f"Sisa      : {remaining} gambar akan diproses")

    # Buat partial function dengan parameter tetap
    worker_fn = partial(_process_single_image,
                        cache_dir=cache_dir,
                        cfg=cfg)

    # Jalankan multiprocessing hanya untuk gambar yang tersisa dengan maxtasksperchild=100 untuk mencegah kebocoran memori
    results = {"ok": 0, "skipped": 0, "failed": 0}

    with mp.Pool(processes=num_workers, maxtasksperchild=100) as pool:
        for result in tqdm(
            pool.imap_unordered(worker_fn, remaining_paths, chunksize=32),
            total    = remaining,
            desc     = f"Precompute [{mode}] {num_workers} workers",
        ):
            results[result] += 1

    results["skipped"] += already_done

    # Tulis flag selesai
    with open(flag_file, "w") as f:
        f.write(
            f"mode={mode} | total={len(all_paths)} | "
            f"ok={results['ok']} | skipped={results['skipped']} | "
            f"failed={results['failed']}"
        )

    # Summary
    print(f"\n{'═'*50}")
    print(f"  Pre-compute Gambar Selesai")
    print(f"{'═'*50}")
    print(f"  Mode     : {mode.upper()}")
    print(f"  Workers  : {num_workers}")
    print(f"  Total    : {len(all_paths)}")
    print(f"  OK       : {results['ok']}")
    print(f"  Skipped  : {results['skipped']} (sudah ada)")
    print(f"  Failed   : {results['failed']}")
    print(f"  Cache    : {cache_dir}")
    print(f"{'═'*50}")

    return cache_dir

# ── Jalankan pre-compute (Kedua Konfigurasi) ──────────
# Mengeksekusi pre-compute secara langsung saat cell dijalankan:

_tmp_cfg_noclahe = make_cfg(model_key=ACTIVE_MODEL, use_clahe=False)
precompute_clahe(
    _tmp_cfg_noclahe,
    force       = False,
    num_workers = _tmp_cfg_noclahe.get("num_workers", 4),
)

_tmp_cfg_clahe = make_cfg(model_key=ACTIVE_MODEL, use_clahe=True)
precompute_clahe(
    _tmp_cfg_clahe,
    force       = False,
    num_workers = _tmp_cfg_clahe.get("num_workers", 4),
)

del _tmp_cfg_noclahe, _tmp_cfg_clahe

LOG.info("Dataset & transforms OK → Lanjut ke Cell 6")

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║ CELL 6 — DataLoader                                  ║
# ╚══════════════════════════════════════════════════════╝
def build_loaders(cfg, cache_dir=None):
    df_train = pd.read_csv(cfg["train_csv"])
    df_val   = pd.read_csv(cfg["val_csv"])
    df_test  = pd.read_csv(cfg["test_csv"])
    
    train_tf = get_transforms(cfg["img_size"], mode="train",
                               use_aug=cfg["use_aug"])
    val_tf   = get_transforms(cfg["img_size"], mode="val",
                               use_aug=False)
                               
    cache_to_ram = cfg.get("cache_to_ram", False)
                               
    nw = cfg["num_workers"]
    _pw = (nw > 0)  # persistent_workers
    _pf = 2 if nw > 0 else None  # prefetch_factor
    train_loader = DataLoader(
        RetinopathyDataset(df_train, cfg["train_dir"], cfg, transform=train_tf, cache_dir=cache_dir, cache_to_ram=cache_to_ram),
        batch_size=cfg["batch_size"], shuffle=True,
        num_workers=nw, pin_memory=True, drop_last=True,
        persistent_workers=_pw, prefetch_factor=_pf)
    val_loader = DataLoader(
        RetinopathyDataset(df_val, cfg["val_dir"], cfg, val_tf, cache_dir=cache_dir, cache_to_ram=cache_to_ram),
        batch_size=cfg["batch_size"], shuffle=False,
        num_workers=nw, pin_memory=True,
        persistent_workers=_pw, prefetch_factor=_pf)
    test_loader = DataLoader(
        RetinopathyDataset(df_test, cfg["test_dir"], cfg, val_tf, cache_dir=cache_dir, cache_to_ram=cache_to_ram),
        batch_size=cfg["batch_size"], shuffle=False,
        num_workers=nw, pin_memory=True,
        persistent_workers=_pw, prefetch_factor=_pf)

    tr_dist = df_train["diagnosis"].value_counts().sort_index()
    va_dist = df_val["diagnosis"].value_counts().sort_index()
    te_dist = df_test["diagnosis"].value_counts().sort_index()
    total   = len(df_train)+len(df_val)+len(df_test)
    CLS     = ["No DR","Mild","Moderate","Severe","Proliferative"]
    print("\n"+"="*75)
    print("  [CELL 6]  DATASET STATISTICS")
    print("="*75)
    print(f"  {'Split':<8} {'Samples':>8}  {'% Total':>8}  Class Distribution")
    print("  "+"-"*73)
    for sn,df_,dist in [("Train",df_train,tr_dist),
                         ("Val",df_val,va_dist),
                         ("Test",df_test,te_dist)]:
        pct  = len(df_)/total*100
        dstr = "  ".join([f"C{i}:{dist.get(i,0)}" for i in range(5)])
        print(f"  {sn:<8} {len(df_):>8}  {pct:>7.1f}%  {dstr}")
    print("  "+"-"*73)
    print(f"  {'TOTAL':<8} {total:>8}  {'100.0%':>8}")
    print("="*75)
    return train_loader, val_loader, test_loader, df_train


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║ CELL 6.1 — Visualisasi Batch & Distribusi Kelas      ║
# ╚══════════════════════════════════════════════════════╝
def show_augmented_batch(loader, class_names, save_path=None):
    images, labels = next(iter(loader))
    n_show = min(len(images), 32); cols = 8
    rows   = max(1, (n_show+cols-1)//cols)
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows*2))
    fig.suptitle("Visualisasi Hasil Augmentasi (Train Batch Preview)", fontsize=14, fontweight="bold")
    mean = np.array([0.485,0.456,0.406]).reshape(3,1,1)
    std  = np.array([0.229,0.224,0.225]).reshape(3,1,1)
    for i, ax in enumerate(axes.flat):
        if i < len(images):
            img = np.clip(images[i].numpy()*std+mean, 0, 1)
            ax.imshow(np.transpose(img,(1,2,0)))
            ax.set_title(class_names[labels[i].item()], fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    p = save_path or os.path.join(OUTPUT_DIR, "augmented_batch_preview.png")
    plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close('all')
    LOG.info(f"Batch visualisasi disimpan → {os.path.basename(p)}")

def plot_class_distribution(cfg):
    splits_data = {k: pd.read_csv(cfg[f"{k}_csv"]) for k in ["train","val","test"]}
    fig = plt.figure(figsize=(20, 14))
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
    fig.suptitle("Distribusi Kelas Dataset APTOS 2019 — Detail Lengkap",
                 fontsize=15, fontweight="bold")

    for col, (split_name, df) in enumerate(splits_data.items()):
        counts  = df["diagnosis"].value_counts().sort_index()
        total   = counts.sum()

        ax_bar = fig.add_subplot(gs[0, col])
        bars = ax_bar.bar(CLASS_NAMES, counts.values, color=PALETTE, alpha=0.85, edgecolor="white")
        ax_bar.set_title(f"{split_name.title()} Set (n={total})", fontweight="bold")
        ax_bar.set_ylabel("Jumlah Sampel"); ax_bar.tick_params(axis="x", rotation=20, labelsize=8)
        for b, v in zip(bars, counts.values):
            ax_bar.text(b.get_x()+b.get_width()/2, b.get_height()+total*0.005,
                        f"{v}\n({v/total*100:.1f}%)", ha="center", va="bottom", fontsize=8)

        ax_pie = fig.add_subplot(gs[1, col])
        wedges, _, at = ax_pie.pie(
            counts.values, labels=CLASS_NAMES, colors=PALETTE, autopct="%1.1f%%",
            startangle=140, pctdistance=0.82,
            wedgeprops=dict(edgecolor="white", linewidth=1.5))
        for a in at: a.set_fontsize(8)
        ax_pie.set_title(f"{split_name.title()} — Proporsi", fontweight="bold")

        ax_area = fig.add_subplot(gs[2, col])
        left = 0
        for i, (c, name) in enumerate(zip(counts.values, CLASS_NAMES)):
            p = c/total*100
            ax_area.barh(0, p, left=left, color=PALETTE[i], height=0.5, label=f"{name} ({p:.1f}%)")
            if p > 3:
                ax_area.text(left+p/2, 0, f"{p:.0f}%", ha="center", va="center", fontsize=8, color="white", fontweight="bold")
            left += p
        ax_area.set_xlim(0,100); ax_area.set_yticks([]); ax_area.set_xlabel("Persentase (%)")
        ax_area.set_title(f"{split_name.title()} — Stacked Bar", fontweight="bold")
        if col == 2: ax_area.legend(bbox_to_anchor=(1.01,1), loc="upper left", fontsize=7)

    p = os.path.join(OUTPUT_DIR, "class_distribution_full.png")
    plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close('all')
    LOG.info(f"Distribusi kelas disimpan → {os.path.basename(p)}")

# Menjalankan visualisasi distribusi & preview data secara otomatis
_tmp_vis = make_cfg(model_key=ACTIVE_MODEL, use_clahe=True, use_aug=True)  # FIX: teruskan ACTIVE_MODEL
_tmp_vis["cache_to_ram"] = False  # Jangan preload cache RAM untuk visualisasi
_tmp_tl, _tmp_vl, _tmp_testl, _ = build_loaders(_tmp_vis)
show_augmented_batch(_tmp_tl, CLASS_NAMES)
plot_class_distribution(_tmp_vis)

In [ ]:
# ================================================================
# CELL 7 — Build Model (MSAG + Attention Head)
# ================================================================


# ----------------------------------------------------------------
# Multi-Sample Dropout Head (Legacy — fallback jika MSAG nonaktif)
# ----------------------------------------------------------------
class MultiSampleDropoutHead(nn.Module):
    """
    Classifier dengan multi-sample dropout untuk ensemble internal.
    Digunakan sebagai fallback ketika MSAG tidak aktif.
    """
    def __init__(self, in_features, num_classes, drop_rate=0.5, num_samples=5):
        super().__init__()
        self.dropouts = nn.ModuleList(
            [nn.Dropout(drop_rate) for _ in range(num_samples)]
        )
        self.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        if self.training:
            return torch.mean(
                torch.stack(
                    [self.fc(drop(x)) for drop in self.dropouts], dim=0
                ),
                dim=0,
            )
        else:
            return self.fc(x)


# ----------------------------------------------------------------
# Attention Classifier Head (Channel Attention + Deeper FC + MSD)
# ----------------------------------------------------------------
class AttentionClassifierHead(nn.Module):
    """
    Enhanced classifier head:
      1. Channel Attention (SE-style) — memilih channel fitur penting
      2. Deeper FC layers (BN + ReLU + Dropout) — kapasitas lebih besar
      3. Multi-Sample Dropout — ensemble kecil saat inference

    Digunakan di dalam MSAGLite atau sebagai pengganti classifier biasa.
    """
    def __init__(self, in_features, num_classes=5, drop_rate=0.4, num_samples=5):
        super().__init__()

        # Channel Attention (Squeeze-and-Excitation)
        self.attention = nn.Sequential(
            nn.Linear(in_features, max(1, in_features // 4)),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(max(1, in_features // 4), in_features),
            nn.Sigmoid(),
        )

        # Deeper FC dengan BatchNorm
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(drop_rate),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(drop_rate * 0.7),
        )

        # Final layer + Multi-Sample Dropout
        self.dropouts = nn.ModuleList(
            [nn.Dropout(drop_rate * 0.5) for _ in range(num_samples)]
        )
        self.fc_final = nn.Linear(256, num_classes)
        self.num_samples = num_samples

    def forward(self, x):
        # Channel attention
        attn = self.attention(x)
        x = x * attn

        # Deeper classification
        x = self.classifier(x)

        # Multi-sample dropout (ensemble saat training)
        if self.training:
            return torch.mean(
                torch.stack(
                    [self.fc_final(d(x)) for d in self.dropouts], dim=0
                ),
                dim=0,
            )
        else:
            return self.fc_final(x)


# ----------------------------------------------------------------
# MSAG-Lite (Multi-Scale Attention Gate — Lightweight)
# ----------------------------------------------------------------
class MSAGLite(nn.Module):
    """
    Multi-Scale Attention Gate — versi ringan untuk EfficientNet-B0.

    Prinsip kerja:
      1. Hook 3 blok backbone di kedalaman berbeda (~30%, ~55%, blok terakhir)
      2. Fitur halus  (stage awal)  -> detail microaneurysms, pembuluh darah
         Fitur menengah (stage tengah) -> lesi, exudates
         Fitur kasar  (stage akhir)  -> pola global, neovascularization
      3. Attention gate per skala:
         Gate = Conv1x1(feature + upsampled_context) -> sigmoid
      4. Fitur ter-attend di-project & di-pool ke vektor tetap
      5. Fused vektor masuk AttentionClassifierHead

    Auto-detect channel dimensions via dummy forward pass saat __init__.
    Bekerja otomatis untuk berbagai arsitektur berbasis blok.
    """

    def __init__(
        self,
        backbone,
        num_classes=5,
        hook_indices=None,
        fused_ch=256,
        drop_rate=0.4,
        img_size=300,
    ):
        super().__init__()
        self.backbone = backbone
        self.num_classes = num_classes
        self._hook_features = {}

        # -- Auto-detect hook indices --
        if hook_indices is None:
            self._detect_hook_indices()
        else:
            self.hook_indices = hook_indices

        # -- Detect channel dimensions via probe --
        fine_ch, mid_ch, coarse_ch = self._probe_channels(img_size)
        self.fine_ch = fine_ch
        self.mid_ch = mid_ch
        self.coarse_ch = coarse_ch

        print(
            f"  [MSAG] Channels : fine={fine_ch}, mid={mid_ch}, "
            f"coarse={coarse_ch} (auto-detected)"
        )
        print(
            f"  [MSAG] Hook idx : {self.hook_indices} "
            f"(dari {len(self._get_blocks())} blok)"
        )

        # -- Attention Gates (spatial, conditioned on coarse context) --
        self.gate_fine = nn.Sequential(
            nn.Conv2d(fine_ch + coarse_ch, 64, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, fine_ch, 1),
            nn.Sigmoid(),
        )
        self.gate_mid = nn.Sequential(
            nn.Conv2d(mid_ch + coarse_ch, 128, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, mid_ch, 1),
            nn.Sigmoid(),
        )

        # -- Projection layers (reduce channels) --
        proj_out = fused_ch // 2  # 128 per scale
        self.proj_fine = nn.Sequential(
            nn.Conv2d(fine_ch, proj_out, 1, bias=False),
            nn.BatchNorm2d(proj_out),
            nn.ReLU(inplace=True),
        )
        self.proj_mid = nn.Sequential(
            nn.Conv2d(mid_ch, proj_out, 1, bias=False),
            nn.BatchNorm2d(proj_out),
            nn.ReLU(inplace=True),
        )

        # -- Classifier (Attention Head on fused features) --
        fused_in = proj_out * 2 + coarse_ch  # 128 + 128 + coarse_ch
        self.classifier = AttentionClassifierHead(
            in_features=fused_in,
            num_classes=num_classes,
            drop_rate=drop_rate,
        )

        # -- Register hooks (setelah semua layer dibuat) --
        self._register_hooks()

    # -- Internal Helpers --

    def _get_blocks(self):
        """Ambil list modul blok dari backbone."""
        blocks = getattr(self.backbone, "blocks", None)
        if blocks is None:
            blocks = getattr(self.backbone, "stages", None)
        if blocks is None:
            raise AttributeError(
                "Backbone tidak memiliki 'blocks' atau 'stages'. "
                "MSAG hanya support arsitektur berbasis blok."
            )
        return blocks

    def _detect_hook_indices(self):
        """Auto-detect 3 hook indices di ~30%, ~55%, ~80% kedalaman."""
        blocks = self._get_blocks()
        n = len(blocks)
        fine_idx = max(1, int(n * 0.30))
        mid_idx = max(fine_idx + 1, int(n * 0.55))
        coarse_idx = n - 1
        self.hook_indices = [fine_idx, mid_idx, coarse_idx]

    def _probe_channels(self, img_size):
        """Jalankan dummy forward untuk deteksi channel di setiap hook point."""
        blocks = self._get_blocks()
        probed = {}

        def make_probe_hook(name):
            def hook_fn(module, inp, out):
                probed[name] = out
            return hook_fn

        # Pasang hook sementara (hanya fine & mid — coarse dari forward_features)
        handles = []
        for idx in self.hook_indices[:2]:  # [FIX] skip hook ke-3 (coarse), tidak diperlukan
            h = blocks[idx].register_forward_hook(make_probe_hook(idx))
            handles.append(h)

        # Forward pass
        device = next(self.backbone.parameters()).device
        dummy = torch.zeros(1, 3, img_size, img_size, device=device)
        self.backbone.eval()
        with torch.no_grad():
            coarse_out = self.backbone.forward_features(dummy)

        # Cleanup hooks
        for h in handles:
            h.remove()

        # Extract channels
        fine_ch = probed[self.hook_indices[0]].shape[1]
        mid_ch  = probed[self.hook_indices[1]].shape[1]
        # coarse channel langsung dari output backbone.forward_features()
        coarse_ch = coarse_out.shape[1]  # [FIX] dim 4 atau 2 sama-sama .shape[1]

        return fine_ch, mid_ch, coarse_ch

    def _register_hooks(self):
        """Pasang forward hook permanen pada blok yang dipilih."""
        blocks = self._get_blocks()
        for idx in self.hook_indices:
            block = blocks[idx]
            block.register_forward_hook(self._make_hook(idx))

    def _make_hook(self, name):
        """Buat hook function yang menyimpan output di dict."""
        def hook_fn(module, inp, out):
            self._hook_features[name] = out
        return hook_fn

    # -- Forward --

    def forward(self, x):
        """
        Alur:
          backbone.forward_features -> hook menangkap 3 skala fitur
          -> attention gate per skala (conditioned on coarse)
          -> project & pool -> fuse -> classify
        """
        self._hook_features.clear()

        # Coarse features dari output akhir backbone
        coarse = self.backbone.forward_features(x)  # (B, C, H, W)

        # Fine & mid features dari hook
        feat_fine = self._hook_features.get(self.hook_indices[0])
        feat_mid = self._hook_features.get(self.hook_indices[1])

        if feat_fine is None or feat_mid is None:
            # Fallback: hooks gagal -> gunakan coarse saja
            pooled = F.adaptive_avg_pool2d(coarse, 1).flatten(1)
            dummy_f = torch.zeros(
                coarse.size(0), self.proj_fine[0].out_channels,
                device=x.device,
            )
            dummy_m = torch.zeros_like(dummy_f)
            return self.classifier(
                torch.cat([dummy_f, dummy_m, pooled], dim=1)
            )

        # Upsample coarse sebagai spatial context untuk gate
        ctx_fine = F.interpolate(
            coarse, size=feat_fine.shape[2:],
            mode="bilinear", align_corners=False,
        )
        ctx_mid = F.interpolate(
            coarse, size=feat_mid.shape[2:],
            mode="bilinear", align_corners=False,
        )

        # Attention gates (belajar region penting per skala)
        gate_f = self.gate_fine(
            torch.cat([feat_fine, ctx_fine], dim=1)
        )
        gate_m = self.gate_mid(
            torch.cat([feat_mid, ctx_mid], dim=1)
        )

        # Apply gates (element-wise multiplication)
        gated_fine = feat_fine * gate_f
        gated_mid = feat_mid * gate_m

        # Project & Global Average Pool
        fine_pooled = F.adaptive_avg_pool2d(
            self.proj_fine(gated_fine), 1
        ).flatten(1)
        mid_pooled = F.adaptive_avg_pool2d(
            self.proj_mid(gated_mid), 1
        ).flatten(1)
        coarse_pooled = F.adaptive_avg_pool2d(coarse, 1).flatten(1)

        # Fuse multi-scale features
        fused = torch.cat(
            [fine_pooled, mid_pooled, coarse_pooled], dim=1
        )

        # Classify
        return self.classifier(fused)


# ================================================================
# Build Model — Entry Point
# ================================================================



# ================================================================
# MSAG Standard / Full (Tanpa Reduksi Channel / Full Feature Depth)
# ================================================================

class MSAGFull(nn.Module):
    """
    Multi-Scale Attention Gate (Full/Standard Version).
    Mempertahankan jumlah channel asli dari fine, mid, dan coarse features
    tanpa proyeksi bottleneck (fused_ch).
    """

    def __init__(
        self,
        backbone,
        num_classes,
        hook_indices,
        drop_rate=0.2,
        img_size=300,
    ):
        super().__init__()
        self.backbone = backbone
        self.num_classes = num_classes
        self.hook_indices = hook_indices
        self._hook_features = {}

        self._register_hooks()

        # Deteksi dimensi channel tiap skala secara dinamis
        with torch.no_grad():
            dummy = torch.zeros(2, 3, img_size, img_size)
            _ = self.backbone.forward_features(dummy)
            self.fine_ch = self._hook_features[hook_indices[0]].shape[1]
            self.mid_ch = self._hook_features[hook_indices[1]].shape[1]
            self.coarse_ch = self.backbone.forward_features(dummy).shape[1]

        self._hook_features.clear()

        # Full Attention Gate (Fine scale)
        self.gate_fine = nn.Sequential(
            nn.Conv2d(
                self.fine_ch + self.coarse_ch,
                self.fine_ch,
                kernel_size=1,
                bias=False,
            ),
            nn.BatchNorm2d(self.fine_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(self.fine_ch, 1, kernel_size=1, bias=True),
            nn.Sigmoid(),
        )

        # Full Attention Gate (Mid scale)
        self.gate_mid = nn.Sequential(
            nn.Conv2d(
                self.mid_ch + self.coarse_ch,
                self.mid_ch,
                kernel_size=1,
                bias=False,
            ),
            nn.BatchNorm2d(self.mid_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(self.mid_ch, 1, kernel_size=1, bias=True),
            nn.Sigmoid(),
        )

        # Total dimensi fusi channel penuh (fine_ch + mid_ch + coarse_ch)
        fused_in = self.fine_ch + self.mid_ch + self.coarse_ch

        # Attention Classifier Head
        self.classifier = AttentionClassifierHead(
            in_features=fused_in,
            num_classes=num_classes,
            drop_rate=drop_rate,
        )

    def _register_hooks(self):
        blocks = getattr(self.backbone, "blocks", None)
        if blocks is None:
            blocks = getattr(self.backbone, "stages", None)
        for idx in self.hook_indices[:2]:
            blocks[idx].register_forward_hook(self._make_hook(idx))

    def _make_hook(self, name):
        def hook_fn(module, inp, out):
            self._hook_features[name] = out
        return hook_fn

    def forward(self, x):
        self._hook_features.clear()

        coarse = self.backbone.forward_features(x)
        feat_fine = self._hook_features.get(self.hook_indices[0])
        feat_mid = self._hook_features.get(self.hook_indices[1])

        if feat_fine is None or feat_mid is None:
            pooled = F.adaptive_avg_pool2d(coarse, 1).flatten(1)
            dummy_f = torch.zeros(coarse.size(0), self.fine_ch, device=x.device)
            dummy_m = torch.zeros(coarse.size(0), self.mid_ch, device=x.device)
            return self.classifier(torch.cat([dummy_f, dummy_m, pooled], dim=1))

        ctx_fine = F.interpolate(
            coarse, size=feat_fine.shape[2:], mode="bilinear", align_corners=False
        )
        ctx_mid = F.interpolate(
            coarse, size=feat_mid.shape[2:], mode="bilinear", align_corners=False
        )

        gate_f = self.gate_fine(torch.cat([feat_fine, ctx_fine], dim=1))
        gate_m = self.gate_mid(torch.cat([feat_mid, ctx_mid], dim=1))

        gated_fine = feat_fine * gate_f
        gated_mid = feat_mid * gate_m

        fine_pooled = F.adaptive_avg_pool2d(gated_fine, 1).flatten(1)
        mid_pooled = F.adaptive_avg_pool2d(gated_mid, 1).flatten(1)
        coarse_pooled = F.adaptive_avg_pool2d(coarse, 1).flatten(1)

        fused = torch.cat([fine_pooled, mid_pooled, coarse_pooled], dim=1)
        return self.classifier(fused)


def build_model(cfg):

    model_name = cfg["model_name"]

    # -- Base arguments --
    kwargs = {
        "pretrained": True,
        "num_classes": cfg["num_classes"],
        "drop_rate": cfg["drop_rate"],
    }

    # -- Tambahkan drop_path_rate jika support --
    use_drop_path = model_name not in MODELS_SKIP_DROP_PATH
    if use_drop_path:
        kwargs["drop_path_rate"] = cfg["drop_path_rate"]

    # -- Build base model dari timm --
    try:
        model = timm.create_model(model_name, **kwargs)
    except TypeError as e:
        print(f"\n[WARNING] {model_name} tidak support drop_path_rate")
        print("[WARNING] Fallback tanpa drop_path_rate")
        print(f"[DETAIL] {e}")
        kwargs.pop("drop_path_rate", None)
        model = timm.create_model(model_name, **kwargs)

    # -- MSAG or Standard Classifier --
    use_msag = cfg.get("use_msag", False)  # [FIX] konsisten dgn make_cfg

    if use_msag:
        # Auto-detect hook indices dari backbone blocks
        blocks = getattr(model, "blocks", None)
        if blocks is None:
            blocks = getattr(model, "stages", None)
        if blocks is None:
            LOG.warn(
                "Backbone tidak punya 'blocks' atau 'stages'. MSAG dinonaktifkan, "
                "pakai classifier standar."
            )
            model = model.to(DEVICE)
            # Fallback ke MSD biasa
            classifier = model.get_classifier()
            if isinstance(classifier, nn.Linear) and cfg.get("use_msd", False):
                in_features = classifier.in_features
                for name, module in model.named_children():
                    if module is classifier:
                        setattr(
                            model, name,
                            MultiSampleDropoutHead(
                                in_features, cfg["num_classes"],
                                drop_rate=cfg["drop_rate"],
                            ).to(DEVICE),
                        )
                        break
        else:
            n_blocks = len(blocks)
            # 3 titik hook: ~30%, ~55%, blok terakhir
            fine_idx = max(1, int(n_blocks * 0.30))
            mid_idx = max(fine_idx + 1, int(n_blocks * 0.55))
            coarse_idx = n_blocks - 1
            hook_indices = [fine_idx, mid_idx, coarse_idx]

            # Wrap backbone dengan MSAG
            # Reset head timm agar tidak membuang memori & LR optimizer
            model.reset_classifier(0)  # [FIX] hapus classifier head sebelum MSAG wrap
            msag_type = cfg.get("msag_type", "lite").lower()
            if msag_type == "full":
                model = MSAGFull(backbone=model, num_classes=cfg["num_classes"], hook_indices=hook_indices, drop_rate=cfg["drop_rate"], img_size=cfg["img_size"]).to(DEVICE)
            else:
                model = MSAGLite(
                backbone=model,
                num_classes=cfg["num_classes"],
                hook_indices=hook_indices,
                fused_ch=cfg.get("msag_fused_ch", 256),
                drop_rate=cfg["drop_rate"],
                img_size=cfg["img_size"],
            ).to(DEVICE)

    else:
        # -- Standard path: timm model + optional MSD --
        model = model.to(DEVICE)

        use_msd = cfg.get("use_msd", False)
        if use_msd:
            classifier = model.get_classifier()
            if isinstance(classifier, nn.Linear):
                in_features = classifier.in_features
                for name, module in model.named_children():
                    if module is classifier:
                        setattr(
                            model, name,
                            MultiSampleDropoutHead(
                                in_features, cfg["num_classes"],
                                drop_rate=cfg["drop_rate"],
                            ).to(DEVICE),
                        )
                        break

    # -- Statistik model --
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )
    size_mb = sum(
        p.numel() * p.element_size() for p in model.parameters()
    ) / 1024**2

    # -- Display --
    print("\n" + "=" * 68)
    print("  [CELL 7]  MODEL SUMMARY")
    print("=" * 68)
    print(f"  {'Architecture':<30}: {cfg['model_name']}")
    print(f"  {'Pretrained':<30}: ImageNet (timm)")
    print(f"  {'Num Classes':<30}: {cfg['num_classes']}")
    print(f"  {'Input Size':<30}: {cfg['img_size']}x{cfg['img_size']}x3 px")
    print(f"  {'Total Parameters':<30}: {total:>15,}")
    print(f"  {'Trainable Parameters':<30}: {trainable:>15,}")
    print(f"  {'Model Size (fp32)':<30}: {size_mb:>14.2f} MB")
    print(f"  {'Dropout':<30}: {cfg['drop_rate']}")
    if use_drop_path:
        print(f"  {'Drop Path':<30}: {cfg['drop_path_rate']}")
    else:
        print(f"  {'Drop Path':<30}: SKIPPED")
    print(f"  {'Device':<30}: {DEVICE}")

    if use_msag and hasattr(model, "backbone"):
        print(f"  {'MSAG':<30}: AKTIF")
        print(
            f"  {'MSAG Channels':<30}: "
            f"fine={model.fine_ch}, mid={model.mid_ch}, "
            f"coarse={model.coarse_ch}"
        )
        print(f"  {'MSAG Hook Indices':<30}: {model.hook_indices}")
        print(
            f"  {'Head':<30}: "
            f"AttentionClassifierHead (SE + Deeper FC + MSD)"
        )
    elif cfg.get("use_msd", False):
        print(f"  {'Head':<30}: MultiSampleDropoutHead")

    print("=" * 68)

    return model


LOG.info("build_model (MSAG + AttentionHead) OK -> Lanjut ke Cell 8")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║ CELL 8 — Loss Function + Optimizer + Scheduler                   ║
# ╚══════════════════════════════════════════════════════════════════╝

# ──────────────────────────────────────────────────────
# Class-Balanced Focal Loss + Ordinal Penalty
# ──────────────────────────────────────────────────────
class ClassBalancedFocalLoss(nn.Module):
    def __init__(self, samples_per_cls, num_classes=5, beta=0.999,
                 gamma=2.0, ordinal_weight=0.3):
        """
        Class-Balanced Focal Loss + Ordinal Penalty (Cui et al., CVPR 2019)

        - Class-Balanced : bobot tiap kelas berdasarkan jumlah sampel
        - Focal Loss     : fokus ke hard examples (1-pt)^gamma
        - Ordinal Penalty: hukum prediksi yang jauh secara klinis lebih berat
                           misal: salah grade 0→4 dihukum lebih berat dari 0→1
        """
        super().__init__()
        self.gamma          = gamma
        self.ordinal_weight = ordinal_weight
        self.num_classes    = num_classes

        # ── Class-Balanced Weights ────────────────────────────────
        effective_num = 1.0 - np.power(beta, samples_per_cls)
        weights       = (1.0 - beta) / np.array(effective_num)
        weights       = weights / np.sum(weights) * num_classes
        
        # Boost weights of severe (class 3) and proliferative (class 4) to address low sensitivity
        weights[3] *= 1.5  # Moderate boost to avoid hurting global accuracy
        weights[4] *= 1.2
        
        self.register_buffer("weights",
                             torch.tensor(weights, dtype=torch.float32))

        # ── Ordinal Distance Matrix ───────────────────────────────
        # dist[i][j] = seberapa jauh kelas i dari kelas j (dinormalisasi 0-1)
        # Contoh untuk 5 kelas:
        # dist[0][4] = 4/4 = 1.0  (sangat jauh, hukuman besar)
        # dist[0][1] = 1/4 = 0.25 (dekat, hukuman kecil)
        dist = torch.zeros(num_classes, num_classes)
        for i in range(num_classes):
            for j in range(num_classes):
                dist[i, j] = abs(i - j)
        self.register_buffer("ordinal_dist", dist / (num_classes - 1))

    def forward(self, logits, targets):
        # ── Focal Loss ────────────────────────────────────────────
        ce_loss      = F.cross_entropy(logits, targets, reduction="none")
        p_t          = torch.exp(-ce_loss)
        focal_weight = torch.pow(1.0 - p_t, self.gamma)
        class_weight = self.weights.to(targets.device)[targets]

        # ── Ordinal Penalty ───────────────────────────────────────
        probs       = torch.softmax(logits, dim=1)
        ord_penalty = (probs * self.ordinal_dist.to(targets.device)[targets]).sum(dim=1)

        # ── Gabungkan ─────────────────────────────────────────────
        loss = class_weight * focal_weight * ce_loss + self.ordinal_weight * ord_penalty
        return loss.mean()


# ──────────────────────────────────────────────────────
# Combined Loss (Focal + Label Smoothing)
# ──────────────────────────────────────────────────────
class CombinedLoss(nn.Module):
    def __init__(self, focal_loss, smoothing=0.1, focal_weight=0.7, ce_weight=0.3):
        """
        Gabungan Focal Loss + CrossEntropy dengan Label Smoothing.

        - Focal Loss      : tangani class imbalance & hard examples
        - Label Smoothing : cegah model terlalu overconfident
                            misal: label [0,0,1,0,0] → [0.025,0.025,0.9,0.025,0.025]
        """
        super().__init__()
        self.focal        = focal_loss
        self.ce_smooth    = nn.CrossEntropyLoss(label_smoothing=smoothing)
        self.focal_weight = focal_weight
        self.ce_weight    = ce_weight

    def forward(self, logits, targets):
        return self.focal_weight * self.focal(logits, targets) + \
               self.ce_weight    * self.ce_smooth(logits, targets)


# ──────────────────────────────────────────────────────
# Build Training Components
# ──────────────────────────────────────────────────────
def build_training_components(model, cfg):
    steps = cfg.get("steps_per_epoch")
    if not steps:
        print("[WARNING] steps_per_epoch belum diset! "
              "Gunakan active_cfg['steps_per_epoch'] = len(train_loader) "
              "sebelum memanggil fungsi ini.")
        steps = 80

    # ── Distribusi kelas dari CSV ─────────────────────
    df_train        = pd.read_csv(cfg["train_csv"])
    samples_per_cls = df_train["diagnosis"].value_counts().sort_index().values

    # ── Loss Function ─────────────────────────────────
    focal = ClassBalancedFocalLoss(
        samples_per_cls = samples_per_cls,
        num_classes     = cfg["num_classes"],
        beta            = cfg["loss_beta"],
        gamma           = cfg["focal_gamma"],
        ordinal_weight  = cfg["ordinal_weight"],
    ).to(DEVICE)

    criterion = CombinedLoss(
        focal, 
        smoothing    = cfg["label_smoothing"],
        focal_weight = cfg["focal_loss_weight"],
        ce_weight    = cfg["ce_loss_weight"]
    ).to(DEVICE)

    # ── Optimizer: Layer-wise LR ──────────────────────
    # Backbone (layer awal) → LR kecil, jangan rusak fitur ImageNet
    # Head/Classifier       → LR besar, perlu adaptasi ke dataset DR
    backbone_params, head_params = [], []
    for name, param in model.named_parameters():
        # [FIX] Sertakan layer MSAG baru (gate/proj) agar dilatih dgn LR penuh
        MSAG_KEYS = ["gate_fine", "gate_mid", "proj_fine", "proj_mid", "attention"]
        if any(k in name for k in ["classifier", "head", "fc"] + MSAG_KEYS):
            head_params.append(param)
        else:
            backbone_params.append(param)

    optimizer = optim.AdamW([
        {"params": backbone_params, "lr": cfg["lr"] * cfg["backbone_lr_mult"]},
        {"params": head_params,     "lr": cfg["lr"]},
    ], weight_decay=cfg["weight_decay"])

    # ── Scheduler: OneCycleLR ─────────────────────────
    scheduler = OneCycleLR(
        optimizer,
        max_lr           = cfg["max_lr"],
        epochs           = cfg["epochs"],
        steps_per_epoch  = steps,
        pct_start        = cfg["pct_start"],
        anneal_strategy  = "cos",
        div_factor       = cfg["div_factor"],
        final_div_factor = cfg["final_div_factor"],
        cycle_momentum   = False,
    )

    # ── Scaler untuk Mixed Precision ──────────────────
    scaler = torch.amp.GradScaler("cuda") if torch.cuda.is_available() \
             else torch.amp.GradScaler("cpu")

    # ── Summary ───────────────────────────────────────
    f_pct = int(cfg["focal_loss_weight"] * 100)
    c_pct = int(cfg["ce_loss_weight"] * 100)
    print(f"\n  OneCycleLR       : steps/ep={steps} | max_lr={cfg['max_lr']:.2e}")
    print(f"  Loss             : CombinedLoss (FocalOrdinal {f_pct}% + LabelSmooth {c_pct}%)")
    print(f"  Optimizer        : AdamW LayerWise "
          f"(backbone lr={cfg['lr']*cfg['backbone_lr_mult']:.1e}, head lr={cfg['lr']:.1e})")
    print(f"  Ordinal Weight   : {cfg['ordinal_weight']}")
    print(f"  Label Smoothing  : {cfg['label_smoothing']}")
    print(f"  Class-Balanced Weights terpasang (Boosted C3/C4) ✅")

    LOG.info("Loss + Optimizer + Scheduler → Lanjut ke Cell 8.1")
    return criterion, optimizer, scheduler, scaler


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║ CELL 8.1 — Visualisasi LR Schedule                   ║
# ╚══════════════════════════════════════════════════════╝
def plot_lr_schedule(cfg):
    dm    = torch.nn.Linear(1,1)
    do    = torch.optim.AdamW(dm.parameters(), lr=cfg["lr"])
    steps = cfg.get("steps_per_epoch") or 80
    max_lr= cfg.get("max_lr", cfg["lr"] * 3)
    s     = OneCycleLR(do, max_lr=max_lr,
                       epochs=cfg["epochs"], steps_per_epoch=steps,
                       pct_start=cfg.get("pct_start", 0.15),
                       anneal_strategy="cos",
                       div_factor=cfg.get("div_factor", 10.0),
                       final_div_factor=cfg.get("final_div_factor", 1e3),
                       cycle_momentum=False)
    raw   = []
    for _ in range(cfg["epochs"] * steps):
        raw.append(do.param_groups[0]["lr"]); s.step()
    lrs = [sum(raw[i*steps:(i+1)*steps])/steps for i in range(cfg["epochs"])]

    fig, axes = plt.subplots(1, 2, figsize=(13,4))
    warmup_ep = max(1, int(cfg["epochs"] * cfg.get("pct_start", 0.15)))
    fig.suptitle(f"LR Schedule — OneCycleLR  max_lr={max_lr:.2e}  pct_start={cfg.get('pct_start',0.15)}", fontsize=13, fontweight="bold")
    eps = range(1, cfg["epochs"]+1)
    axes[0].plot(eps, lrs, "b-o", markersize=4)
    axes[0].axvline(x=warmup_ep, color="red", linestyle="--", label=f"Peak LR ep{warmup_ep}")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("LR"); axes[0].legend()
    axes[0].set_title("Linear Scale")

    axes[1].semilogy(eps, lrs, "g-o", markersize=4)
    axes[1].axvline(x=warmup_ep, color="red", linestyle="--", label=f"Peak LR ep{warmup_ep}")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("LR (log)"); axes[1].legend()
    axes[1].set_title("Log Scale")

    print("\n"+"="*68)
    print("  [CELL 8.1]  LR SCHEDULE -- WARMUP + COSINE DECAY")
    print("="*68)
    print(f"  Warmup : ep 1-{warmup_ep}    {max_lr/cfg.get('div_factor',10):.2e} -> {max_lr:.2e} (peak)")
    print(f"  Decay  : ep {warmup_ep+1}-{cfg['epochs']}   {max_lr:.2e} -> {max_lr/cfg.get('final_div_factor',1e3):.2e}")
    print("\n  Epoch snapshot (setiap 5 ep):")
    max_lr_val = max(lrs)
    for ei,lv in enumerate(lrs):
        if ei==0 or (ei+1)%5==0 or ei==len(lrs)-1:
            ph="Warmup" if ei<warmup_ep else "Decay "
            bar="#"*max(1,int(lv/max_lr_val*32))+"."*(32-max(1,int(lv/max_lr_val*32)))
            print(f"    ep{ei+1:>3}  {lv:.4e}  {ph:<8}  |{bar}|")
    print("="*68)
    plt.tight_layout()
    p=os.path.join(OUTPUT_DIR,"lr_schedule.png")
    plt.savefig(p,dpi=150); plt.show()
    LOG.info(f"LR schedule plot disimpan -> {os.path.basename(p)}")

_tmp_cfg = make_cfg()
plot_lr_schedule(_tmp_cfg)
del _tmp_cfg

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║ CELL 9 — Train & Validate (Bersih Tanpa 'cfg')       ║
# ╚══════════════════════════════════════════════════════╝
# FIX: device string untuk autocast & empty_cache agar aman di CPU
_AMP_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def train_one_epoch(model, loader, optimizer, criterion, scaler, scheduler=None, max_norm=1.0, mixup_alpha=0.0, model_ema=None):
    model.train()
    t_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels  = [], []
    for images, labels in tqdm(loader, desc="Train", leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        # ── Mixup ──────────────────────────────────────
        # Balanced (Intra-Class) MixUp: mix within the same class to preserve clinic labels
        use_mixup = mixup_alpha > 0 and np.random.rand() < 0.5
        if use_mixup:
            lam = float(np.random.beta(mixup_alpha, mixup_alpha))
            idx = []
            labels_b = labels.clone()
            for i in range(images.size(0)):
                label_i = labels[i].item()
                same_cls_indices = (labels == label_i).nonzero(as_tuple=True)[0]
                same_cls_indices_filtered = same_cls_indices[same_cls_indices != i]
                if len(same_cls_indices_filtered) > 0:
                    mix_idx = same_cls_indices_filtered[torch.randint(0, len(same_cls_indices_filtered), (1,)).item()]
                    idx.append(mix_idx)
                else:
                    mix_idx = torch.randint(0, images.size(0), (1,)).item()
                    idx.append(mix_idx)
                    labels_b[i] = labels[mix_idx]
            idx = torch.tensor(idx, device=DEVICE)
            images_m = lam * images + (1.0 - lam) * images[idx]
        else:
            images_m = images
        with torch.amp.autocast(_AMP_DEVICE):
            out = model(images_m)
            if use_mixup:
                loss = lam * criterion(out, labels) + (1.0 - lam) * criterion(out, labels_b)
            else:
                loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)
        scaler.step(optimizer); scaler.update()
        if model_ema is not None:
            model_ema.update(model)
        if scheduler is not None: scheduler.step()
        t_loss  += loss.item() * images.size(0)
        preds    = out.argmax(1)
        correct += (preds == labels).sum().item()
        total   += images.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    f1  = f1_score(all_labels, all_preds, labels=[0,1,2,3,4],
                   average="macro", zero_division=0)
    qwk = cohen_kappa_score(all_labels, all_preds,
                             labels=[0,1,2,3,4], weights="quadratic")
    return t_loss/total, correct/total, f1, qwk

def _tta_original(x):   return x
def _tta_hflip(x):      return torch.flip(x, dims=[3])
def _tta_vflip(x):      return torch.flip(x, dims=[2])
def _tta_rot90(x):      return torch.rot90(x, 1, [2, 3])
def _tta_rot270(x):     return torch.rot90(x, 3, [2, 3])
def _tta_rot90_hflip(x):return torch.flip(torch.rot90(x, 1, [2,3]), dims=[3])
def _tta_rot270_hflip(x):return torch.flip(torch.rot90(x, 3, [2,3]), dims=[3])
def _tta_rot180(x):     return torch.rot90(x, 2, [2, 3])

def tta_predict(model, images, n_tta=8):
    transforms_list = [
        _tta_original,
        _tta_hflip,
        _tta_vflip,
        _tta_rot90,
        _tta_rot270,
        _tta_rot90_hflip,
        _tta_rot270_hflip,
        _tta_rot180,
    ]
    probs_list = []
    for tf in transforms_list[:n_tta]:
        with torch.no_grad():
            with torch.amp.autocast(_AMP_DEVICE):
                p = torch.softmax(model(tf(images)), dim=1)
            probs_list.append(p)
    return torch.stack(probs_list).mean(dim=0)

def validate(model, loader, criterion, use_tta=False):
    model.eval()
    t_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels  = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Val", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            with torch.amp.autocast(_AMP_DEVICE):
                out  = model(images)
                loss = criterion(out, labels)
            t_loss  += loss.item() * images.size(0)

            if use_tta:
                probs = tta_predict(model, images)
                preds = probs.argmax(1)
            else:
                preds = out.argmax(1)

            correct += (preds == labels).sum().item()
            total   += images.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    f1  = f1_score(all_labels, all_preds, labels=[0,1,2,3,4],
                   average="macro", zero_division=0)
    qwk = cohen_kappa_score(all_labels, all_preds,
                             labels=[0,1,2,3,4], weights="quadratic")
    return t_loss/total, correct/total, f1, qwk

LOG.info("train & validate OK → Lanjut ke Cell 10")


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║ CELL 10 — Training Loop                           ║
# ╚══════════════════════════════════════════════════════╝
def training_loop(model, cfg, train_loader, val_loader, criterion,
                  optimizer, scheduler, scaler, log, resume_path=None):
    best_qwk    = -1.0
    best_acc    = -1.0
    best_path   = os.path.join(OUTPUT_DIR, f"best_{cfg['run_name']}.pth")
    no_improve  = 0
    start_epoch = 1
    best_epoch  = 1

    if resume_path and os.path.exists(resume_path):
        LOG.info(f"Resume dari: {os.path.basename(resume_path)}")
        ckpt = torch.load(resume_path, map_location=DEVICE, weights_only=False)
        clean_sd = {k: v for k, v in ckpt["state_dict"].items()
                    if not k.endswith(("total_ops","total_params"))}
        model.load_state_dict(clean_sd, strict=True)
        best_qwk    = ckpt.get("best_qwk", -1.0)
        best_acc    = ckpt.get("best_acc", -1.0)
        start_epoch = ckpt.get("epoch", 0) + 1
        best_epoch  = ckpt.get("epoch", 0)
        LOG.info(f"Resume ep {start_epoch} | Best QWK: {best_qwk:.4f} | Best Acc: {best_acc:.4f}")
    elif resume_path:
        LOG.warn(f"resume_path tidak ditemukan: {resume_path}")

    # ── Inisialisasi EMA jika diaktifkan ─────────────────
    use_ema = cfg.get("use_ema", False)
    model_ema = None
    if use_ema:
        from timm.utils import ModelEmaV2
        # [NOTE] EMA membuat deep copy model. Forward hook di MSAGLite._register_hooks()
        # terdaftar saat __init__, sehingga model_ema.module juga memiliki hook sendiri.
        # ModelEmaV2 hanya meng-update parameter (bukan forward), jadi hook tetap berjalan
        # normal saat validate(model_ema.module, ...) dipanggil. Aman.
        model_ema = ModelEmaV2(model, decay=cfg.get("ema_decay", 0.9999))
        LOG.info(f"Model EMA diaktifkan (decay={cfg.get('ema_decay', 0.9999)})")

    history = {k: [] for k in ["train_loss","val_loss","train_acc",
                                 "val_acc","train_f1","val_f1",
                                 "train_qwk","val_qwk","lr","epoch_sec"]}
    print("\n"+"="*100)
    print(f"  TRAINING START — {cfg['run_name']}")
    print("="*100)
    print(f"  Ep={cfg['epochs']} | Batch={cfg['batch_size']} | "
          f"Img={cfg['img_size']}px | patience={cfg['patience']}")
    print("-"*100)

    for epoch in range(start_epoch, cfg["epochs"]+1):
        # ── Freeze / Unfreeze Backbone ───────────────────
        freeze_ep = cfg.get("freeze_epochs", 0)
        if freeze_ep > 0:
            if epoch <= freeze_ep:
                # Freeze backbone, keep classifier/head/fc active
                for name, param in model.named_parameters():
                    # [FIX] Sertakan layer MSAG baru agar tetap aktif saat backbone freeze
                    MSAG_TRAIN_KEYS = ["classifier", "head", "fc", "gate_fine", "gate_mid", "proj_fine", "proj_mid", "attention"]
                    if any(k in name for k in MSAG_TRAIN_KEYS):
                        param.requires_grad = True
                    else:
                        param.requires_grad = False
                if epoch == start_epoch or epoch == freeze_ep:
                    LOG.info(f"Epoch {epoch} <= {freeze_ep}: Backbone FROZEN (hanya latih head)")
            else:
                # Unfreeze everything
                for param in model.parameters():
                    param.requires_grad = True
                if epoch == freeze_ep + 1:
                    LOG.info(f"Epoch {epoch} > {freeze_ep}: Backbone UNFROZEN (fine-tuning penuh)")
        # ─────────────────────────────────────────────────

        ep_start = time.time()
        # 🌟 SINKRONISASI DI SINI: Panggil train_one_epoch dengan parameter grad_clip dari config
        tl,ta,tf,tq = train_one_epoch(model, train_loader, optimizer,
                                       criterion, scaler, scheduler,
                                       max_norm=cfg.get("grad_clip", 1.0),
                                       mixup_alpha=cfg.get("mixup_alpha", 0.0),
                                       model_ema=model_ema)
        if model_ema is not None:
            vl,va,vf,vq = validate(model_ema.module, val_loader, criterion, use_tta=False)
            tag_ema = " [EMA]"
        else:
            vl,va,vf,vq = validate(model, val_loader, criterion, use_tta=False)
            tag_ema = ""
        lr  = optimizer.param_groups[0]["lr"]
        sec = time.time() - ep_start

        for k, v in zip(
            ["train_loss","val_loss","train_acc","val_acc",
             "train_f1","val_f1","train_qwk","val_qwk","lr","epoch_sec"],
            [tl, vl, ta, va, tf, vf, tq, vq, lr, sec]):
            history[k].append(v)

        # ── OPSI 1: Logika Pemilihan Best Epoch Aman & Balance ──
        is_best = False
        trigger_metric = ""
        trigger_val = 0.0
        trigger_extra = ""
        
        if va > best_acc:
            is_best = True
            trigger_metric = "Acc"
            trigger_val = va
        elif abs(va - best_acc) < 1e-7 and vq > best_qwk:
            is_best = True
            trigger_metric = "QWK"
            trigger_val = vq
            trigger_extra = "Acc equal"
        elif (best_acc - va) <= 0.005 and vq > best_qwk:
            is_best = True
            trigger_metric = "QWK"
            trigger_val = vq
            trigger_extra = f"Acc drop {best_acc - va:.4f}"

        if is_best:
            best_qwk = vq; best_acc = va; no_improve = 0; best_epoch = epoch
            active_model = model_ema.module if model_ema is not None else model
            sd = (active_model.module.state_dict() if isinstance(active_model, nn.DataParallel) else active_model.state_dict())
            torch.save({
                "epoch": epoch, "state_dict": sd,
                "best_qwk": best_qwk, "best_f1": vf,
                "best_acc": best_acc, "cfg": cfg,
                "timestamp": now_wib().isoformat()
            }, best_path)
            import inspect
            if len(inspect.signature(log.best).parameters) >= 4:
                log.best(epoch, trigger_metric, trigger_val, trigger_extra)
            else:
                log.best(epoch, trigger_val)
            tag = f"🏆 BEST ({trigger_metric}={trigger_val:.4f}){tag_ema}"
        else:
            no_improve += 1
            tag = f"[-{no_improve}/{cfg['patience']}]{tag_ema}"

        log.metric(epoch, cfg["epochs"], tl, vl, ta, va, tf, vf, tq, vq, lr, tag, sec)
        if torch.cuda.is_available(): torch.cuda.empty_cache()

        if no_improve >= cfg["patience"]:
            log.warn(f"Early stopping ep {epoch}"); break

    ae = len(history["train_loss"])
    print("-"*100)
    print(f"  SELESAI | BestEp={best_epoch} | BestQWK={best_qwk:.4f} | BestAcc={best_acc:.4f}")
    print(f"  EarlyStopping={'TRIGGERED ep='+str(ae) if ae<cfg['epochs'] else 'NOT triggered'}")
    print("="*100)
    log.done(f"Best QWK={best_qwk:.4f} | BestAcc={best_acc:.4f} | BestEp={best_epoch}")
    return history, best_path

LOG.info("training_loop OK → Lanjut ke Cell 11")

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║ CELL 11 — Evaluasi & Visualisasi (Full)              ║
# ╚══════════════════════════════════════════════════════╝

# ══════════════════════════════════════════════════════
# FUNGSI UTAMA: evaluate_split
# ══════════════════════════════════════════════════════

def evaluate_split(model_path, loader, cfg, split_name="Val"):
    ckpt       = torch.load(model_path, map_location=DEVICE, weights_only=False)
    eval_model = build_model(cfg)
    sd         = {k: v for k, v in ckpt["state_dict"].items()
                  if not k.endswith(("total_ops","total_params"))}
    eval_model.load_state_dict(sd, strict=True)
    eval_model.eval()

    all_preds, all_labels, all_probs = [], [], []
    bad_samples  = []
    good_samples = []

    # Read use_tta and n_tta from config
    use_tta = cfg["use_tta"]
    n_tta = cfg["n_tta"] if use_tta else 1

    LOG.info(f"Evaluasi {split_name} — TTA x{n_tta} {'aktif' if use_tta else 'nonaktif'}...")

    with torch.no_grad():
        for images, labels in tqdm(loader, desc=f"Inference {split_name}", leave=False):
            img_d = images.to(DEVICE)
            probs = tta_predict(eval_model, img_d, n_tta=n_tta)
            preds = probs.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

            # Kumpulkan bad samples (salah prediksi)
            mask_bad = preds.cpu() != labels
            for i in mask_bad.nonzero(as_tuple=True)[0]:
                if len(bad_samples) < 16:
                    bad_samples.append({
                        "img" : images[i],
                        "true": labels[i].item(),
                        "pred": preds[i].item(),
                        "prob": probs[i][preds[i]].item()
                    })

            # Kumpulkan good samples (benar prediksi, confidence tinggi)
            mask_good = preds.cpu() == labels
            for i in mask_good.nonzero(as_tuple=True)[0]:
                conf = probs[i][preds[i]].item()
                if len(good_samples) < 16 and conf >= 0.85:
                    good_samples.append({
                        "img" : images[i],
                        "true": labels[i].item(),
                        "pred": preds[i].item(),
                        "prob": conf
                    })

    all_probs = np.array(all_probs)
    y_true    = np.array(all_labels)
    y_pred    = np.array(all_preds)

    # ── Metrik Utama ──────────────────────────────────
    acc      = (y_true == y_pred).mean()
    qwk      = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    f1_macro = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    f1_w     = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    report   = classification_report(y_true, y_pred,
                                      target_names=CLASS_NAMES,
                                      output_dict=True)

    # ── ROC-AUC ───────────────────────────────────────
    y_bin = np.zeros((len(y_true), len(CLASS_NAMES)), dtype=np.float32)
    y_bin[np.arange(len(y_true)), y_true] = 1.0
    roc_auc = {}
    fpr_all = {}
    tpr_all = {}
    for i in range(5):
        fpr_i, tpr_i, _ = roc_curve(y_bin[:, i], all_probs[:, i])
        roc_auc[i]      = auc(fpr_i, tpr_i)
        fpr_all[i]      = fpr_i
        tpr_all[i]      = tpr_i
    macro_auc = float(np.mean(list(roc_auc.values())))

    # ── Per-class Accuracy ────────────────────────────
    per_class_acc = {}
    for ci in range(cfg["num_classes"]):
        mask = y_true == ci
        per_class_acc[CLASS_NAMES[ci]] = (
            round(float((y_pred[mask] == ci).sum() / mask.sum()), 4)
            if mask.sum() > 0 else 0.0)

    # ── Sensitivity & Specificity ─────────────────────
    sens_spec_results = []
    for cls_id in range(len(CLASS_NAMES)):
        y_true_bin = (y_true == cls_id).astype(int)
        y_pred_bin = (y_pred == cls_id).astype(int)
        TP = ((y_true_bin == 1) & (y_pred_bin == 1)).sum()
        TN = ((y_true_bin == 0) & (y_pred_bin == 0)).sum()
        FP = ((y_true_bin == 0) & (y_pred_bin == 1)).sum()
        FN = ((y_true_bin == 1) & (y_pred_bin == 0)).sum()
        sens_spec_results.append({
            "class_id"   : cls_id,
            "class_name" : CLASS_NAMES[cls_id],
            "TP": int(TP), "TN": int(TN),
            "FP": int(FP), "FN": int(FN),
            "sensitivity": round(TP / (TP + FN) if (TP + FN) > 0 else 0.0, 4),
            "specificity": round(TN / (TN + FP) if (TN + FP) > 0 else 0.0, 4),
            "ppv"        : round(TP / (TP + FP) if (TP + FP) > 0 else 0.0, 4),
            "npv"        : round(TN / (TN + FN) if (TN + FN) > 0 else 0.0, 4),
        })

    # ══════════════════════════════════════════════════
    # PRINT — E: Classification Report Lengkap
    # ══════════════════════════════════════════════════
    print(f"\n{'═'*75}")
    print(f"  HASIL {split_name.upper()} — {cfg['run_name']}")
    print(f"{'═'*75}")
    print(f"  {'Accuracy':<20} {acc*100:>9.2f}%")
    print(f"  {'QWK':<20} {qwk:>10.4f}")
    print(f"  {'F1-Macro':<20} {f1_macro:>10.4f}")
    print(f"  {'F1-Weighted':<20} {f1_w:>10.4f}")
    print(f"  {'Macro-AUC':<20} {macro_auc:>10.4f}")
    print(f"{'─'*75}")

    # E — Classification report per kelas
    print(f"  {'Kelas':<22} {'Prec':>7} {'Rec':>7} {'F1':>7} "
          f"{'AUC':>7} {'Acc':>7} {'Support':>8}")
    print(f"  {'-'*70}")
    for i, name in enumerate(CLASS_NAMES):
        r   = report[name]
        acc_i = per_class_acc[name]
        print(f"  {name:<22} {r['precision']:>7.4f} {r['recall']:>7.4f} "
              f"{r['f1-score']:>7.4f} {roc_auc[i]:>7.4f} "
              f"{acc_i:>7.4f} {int(r['support']):>8}")
    print(f"{'─'*75}")

    # A — Per-class detail: benar vs salah
    print(f"\n  PER-CLASS DETAIL (Benar vs Salah)")
    print(f"  {'-'*70}")
    print(f"  {'Kelas':<22} {'Total':>7} {'Benar':>7} {'Salah':>7} {'Acc%':>8}")
    print(f"  {'-'*70}")
    for ci, name in enumerate(CLASS_NAMES):
        mask    = y_true == ci
        total   = mask.sum()
        benar   = (y_pred[mask] == ci).sum()
        salah   = total - benar
        acc_pct = benar / total * 100 if total > 0 else 0.0
        flag    = " ⚠️" if acc_pct < 70.0 else ""
        print(f"  {name:<22} {total:>7} {benar:>7} {salah:>7} {acc_pct:>7.2f}%{flag}")
    print(f"{'═'*75}")

    # Sensitivity & Specificity tabel
    print(f"\n{'═'*90}")
    print(f"  SENSITIVITY & SPECIFICITY — {split_name}")
    print(f"{'═'*90}")
    print(f"  {'Kelas':<22} {'Sens':>8} {'Spec':>8} {'PPV':>8} {'NPV':>8} "
          f"{'TP':>5} {'TN':>5} {'FP':>5} {'FN':>5}")
    print(f"  {'-'*88}")
    sens_list, spec_list = [], []
    for r in sens_spec_results:
        flag = " ⚠️" if r["sensitivity"] < 0.70 else ""
        print(f"  {r['class_name']:<22} "
              f"{r['sensitivity']:>8.4f} {r['specificity']:>8.4f} "
              f"{r['ppv']:>8.4f} {r['npv']:>8.4f} "
              f"{r['TP']:>5} {r['TN']:>5} {r['FP']:>5} {r['FN']:>5}{flag}")
        sens_list.append(r["sensitivity"])
        spec_list.append(r["specificity"])
    print(f"  {'-'*88}")
    print(f"  {'Macro Average':<22} "
          f"{np.mean(sens_list):>8.4f} {np.mean(spec_list):>8.4f}")
    print(f"{'═'*90}")
    print(f"  ⚠️ = Sensitivity < 0.70 → perlu perhatian klinis")
    print(f"  Sens=Sensitivity | Spec=Specificity | PPV=Precision | NPV=Neg.Predictive.Value")
    print(f"{'═'*90}")

    # ══════════════════════════════════════════════════
    # SIMPAN CSV
    # ══════════════════════════════════════════════════
    df_report = pd.DataFrame(report).transpose().round(4)
    df_report.to_csv(os.path.join(OUTPUT_DIR,
                     f"report_{split_name}_{cfg['run_name']}.csv"))

    pd.DataFrame(sens_spec_results).assign(
        run_name=cfg["run_name"], split=split_name
    ).to_csv(os.path.join(OUTPUT_DIR,
             f"sens_spec_{split_name}_{cfg['run_name']}.csv"), index=False)

    if split_name == "Test":
        np.save(os.path.join(OUTPUT_DIR,
                f"yprob_{split_name}_{cfg['run_name']}.npy"), all_probs)
        np.save(os.path.join(OUTPUT_DIR,
                f"ytrue_{split_name}_{cfg['run_name']}.npy"), y_true)
        LOG.info(f"y_prob & y_true disimpan → npy")

    LOG.info(f"CSV report & sens_spec disimpan")

    del eval_model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    return {
        "acc"           : round(float(acc), 4),
        "qwk"           : round(float(qwk), 4),
        "f1_macro"      : round(float(f1_macro), 4),
        "f1_weighted"   : round(float(f1_w), 4),
        "macro_auc"     : round(float(macro_auc), 4),
        "roc_auc"       : roc_auc,
        "fpr_all"       : fpr_all,
        "tpr_all"       : tpr_all,
        "y_true"        : y_true,
        "y_pred"        : y_pred,
        "y_prob"        : all_probs,
        "bad_samples"   : bad_samples,
        "good_samples"  : good_samples,
        "report"        : report,
        "per_class_acc" : per_class_acc,
        "sens_spec"     : sens_spec_results,
    }


# ══════════════════════════════════════════════════════
# A+D — Visualisasi Bad Samples & Good Samples
# ══════════════════════════════════════════════════════

# BARU — fix:
def _plot_samples(samples, cfg, split_name, title, color):
    if not samples:
        LOG.info(f"Tidak ada sampel untuk ditampilkan: {title}")
        return
    n    = len(samples)
    cols = min(n, 4)
    rows = math.ceil(n / cols)
    mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std  = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    fig.suptitle(f"{title} — {split_name} | {cfg['run_name']} ({n} sampel)",
                 fontsize=13, fontweight="bold")

    # FIX: simpan sebagai list dulu sekali, pakai berulang
    ax_list = list(axes.flat) if hasattr(axes, "flat") else [axes]

    # Matikan semua axes dulu
    for ax in ax_list:
        ax.axis("off")

    # Isi axes satu per satu
    for idx, s in enumerate(samples):
        ax  = ax_list[idx]       # ← pakai list yang sama, tidak habis
        img = np.clip(s["img"].numpy() * std + mean, 0, 1)
        img = np.transpose(img, (1, 2, 0))
        ax.imshow(img)
        ax.set_title(
            f"True: {CLASS_NAMES[s['true']]}\n"
            f"Pred: {CLASS_NAMES[s['pred']]}\n"
            f"Conf: {s['prob']:.2%}",
            fontsize=9, color=color
        )
        ax.axis("off")

    plt.tight_layout()
    fname = title.lower().replace(" ", "_")
    p = os.path.join(OUTPUT_DIR, f"{fname}_{split_name}_{cfg['run_name']}.png")
    plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.close("all")
    LOG.info(f"{title} plot → {os.path.basename(p)}")


def visualize_bad_samples(bad_samples, cfg, split_name="Test"):
    _plot_samples(bad_samples, cfg, split_name,
                  title="Misclassified Samples", color="red")


def visualize_good_samples(good_samples, cfg, split_name="Test"):
    _plot_samples(good_samples, cfg, split_name,
                  title="Correct High-Confidence Samples", color="green")


# ══════════════════════════════════════════════════════
# B — Confusion Matrix di Cell 11
# ══════════════════════════════════════════════════════

def plot_confusion_matrix_cell11(y_true, y_pred, cfg, split_name="Test"):
    cm       = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
    row_sums = cm.sum(axis=1)[:, np.newaxis]
    cm_norm  = np.where(row_sums > 0, cm.astype("float") / row_sums, 0.0)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle(f"Confusion Matrix — {cfg['run_name']} ({split_name})",
                 fontsize=13, fontweight="bold")

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                cbar_kws={"label": "Jumlah Sampel"}, square=True,
                annot_kws={"size": 11, "weight": "bold"})
    axes[0].set_title("Absolut", fontweight="bold")
    axes[0].set_xlabel("Predicted", fontweight="bold")
    axes[0].set_ylabel("True",      fontweight="bold")

    sns.heatmap(cm_norm, annot=True, fmt=".1%", cmap="YlOrRd", ax=axes[1],
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                cbar_kws={"label": "Rasio"}, square=True,
                vmin=0.0, vmax=1.0,
                annot_kws={"size": 11, "weight": "bold"})
    axes[1].set_title("Normalisasi", fontweight="bold")
    axes[1].set_xlabel("Predicted",  fontweight="bold")
    axes[1].set_ylabel("True",       fontweight="bold")

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, f"cm_{split_name}_{cfg['run_name']}.png")
    plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")
    LOG.info(f"Confusion Matrix → {os.path.basename(p)}")

    pd.DataFrame(cm, index=CLASS_NAMES,
                 columns=CLASS_NAMES).to_csv(
        os.path.join(OUTPUT_DIR, f"cm_{split_name}_{cfg['run_name']}.csv"))


# ══════════════════════════════════════════════════════
# C — ROC Curve per Kelas
# ══════════════════════════════════════════════════════

def plot_roc_curve_per_class(fpr_all, tpr_all, roc_auc, cfg, split_name="Test"):
    colors = ["#E74C3C", "#3498DB", "#2ECC71", "#F39C12", "#9B59B6"]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"ROC Curve — {cfg['run_name']} ({split_name})",
                 fontsize=13, fontweight="bold")

    # Plot 1 — ROC per kelas terpisah
    for i, name in enumerate(CLASS_NAMES):
        axes[0].plot(fpr_all[i], tpr_all[i],
                     color=colors[i], lw=2,
                     label=f"{name} (AUC={roc_auc[i]:.4f})")
    axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="Random")
    axes[0].set_title("ROC per Kelas", fontweight="bold")
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate")
    axes[0].legend(loc="lower right", fontsize=8)
    axes[0].grid(alpha=0.3)

    # Plot 2 — Semua kelas dalam satu panel + macro average disorot
    macro_auc_val = np.mean(list(roc_auc.values()))
    for i, name in enumerate(CLASS_NAMES):
        axes[1].plot(fpr_all[i], tpr_all[i],
                     color=colors[i], lw=1.5, alpha=0.7,
                     label=f"C{i} AUC={roc_auc[i]:.3f}")
    axes[1].plot([0, 1], [0, 1], "k--", lw=1)
    axes[1].set_title(f"Semua Kelas | Macro-AUC={macro_auc_val:.4f}",
                      fontweight="bold")
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].legend(loc="lower right", fontsize=8)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, f"roc_{split_name}_{cfg['run_name']}.png")
    plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.close("all")
    LOG.info(f"ROC Curve → {os.path.basename(p)}")


# ══════════════════════════════════════════════════════
# F — Confidence Distribution
# ══════════════════════════════════════════════════════

def plot_confidence_distribution(y_true, y_pred, y_prob, cfg, split_name="Test"):
    correct_conf = []
    wrong_conf   = []

    for i in range(len(y_true)):
        conf = y_prob[i][y_pred[i]]
        if y_true[i] == y_pred[i]:
            correct_conf.append(conf)
        else:
            wrong_conf.append(conf)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Confidence Distribution — {cfg['run_name']} ({split_name})",
                 fontsize=13, fontweight="bold")

    # Plot 1 — Histogram benar vs salah
    bins = np.linspace(0, 1, 25)
    axes[0].hist(correct_conf, bins=bins, color="#2ECC71",
                 alpha=0.7, label=f"Benar (n={len(correct_conf)})")
    axes[0].hist(wrong_conf,   bins=bins, color="#E74C3C",
                 alpha=0.7, label=f"Salah (n={len(wrong_conf)})")
    axes[0].set_title("Histogram: Benar vs Salah", fontweight="bold")
    axes[0].set_xlabel("Confidence")
    axes[0].set_ylabel("Jumlah Sampel")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Plot 2 — Confidence per kelas (boxplot)
    conf_per_class = []
    for ci in range(len(CLASS_NAMES)):
        mask = y_true == ci
        if mask.sum() > 0:
            conf_per_class.append(y_prob[mask, ci])
        else:
            conf_per_class.append(np.array([0.0]))

    axes[1].boxplot(conf_per_class, labels=CLASS_NAMES, patch_artist=True,
                    boxprops=dict(facecolor="#3498DB", alpha=0.6))
    axes[1].set_title("Confidence per Kelas (True Label)", fontweight="bold")
    axes[1].set_xlabel("Kelas")
    axes[1].set_ylabel("Confidence")
    axes[1].tick_params(axis="x", rotation=15)
    axes[1].grid(alpha=0.3)

    # Plot 3 — Confidence salah prediksi per kelas
    wrong_conf_per_class = {name: [] for name in CLASS_NAMES}
    for i in range(len(y_true)):
        if y_true[i] != y_pred[i]:
            wrong_conf_per_class[CLASS_NAMES[y_true[i]]].append(y_prob[i][y_pred[i]])

    means  = [np.mean(v) if v else 0.0
              for v in wrong_conf_per_class.values()]
    colors = ["#E74C3C" if m > 0.7 else "#F39C12" if m > 0.5 else "#2ECC71"
              for m in means]
    bars = axes[2].bar(CLASS_NAMES, means, color=colors, alpha=0.85)
    axes[2].set_title("Rata-rata Confidence saat Salah\n(Tinggi = Berbahaya)",
                      fontweight="bold")
    axes[2].set_xlabel("Kelas True")
    axes[2].set_ylabel("Mean Confidence saat Salah")
    axes[2].tick_params(axis="x", rotation=15)
    axes[2].set_ylim(0, 1.1)
    axes[2].grid(alpha=0.3, axis="y")
    axes[2].axhline(y=0.7, color="red", linestyle="--",
                    alpha=0.5, label="Threshold 0.7")
    axes[2].legend(fontsize=8)
    for bar, m in zip(bars, means):
        if m > 0:
            axes[2].text(bar.get_x() + bar.get_width()/2,
                         m + 0.02, f"{m:.2f}",
                         ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, f"conf_dist_{split_name}_{cfg['run_name']}.png")
    plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.close("all")
    LOG.info(f"Confidence Distribution → {os.path.basename(p)}")


# ══════════════════════════════════════════════════════
# SENS/SPEC — Plot Visualisasi
# ══════════════════════════════════════════════════════

def plot_sensitivity_specificity(sens_spec_results, cfg, split_name="Test"):
    names     = [r["class_name"]  for r in sens_spec_results]
    sens_vals = [r["sensitivity"] for r in sens_spec_results]
    spec_vals = [r["specificity"] for r in sens_spec_results]
    ppv_vals  = [r["ppv"]         for r in sens_spec_results]
    npv_vals  = [r["npv"]         for r in sens_spec_results]

    x     = np.arange(len(names))
    width = 0.2

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"Sensitivity & Specificity — {cfg['run_name']} ({split_name})",
                 fontsize=13, fontweight="bold")

    # Plot 1
    b1 = axes[0].bar(x - width/2, sens_vals, width,
                     label="Sensitivity", color="#E74C3C", alpha=0.85)
    b2 = axes[0].bar(x + width/2, spec_vals, width,
                     label="Specificity", color="#2ECC71", alpha=0.85)
    axes[0].set_title("Sensitivity vs Specificity", fontweight="bold")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(names, rotation=15, ha="right")
    axes[0].set_ylim(0, 1.15)
    axes[0].axhline(y=0.70, color="red", linestyle="--",
                    alpha=0.5, label="Threshold 0.70")
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3, axis="y")
    for bar in list(b1) + list(b2):
        h = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2,
                     h + 0.01, f"{h:.2f}",
                     ha="center", va="bottom", fontsize=8)

    # Plot 2
    b3 = axes[1].bar(x - width/2, ppv_vals, width,
                     label="PPV (Precision)", color="#3498DB", alpha=0.85)
    b4 = axes[1].bar(x + width/2, npv_vals, width,
                     label="NPV", color="#9B59B6", alpha=0.85)
    axes[1].set_title("PPV vs NPV", fontweight="bold")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(names, rotation=15, ha="right")
    axes[1].set_ylim(0, 1.15)
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3, axis="y")
    for bar in list(b3) + list(b4):
        h = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2,
                     h + 0.01, f"{h:.2f}",
                     ha="center", va="bottom", fontsize=8)

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR,
                     f"sens_spec_{split_name}_{cfg['run_name']}.png")
    plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.close("all")
    LOG.info(f"Sens/Spec plot → {os.path.basename(p)}")


# ══════════════════════════════════════════════════════
# WRAPPER — Panggil semua visualisasi sekaligus
# ══════════════════════════════════════════════════════

def run_full_evaluation(best_path, val_loader, test_loader, cfg, show_plots=True):
    val_metrics_out  = None
    test_metrics_out = None

    for split_name, loader in [("Val", val_loader), ("Test", test_loader)]:
        LOG.section(f"FULL EVALUATION — {split_name}")

        metrics = evaluate_split(best_path, loader, cfg, split_name)

        # simpan masing-masing
        if split_name == "Val":
            val_metrics_out = metrics
        else:
            test_metrics_out = metrics

        # B — Confusion Matrix
        plot_confusion_matrix_cell11(
            metrics["y_true"], metrics["y_pred"], cfg, split_name)

        if not show_plots:
            plt.ioff()

        # C — ROC Curve per kelas
        plot_roc_curve_per_class(
            metrics["fpr_all"], metrics["tpr_all"],
            metrics["roc_auc"], cfg, split_name)

        # D — Good samples
        visualize_good_samples(metrics["good_samples"], cfg, split_name)

        # A+bad — Bad samples
        visualize_bad_samples(metrics["bad_samples"], cfg, split_name)

        # F — Confidence distribution
        plot_confidence_distribution(
            metrics["y_true"], metrics["y_pred"],
            metrics["y_prob"], cfg, split_name)

        # Sens/Spec plot
        plot_sensitivity_specificity(
            metrics["sens_spec"], cfg, split_name)

        if not show_plots:
            plt.ion()

    return val_metrics_out, test_metrics_out




# ══════════════════════════════════════════════════════
# THRESHOLD OPTIMIZATION — Cari boost probabilitas per kelas
# ══════════════════════════════════════════════════════

def find_best_threshold(val_probs, val_labels,
                        search_classes=(1, 3, 4),
                        boost_max=0.20, step=0.01):
    """
    Greedy search: cari boost optimal di VAL SET per kelas minoritas menggunakan Akurasi Global.
    Jika akurasi sama, QWK digunakan sebagai tie-breaker untuk membantu sensitivitas.
    Returns: best_boost (array), best_val_acc (float)
    """
    n_cls      = val_probs.shape[1]
    best_acc   = (val_probs.argmax(axis=1) == val_labels).mean()
    best_qwk   = cohen_kappa_score(val_labels, val_probs.argmax(axis=1), weights="quadratic")
    best_boost = np.zeros(n_cls)

    for cls in search_classes:
        cls_best_acc   = best_acc
        cls_best_qwk   = best_qwk
        cls_best_boost = 0.0
        for b in np.arange(0.0, boost_max + step, step):
            trial = best_boost.copy(); trial[cls] = b
            preds = (val_probs + trial).argmax(axis=1)
            acc   = (preds == val_labels).mean()
            qwk   = cohen_kappa_score(val_labels, preds, weights="quadratic")
            if acc > cls_best_acc or (abs(acc - cls_best_acc) < 1e-7 and qwk > cls_best_qwk):
                cls_best_acc   = acc
                cls_best_qwk   = qwk
                cls_best_boost = b
        best_boost[cls] = cls_best_boost
        best_acc        = cls_best_acc
        best_qwk        = cls_best_qwk

    return best_boost, best_acc


def apply_threshold_and_evaluate(val_probs, val_labels,
                                  test_probs, test_labels,
                                  cfg, split_name="Test"):
    """
    1. Cari threshold optimal di val set (tidak sentuh test!)
    2. Apply ke test set
    3. Print perbandingan sebelum & sesudah
    """
    print(f"\n{'═'*65}")
    print(f"  THRESHOLD OPTIMIZATION — {cfg['run_name']}")
    print(f"{'═'*65}")

    base_val_acc  = (val_probs.argmax(axis=1)  == val_labels).mean()
    base_test_acc = (test_probs.argmax(axis=1) == test_labels).mean()
    print(f"  Baseline Val  Acc : {base_val_acc*100:.2f}%")
    print(f"  Baseline Test Acc : {base_test_acc*100:.2f}%")

    best_boost, boosted_val_acc = find_best_threshold(val_probs, val_labels)

    print(f"\n  Optimal Boost per Kelas:")
    for i, name in enumerate(CLASS_NAMES):
        if best_boost[i] > 0:
            print(f"    {name:<20}: +{best_boost[i]:.3f}")

    adj_test         = test_probs + best_boost
    boosted_preds    = adj_test.argmax(axis=1)
    boosted_test_acc = (boosted_preds == test_labels).mean()
    boosted_qwk      = cohen_kappa_score(test_labels, boosted_preds,
                                          labels=[0,1,2,3,4], weights="quadratic")
    boosted_f1       = f1_score(test_labels, boosted_preds,
                                 average="macro", zero_division=0)

    print(f"\n  Val  Acc : {base_val_acc*100:.2f}% → {boosted_val_acc*100:.2f}%  (Δ{(boosted_val_acc-base_val_acc)*100:+.2f}%)")
    print(f"  Test Acc : {base_test_acc*100:.2f}% → {boosted_test_acc*100:.2f}%  (Δ{(boosted_test_acc-base_test_acc)*100:+.2f}%)")
    print(f"  Test QWK : {boosted_qwk:.4f}")
    print(f"  Test F1  : {boosted_f1:.4f}")

    print(f"\n  Per-class Accuracy setelah Threshold Boost:")
    print(f"  {'Kelas':<20} {'Sebelum':>9} {'Sesudah':>9} {'Δ':>7}")
    print(f"  {'-'*50}")
    for ci, name in enumerate(CLASS_NAMES):
        mask = test_labels == ci
        if mask.sum() == 0: continue
        before = (test_probs.argmax(axis=1)[mask] == ci).mean()
        after  = (boosted_preds[mask] == ci).mean()
        flag   = " ⚠️" if after < 0.70 else ""
        print(f"  {name:<20} {before*100:>8.1f}% {after*100:>8.1f}% {(after-before)*100:>+6.1f}%{flag}")
    print(f"{'═'*65}")

    pd.DataFrame({
        "class_id": range(len(CLASS_NAMES)),
        "class_name": CLASS_NAMES,
        "boost": list(best_boost),
    }).to_csv(os.path.join(OUTPUT_DIR, f"threshold_boost_{cfg['run_name']}.csv"), index=False)
    LOG.info(f"Threshold boost CSV → threshold_boost_{cfg['run_name']}.csv")

    return boosted_preds, best_boost, boosted_test_acc


LOG.info("Cell 11 OK → semua fungsi evaluasi siap")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Fungsi Plot History
# ─────────────────────────────────────────────────────────────────────────
def plot_history(history, cfg):
    """Plot training history (loss, acc, f1, qwk, lr) dan simpan CSV."""
    epochs = list(range(1, len(history["train_loss"]) + 1))

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"Training History — {cfg['run_name']}", fontsize=14, fontweight="bold")

    pairs = [
        ("train_loss",  "val_loss",  "Loss",      axes[0, 0]),
        ("train_acc",   "val_acc",   "Accuracy",  axes[0, 1]),
        ("train_f1",    "val_f1",    "F1-Macro",  axes[0, 2]),
        ("train_qwk",   "val_qwk",   "QWK",       axes[1, 0]),
    ]
    for tr_key, va_key, title, ax in pairs:
        ax.plot(epochs, history[tr_key], "b-o", markersize=3, label="Train")
        ax.plot(epochs, history[va_key], "r-o", markersize=3, label="Val")
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.3)

    axes[1, 1].plot(epochs, history["lr"], "g-o", markersize=3)
    axes[1, 1].set_title("Learning Rate", fontweight="bold")
    axes[1, 1].set_xlabel("Epoch"); axes[1, 1].set_yscale("log"); axes[1, 1].grid(alpha=0.3)

    axes[1, 2].plot(epochs, history["epoch_sec"], "m-o", markersize=3)
    axes[1, 2].set_title("Epoch Time (s)", fontweight="bold")
    axes[1, 2].set_xlabel("Epoch"); axes[1, 2].grid(alpha=0.3)

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, f"history_{cfg['run_name']}.png")
    plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show(); plt.close("all")
    LOG.info(f"History plot → {os.path.basename(p)}")

    # Simpan CSV history agar Cell 13 bisa membaca per-epoch metrics
    df_hist = pd.DataFrame({
        "epoch":      epochs,
        "train_loss": history["train_loss"],
        "val_loss":   history["val_loss"],
        "train_acc":  history["train_acc"],
        "val_acc":    history["val_acc"],
        "train_f1":   history["train_f1"],
        "val_f1":     history["val_f1"],
        "train_qwk":  history["train_qwk"],
        "val_qwk":    history["val_qwk"],
        "lr":         history["lr"],
        "epoch_sec":  history["epoch_sec"],
    })
    csv_p = os.path.join(OUTPUT_DIR, f"history_{cfg['run_name']}.csv")
    df_hist.to_csv(csv_p, index=False)
    LOG.info(f"History CSV → {os.path.basename(csv_p)}")


# ╔══════════════════════════════════════════════════════╗
# ║ CELL 12 — Multi-Experiment Loop                      ║
# ╚══════════════════════════════════════════════════════╝
EXPERIMENT_COMBINATIONS = [
    {"use_clahe": True,  "use_aug": True },
    {"use_clahe": False, "use_aug": True },
    {"use_clahe": True,  "use_aug": False},
    {"use_clahe": False, "use_aug": False},
]


all_experiment_results = []
flops_str = params_str = model_size_mb = inf_time_ms = None

# FIX: Isi path .pth untuk resume training dari checkpoint, atau biarkan None untuk mulai dari awal
RESUME_FROM = None  # Contoh: "/kaggle/working/best_VGG16_CLAHE_AUG_FOCAL.pth"

for exp_idx, combo in enumerate(EXPERIMENT_COMBINATIONS, 1):
    active_cfg = make_cfg(
        model_key  = ACTIVE_MODEL,
        use_clahe  = combo["use_clahe"],
        use_aug    = combo["use_aug"],
    )
    # ─── Gunakan split original dari dataset (val & test tidak disentuh) ───
    # create_stratified_split dihapus — val & test pakai CSV asli dari PATHS
    exp_log = Logger(active_cfg["run_name"])
    set_seed(active_cfg["seed"])

    cs  = "ON" if combo["use_clahe"] else "OFF"
    as_ = "ON" if combo["use_aug"]   else "OFF"
    print("\n"+"="*100)
    print(f"  EKSPERIMEN {exp_idx}/4 — {active_cfg['run_name']}")
    print("="*100)
    print(f"  Model={active_cfg['model_name']} | IMG={active_cfg['img_size']}px | "
          f"BATCH={active_cfg['batch_size']} | CLAHE={cs} | Aug={as_} | Ep={active_cfg['epochs']}")
    print("-"*100)

    # Jalankan pre-compute CLAHE/Resize untuk konfigurasi aktif ini
    active_cache_dir = precompute_clahe(active_cfg, force=False, num_workers=active_cfg.get("num_workers", 4))

    # Dapatkan Loader
    train_loader, val_loader, test_loader, df_train = build_loaders(active_cfg, cache_dir=active_cache_dir)
    active_cfg["steps_per_epoch"] = len(train_loader)

    # Bangun Model
    model = build_model(active_cfg)

    # Pengukuran Efisiensi (Profil Komputasi Model)
    if exp_idx == 1:
        dummy = torch.zeros(1, 3, active_cfg["img_size"], active_cfg["img_size"]).to(DEVICE)
        flops, params = profile(model, inputs=(dummy,), verbose=False)
        flops_str, params_str = clever_format([flops, params], "%.2f")
        ps = sum(p.nelement()*p.element_size() for p in model.parameters())
        bs = sum(b.nelement()*b.element_size() for b in model.buffers())
        model_size_mb = (ps+bs)/1024**2
        model.eval()
        with torch.no_grad():
            for _ in range(10): model(dummy)
            if torch.cuda.is_available(): torch.cuda.synchronize()
        ts = []
        with torch.no_grad():
            for _ in range(100):
                t0 = time.perf_counter(); model(dummy)
                if torch.cuda.is_available(): torch.cuda.synchronize()
                ts.append(time.perf_counter()-t0)
        inf_time_ms = sum(ts)/len(ts)*1000
        print(f"\n  FLOPs={flops_str} | Params={params_str} | Size={model_size_mb:.1f}MB | Inf={inf_time_ms:.2f}ms/img")

    # Inisialisasi komponen latihan
    criterion, optimizer, scheduler, scaler = build_training_components(model, active_cfg)

    # Proses Training Loop Utama
    exp_start   = time.time()
    resume_from = active_cfg.get("resume_path", None)
    history, best_path = training_loop(
        model, active_cfg, train_loader, val_loader,
        criterion, optimizer, scheduler, scaler,
        exp_log, resume_path=resume_from)

    val_metrics, test_metrics = run_full_evaluation(best_path, val_loader, test_loader, active_cfg, show_plots=False)

    # ── Threshold Optimization (search di val, apply ke test) ──────────────
    if val_metrics is not None and test_metrics is not None:
        _vp = np.array(val_metrics["y_prob"])
        _vl = np.array(val_metrics["y_true"])
        _tp = np.array(test_metrics["y_prob"])
        _tl = np.array(test_metrics["y_true"])
        _, _boost, _boosted_acc = apply_threshold_and_evaluate(
            _vp, _vl, _tp, _tl, active_cfg, split_name="Test")

    # Plot visualisasi history
    plot_history(history, active_cfg)

    # Catat seluruh hasil metrik
    training_time_min = (time.time() - exp_start) / 60
    best_epoch_idx    = history["val_qwk"].index(max(history["val_qwk"]))

    # Filter test_metrics: hanya scalar (aman untuk CSV & DataFrame)
    _SKIP_KEYS = {"y_true", "y_pred", "y_prob", "bad_samples", "report",
                  "per_class_acc", "roc_auc"}
    test_metrics_scalar = {k: v for k, v in test_metrics.items()
                           if k not in _SKIP_KEYS}

    result_row = {
        "run_name"          : active_cfg["run_name"],
        "model_key"         : active_cfg["model_key"],
        "model_name"        : active_cfg["model_name"],
        "use_clahe"         : active_cfg["use_clahe"],
        "use_aug"           : active_cfg["use_aug"],
        "img_size"          : active_cfg["img_size"],
        "batch_size"        : active_cfg["batch_size"],
        "best_epoch"        : best_epoch_idx + 1,
        "val_qwk"           : max(history["val_qwk"]),
        "val_f1_macro"      : history["val_f1"][best_epoch_idx],
        "val_f1_weighted"   : val_metrics.get("f1_weighted", 0.0),
        "val_acc"           : history["val_acc"][best_epoch_idx],
        "val_macro_auc"     : val_metrics.get("macro_auc", 0.0),
        "flops"             : flops_str,
        "params"            : params_str,
        "model_size_mb"     : round(model_size_mb, 2) if model_size_mb else None,
        "inference_ms"      : round(inf_time_ms, 2)   if inf_time_ms  else None,
        "training_time_min" : round(training_time_min, 2),
        **{f"test_{k}": v for k, v in test_metrics_scalar.items()},
    }
    all_experiment_results.append(result_row)

    # Simpan sebagai CSV dinamis per-run
    csv_path = os.path.join(OUTPUT_DIR, f"results_{active_cfg['run_name']}.csv")
    pd.DataFrame([result_row]).to_csv(csv_path, index=False)
    exp_log.info(f"CSV saved → {os.path.basename(csv_path)}")

    # GC & MEMORY FLUSH
    del model, optimizer, scheduler, scaler, criterion
    del train_loader, val_loader, test_loader
    gc.collect()
    torch.cuda.empty_cache()

    try:
        libc = ctypes.CDLL("libc.so.6")
        libc.malloc_trim(0)
    except Exception:
        pass

    exp_log.done(f"{active_cfg['run_name']} selesai | Best QWK={max(history['val_qwk']):.4f}")


# ═════════════════════════════════════════════════════════════════════════─
# CELL 12 (lanjutan) — Confusion Matrix
# ═════════════════════════════════════════════════════════════════════════─


def generate_and_plot_confusion_matrix(model, loader, device, class_names, save_dir="/kaggle/working", run_name="MODEL_KITA"):
    """
    Fungsi untuk melakukan prediksi pada data loader, menghitung confusion matrix,
    serta menggambar heatmap absolut dan ternormalisasi secara side-by-side.
    """
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc=f"Mengumpulkan Prediksi ({run_name})", leave=False):
            images = images.to(device)
            amp_dev = "cuda" if torch.cuda.is_available() else "cpu"
            with torch.amp.autocast(amp_dev):
                outputs = model(images)
            preds = outputs.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    row_sums = cm.sum(axis=1)[:, np.newaxis]
    cm_norm = np.where(row_sums > 0, cm.astype('float') / row_sums, 0.0)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle(f"Analisis Confusion Matrix — {run_name}", fontsize=14, fontweight="bold", y=0.98)

    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
        xticklabels=class_names, yticklabels=class_names,
        cbar_kws={'label': 'Jumlah Sampel'}, square=True,
        annot_kws={"size": 11, "weight": "bold"}
    )
    axes[0].set_title("Confusion Matrix (Nilai Absolut)", fontsize=12, fontweight="bold", pad=10)
    axes[0].set_xlabel("Kelas Prediksi (Predicted)", fontsize=10, fontweight="bold")
    axes[0].set_ylabel("Kelas Sebenarnya (True)", fontsize=10, fontweight="bold")

    sns.heatmap(
        cm_norm, annot=True, fmt=".1%", cmap="YlOrRd", ax=axes[1],
        xticklabels=class_names, yticklabels=class_names,
        cbar_kws={'label': 'Rasio Akurasi'}, square=True,
        vmin=0.0, vmax=1.0, annot_kws={"size": 11, "weight": "bold"}
    )
    axes[1].set_title("Confusion Matrix (Normalisasi / Persentase)", fontsize=12, fontweight="bold", pad=10)
    axes[1].set_xlabel("Kelas Prediksi (Predicted)", fontsize=10, fontweight="bold")
    axes[1].set_ylabel("Kelas Sebenarnya (True)", fontsize=10, fontweight="bold")

    plt.tight_layout()

    image_path = os.path.join(save_dir, f"CM_Final_{run_name}.png")
    csv_path = os.path.join(save_dir, f"CM_Final_{run_name}.csv")

    plt.savefig(image_path, dpi=200, bbox_inches="tight")
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(csv_path)
    plt.show()
    plt.close('all')

    if 'LOG' in globals() and hasattr(globals()['LOG'], 'info'):
        globals()['LOG'].info(f"Confusion Matrix disimpan → {image_path}")
    else:
        print(f"\n🚀 Confusion Matrix disimpan → {image_path}")


def trigger_auto_confusion_matrix():
    device_to_use = globals().get('DEVICE', torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
    classes_to_use = globals().get('CLASS_NAMES', ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"])
    output_dir_to_use = globals().get('OUTPUT_DIR', "/kaggle/working")

    if 'model' in globals():
        loader_to_use = None
        loader_name_found = ""
        for name in ['test_loader', 'val_loader', 'train_loader']:
            if name in globals():
                loader_to_use = globals()[name]
                loader_name_found = name
                break

        if loader_to_use is not None:
            run_name_to_use = "MODEL_KITA"
            if 'active_cfg' in globals():
                run_name_to_use = globals()['active_cfg'].get("run_name", "MODEL_KITA")
            elif 'cfg' in globals():
                run_name_to_use = globals()['cfg'].get("run_name", "MODEL_KITA")

            print(f"\n🔍 [AUTO-RUN] Mendeteksi 'model' dan '{loader_name_found}' aktif di memori!")
            generate_and_plot_confusion_matrix(
                model=globals()['model'], loader=loader_to_use,
                device=device_to_use, class_names=classes_to_use,
                run_name=run_name_to_use
            )
            return

    pth_files = glob.glob(os.path.join(output_dir_to_use, "best_*.pth"))

    if pth_files and 'build_model' in globals() and 'build_loaders' in globals():
        print(f"\n🔍 [AUTO-RUN] Mendeteksi {len(pth_files)} model tersimpan di disk ({output_dir_to_use})")

        for pth_path in pth_files:
            run_name = os.path.basename(pth_path).replace("best_", "").replace(".pth", "")
            print(f"\n⏳ Memproses model: {run_name} ...")

            ckpt = torch.load(pth_path, map_location=device_to_use, weights_only=False)
            cfg = ckpt.get("cfg", None)
            if cfg is None:
                print(f"   ⚠️ Skip: Config tidak ditemukan di dalam {pth_path}")
                continue

            cfg_eval = cfg.copy()
            cfg_eval["cache_to_ram"] = False
            _, _, test_loader_d, _ = build_loaders(cfg_eval)

            eval_model = build_model(cfg)
            clean_sd = {k: v for k, v in ckpt["state_dict"].items() if not k.endswith(("total_ops","total_params"))}
            eval_model.load_state_dict(clean_sd, strict=True)
            eval_model.to(device_to_use)

            generate_and_plot_confusion_matrix(
                model=eval_model, loader=test_loader_d,
                device=device_to_use, class_names=classes_to_use,
                run_name=run_name
            )

            del eval_model, test_loader_d
            torch.cuda.empty_cache()
            import gc; gc.collect()
        return

    print("\n⚠️ [AUTO-RUN] Gagal menjalankan Confusion Matrix secara otomatis.")
    print("💡 Alasan: Tidak ada model di RAM maupun file 'best_*.pth' di disk.")
    print("👉 Jalankan proses training di Cell 12 terlebih dahulu!")

trigger_auto_confusion_matrix()

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║ CELL 13 — Final Comparison & Publication Report   ║
# ╚══════════════════════════════════════════════════════╝


import matplotlib.pyplot as plt
plt.ioff()

warnings.filterwarnings("ignore")

# ── Academic Style Global ─────────────────────────────
rcParams.update({
    "font.family"      : "serif",
    "font.serif"       : ["DejaVu Serif"],
    "font.size"        : 11,
    "axes.titlesize"   : 12,
    "axes.labelsize"   : 11,
    "xtick.labelsize"  : 9,
    "ytick.labelsize"  : 9,
    "legend.fontsize"  : 9,
    "figure.dpi"       : 150,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "axes.grid"        : True,
    "grid.alpha"       : 0.3,
    "grid.linestyle"   : "--",
    "grid.linewidth"   : 0.5,
    "savefig.bbox"     : "tight",
    "savefig.dpi"      : 300,
})

# Grayscale palette untuk 4 kombinasi
GS_PALETTE  = ["#000000", "#444444", "#888888", "#BBBBBB"]
GS_HATCHES  = ["/", "\\", "x", "o"]
GS_MARKERS  = ["o", "s", "^", "D"]
GS_LINES    = ["-", "--", "-.", ":"]

COMBO_LABELS = {
    (True,  True ): "CLAHE+Aug",
    (False, True ): "NoCLAHE+Aug",
    (True,  False): "CLAHE+NoAug",
    (False, False): "NoCLAHE+NoAug",
}


# ══════════════════════════════════════════════════════
# HELPER: load data
# ══════════════════════════════════════════════════════

def _load_results(all_results):
    if not all_results:
        csv_files = glob.glob(os.path.join(OUTPUT_DIR, "results_*.csv"))
        if not csv_files:
            LOG.warn("Tidak ada data. Jalankan Cell 12 dulu.")
            return None
        all_results = [pd.read_csv(f).iloc[0].to_dict() for f in csv_files]

    # Fallback/heal: Populate val_f1_weighted if missing/0 from report_Val_*.csv
    for r in all_results:
        if "val_f1_weighted" not in r or r["val_f1_weighted"] == 0.0:
            val_rpt_path = os.path.join(OUTPUT_DIR, f"report_Val_{r['run_name']}.csv")
            if os.path.exists(val_rpt_path):
                try:
                    df_rpt = pd.read_csv(val_rpt_path, index_col=0)
                    if "weighted avg" in df_rpt.index:
                        r["val_f1_weighted"] = float(df_rpt.loc["weighted avg", "f1-score"])
                except Exception:
                    pass

    df = pd.DataFrame(all_results)
    df = df.dropna(subset=["test_qwk"]).reset_index(drop=True)
    df = df.sort_values("test_qwk", ascending=False).reset_index(drop=True)
    df.insert(0, "rank", range(1, len(df) + 1))

    for col in ["val_macro_auc", "val_acc", "test_f1_weighted",
                "val_f1_macro", "val_qwk", "val_f1_weighted"]:
        if col not in df.columns:
            df[col] = 0.0

    # Label ringkas kombinasi
    df["combo_label"] = df.apply(
        lambda r: COMBO_LABELS.get(
            (bool(r["use_clahe"]), bool(r["use_aug"])), r["run_name"]
        ), axis=1
    )
    return df


# ══════════════════════════════════════════════════════
# SECTION 1 — Terminal Summary
# ══════════════════════════════════════════════════════

def _print_summary(df):
    best  = df.iloc[0]
    worst = df.iloc[-1]
    N     = len(df)

    print("\n" + "="*110)
    print("  [CELL 13]  FINAL COMPARISON REPORT")
    print(f"  Model: {df['model_name'].iloc[0]}  |  "
          f"Total Kombinasi: {N}  |  "
          f"Dataset: APTOS 2019")
    print("="*110)
    print(f"  {'Rank':<4} {'Kombinasi':<18} {'Val QWK':>9} {'Val F1':>9} "
          f"{'Val Acc':>9} {'Test QWK':>9} {'Test F1':>9} "
          f"{'Test Acc':>9} {'Test AUC':>9} {'Ep':>4} {'min':>6}")
    print("  " + "-"*108)

    for _, r in df.iterrows():
        marker = " ◀ BEST" if r["rank"] == 1 else ""
        print(f"  {int(r['rank']):<4} {r['combo_label']:<18}"
              f" {r['val_qwk']:>9.4f} {r['val_f1_macro']:>9.4f}"
              f" {r['val_acc']*100:>8.2f}%"
              f" {r['test_qwk']:>9.4f} {r['test_f1_macro']:>9.4f}"
              f" {r['test_acc']*100:>8.2f}%"
              f" {r['test_macro_auc']:>9.4f}"
              f" {int(r.get('best_epoch', 0)):>4}"
              f" {r.get('training_time_min', 0):>6.1f}"
              f"{marker}")

    print("  " + "-"*108)
    dqwk = best["test_qwk"]      - worst["test_qwk"]
    df1  = best["test_f1_macro"] - worst["test_f1_macro"]
    dacc = (best["test_acc"]     - worst["test_acc"]) * 100
    print(f"  Best  : {best['combo_label']}  "
          f"QWK={best['test_qwk']:.4f}  "
          f"F1={best['test_f1_macro']:.4f}  "
          f"Acc={best['test_acc']*100:.2f}%  "
          f"AUC={best['test_macro_auc']:.4f}")
    print(f"  Worst : {worst['combo_label']}  "
          f"QWK={worst['test_qwk']:.4f}  "
          f"F1={worst['test_f1_macro']:.4f}  "
          f"Acc={worst['test_acc']*100:.2f}%")
    print(f"  Delta : ΔQWK={dqwk:+.4f}  ΔF1={df1:+.4f}  ΔAcc={dacc:+.2f}%")
    print("="*110)

    # Rekomendasi otomatis
    print(f"\n  📌 REKOMENDASI OTOMATIS")
    print(f"  {'─'*60}")
    print(f"  Konfigurasi terbaik : {best['combo_label']}")
    print(f"  CLAHE               : {'Aktif' if best['use_clahe'] else 'Tidak aktif'}")
    print(f"  Augmentasi          : {'Aktif' if best['use_aug']   else 'Tidak aktif'}")
    print(f"  Best epoch          : {int(best.get('best_epoch', 0))}")

    clahe_on  = df[df["use_clahe"] == True ]["test_qwk"].mean()
    clahe_off = df[df["use_clahe"] == False]["test_qwk"].mean()
    aug_on    = df[df["use_aug"]   == True ]["test_qwk"].mean()
    aug_off   = df[df["use_aug"]   == False]["test_qwk"].mean()

    print(f"\n  Efek rata-rata CLAHE : "
          f"ON={clahe_on:.4f}  OFF={clahe_off:.4f}  "
          f"({'CLAHE membantu' if clahe_on > clahe_off else 'CLAHE tidak membantu'})")
    print(f"  Efek rata-rata Aug   : "
          f"ON={aug_on:.4f}  OFF={aug_off:.4f}  "
          f"({'Aug membantu' if aug_on > aug_off else 'Aug tidak membantu'})")
    print(f"  {'─'*60}\n")


# ══════════════════════════════════════════════════════
# SECTION 2 — Figure 1: Performance Overview
# (Bar chart metrik utama + Ablation effect)
# ══════════════════════════════════════════════════════

def _fig1_performance_overview(df):
    combos  = df["combo_label"].tolist()
    N       = len(combos)
    metrics = ["test_qwk", "test_f1_macro", "test_acc", "test_macro_auc"]
    mlabels = ["QWK", "F1-Macro", "Accuracy", "Macro-AUC"]

    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    fig.suptitle(
        f"Figure 1. Performance Comparison Across Preprocessing Combinations\n"
        f"Model: {df['model_name'].iloc[0]}",
        fontsize=12, fontweight="bold", y=1.01
    )

    for ax, metric, label in zip(axes.flat, metrics, mlabels):
        vals  = df[metric].tolist()
        if "acc" in metric:
            vals_plot = [v * 100 for v in vals]
            ylabel    = f"{label} (%)"
            fmt       = "{:.2f}%"
        else:
            vals_plot = vals
            ylabel    = label
            fmt       = "{:.4f}"

        bars = ax.bar(combos, vals_plot,
                      color=GS_PALETTE[:N],
                      hatch=GS_HATCHES[:N] if N <= 4 else None,
                      edgecolor="black", linewidth=0.8, alpha=0.85)

        # Nilai di atas bar
        for bar, v in zip(bars, vals_plot):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(vals_plot)*0.01,
                    fmt.format(v),
                    ha="center", va="bottom", fontsize=8.5, fontweight="bold")

        # Garis best score
        ax.axhline(y=max(vals_plot), color="black",
                   linestyle=":", linewidth=1, alpha=0.5)

        ax.set_title(label, fontweight="bold")
        ax.set_ylabel(ylabel)
        ax.set_xticklabels(combos, rotation=20, ha="right")
        ymin = min(vals_plot) * 0.97
        ymax = max(vals_plot) * 1.05
        ax.set_ylim(ymin, ymax)

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, "fig1_performance_overview.png")
    plt.savefig(p, dpi=300); plt.close("all")
    LOG.info(f"Fig 1 → {os.path.basename(p)}")


# ══════════════════════════════════════════════════════
# SECTION 3 — Figure 2: Ablation Study (2×2 grid)
# Effect of CLAHE and Augmentation separately
# ══════════════════════════════════════════════════════

def _fig2_ablation_study(df):
    metrics = ["test_qwk", "test_f1_macro", "test_acc", "test_macro_auc"]
    mlabels = ["QWK", "F1-Macro", "Accuracy", "Macro-AUC"]

    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    fig.suptitle(
        "Figure 2. Ablation Study: Effect of CLAHE and Augmentation",
        fontsize=12, fontweight="bold"
    )

    # Plot 1 — Efek CLAHE
    ax = axes[0]
    x  = np.arange(len(metrics))
    w  = 0.35
    clahe_on  = [df[df["use_clahe"] == True ][m].mean() for m in metrics]
    clahe_off = [df[df["use_clahe"] == False][m].mean() for m in metrics]

    b1 = ax.bar(x - w/2, clahe_on,  w, label="CLAHE ON",
                color="#333333", hatch="/", edgecolor="black", alpha=0.85)
    b2 = ax.bar(x + w/2, clahe_off, w, label="CLAHE OFF",
                color="#AAAAAA", hatch="\\", edgecolor="black", alpha=0.85)

    for bars in [b1, b2]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2,
                    h + 0.003, f"{h:.4f}",
                    ha="center", va="bottom", fontsize=7.5)

    ax.set_title("Effect of CLAHE Preprocessing", fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(mlabels)
    ax.set_ylabel("Score (mean across aug combinations)")
    ax.legend()
    ax.set_ylim(min(clahe_on + clahe_off) * 0.97,
                max(clahe_on + clahe_off) * 1.04)

    # Plot 2 — Efek Augmentasi
    ax = axes[1]
    aug_on  = [df[df["use_aug"] == True ][m].mean() for m in metrics]
    aug_off = [df[df["use_aug"] == False][m].mean() for m in metrics]

    b3 = ax.bar(x - w/2, aug_on,  w, label="Augmentation ON",
                color="#333333", hatch="x", edgecolor="black", alpha=0.85)
    b4 = ax.bar(x + w/2, aug_off, w, label="Augmentation OFF",
                color="#AAAAAA", hatch="o", edgecolor="black", alpha=0.85)

    for bars in [b3, b4]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2,
                    h + 0.003, f"{h:.4f}",
                    ha="center", va="bottom", fontsize=7.5)

    ax.set_title("Effect of Data Augmentation", fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(mlabels)
    ax.set_ylabel("Score (mean across CLAHE combinations)")
    ax.legend()
    ax.set_ylim(min(aug_on + aug_off) * 0.97,
                max(aug_on + aug_off) * 1.04)

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, "fig2_ablation_study.png")
    plt.savefig(p, dpi=300); plt.close("all")
    LOG.info(f"Fig 2 → {os.path.basename(p)}")


# ══════════════════════════════════════════════════════
# SECTION 4 — Figure 3: Training Curve per kombinasi
# ══════════════════════════════════════════════════════

def _fig3_training_curves(df):
    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    fig.suptitle(
        "Figure 3. Training and Validation Curves per Preprocessing Combination",
        fontsize=12, fontweight="bold"
    )

    pairs = [
        ("train_loss", "val_loss", "Loss",     axes[0, 0]),
        ("train_acc",  "val_acc",  "Accuracy", axes[0, 1]),
        ("train_f1",   "val_f1",   "F1-Macro", axes[1, 0]),
        ("train_qwk",  "val_qwk",  "QWK",      axes[1, 1]),
    ]

    any_data = False
    for idx, (_, row) in enumerate(df.iterrows()):
        hist_path = os.path.join(OUTPUT_DIR,
                                 f"history_{row['run_name']}.csv")
        if not os.path.exists(hist_path):
            continue
        dh    = pd.read_csv(hist_path)
        color = GS_PALETTE[idx % len(GS_PALETTE)]
        ls    = GS_LINES[idx % len(GS_LINES)]
        label = row["combo_label"]
        any_data = True

        for tr_key, va_key, title, ax in pairs:
            if tr_key in dh.columns:
                ax.plot(dh["epoch"], dh[tr_key],
                        color=color, ls=ls, lw=1.5,
                        marker=GS_MARKERS[idx], markersize=3,
                        markevery=max(1, len(dh)//10),
                        label=f"{label} (train)" if idx == 0 else "_")
            if va_key in dh.columns:
                ax.plot(dh["epoch"], dh[va_key],
                        color=color, ls=ls, lw=2,
                        marker=GS_MARKERS[idx], markersize=4,
                        markevery=max(1, len(dh)//10),
                        label=f"{label} (val)")

    for tr_key, va_key, title, ax in pairs:
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(title)
        if any_data:
            ax.legend(fontsize=7, ncol=2)
        else:
            ax.text(0.5, 0.5,
                    "history_*.csv tidak ditemukan",
                    ha="center", va="center",
                    transform=ax.transAxes,
                    fontsize=9, color="gray", style="italic")

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, "fig3_training_curves.png")
    plt.savefig(p, dpi=300); plt.close("all")
    LOG.info(f"Fig 3 → {os.path.basename(p)}")


# ══════════════════════════════════════════════════════
# SECTION 5 — Figure 4: Per-class Metrics Heatmap
# ══════════════════════════════════════════════════════

def _fig4_perclass_heatmap(df):
    metrics_pc = ["precision", "recall", "f1-score"]
    n_combos   = len(df)
    n_classes  = len(CLASS_NAMES)

    fig, axes = plt.subplots(1, 3, figsize=(16, max(4, n_combos * 1.2 + 2)))
    fig.suptitle(
        "Figure 4. Per-Class Metrics Heatmap (Test Set)",
        fontsize=12, fontweight="bold"
    )

    for ax, metric in zip(axes, metrics_pc):
        matrix = np.zeros((n_combos, n_classes))
        for i, (_, row) in enumerate(df.iterrows()):
            rpt_path = os.path.join(OUTPUT_DIR,
                                    f"report_Test_{row['run_name']}.csv")
            if not os.path.exists(rpt_path):
                continue
            try:
                df_rpt = pd.read_csv(rpt_path, index_col=0)
                for j, name in enumerate(CLASS_NAMES):
                    if name in df_rpt.index and metric in df_rpt.columns:
                        matrix[i, j] = float(df_rpt.loc[name, metric])
            except Exception:
                pass

        sns.heatmap(
            matrix,
            annot=True, fmt=".3f",
            cmap="viridis",
            xticklabels=[f"C{i}\n{n[:6]}" for i, n in enumerate(CLASS_NAMES)],
            yticklabels=df["combo_label"].tolist(),
            ax=ax,
            vmin=0, vmax=1,
            linewidths=0.5,
            linecolor="white",
            cbar_kws={"label": metric.title(), "shrink": 0.8},
            annot_kws={"size": 9}
        )
        ax.set_title(metric.title(), fontweight="bold")
        ax.set_xlabel("Class")
        ax.set_ylabel("")
        ax.tick_params(axis="x", rotation=0)
        ax.tick_params(axis="y", rotation=0)

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, "fig4_perclass_heatmap.png")
    plt.savefig(p, dpi=300); plt.close("all")
    LOG.info(f"Fig 4 → {os.path.basename(p)}")


# ══════════════════════════════════════════════════════
# SECTION 6 — Figure 5: Sensitivity & Specificity
# per kombinasi per kelas
# ══════════════════════════════════════════════════════

def _fig5_sens_spec_comparison(df):
    n_combos = len(df)
    n_classes = len(CLASS_NAMES)

    sens_matrix = np.zeros((n_combos, n_classes))
    spec_matrix = np.zeros((n_combos, n_classes))

    for i, (_, row) in enumerate(df.iterrows()):
        ss_path = os.path.join(OUTPUT_DIR,
                               f"sens_spec_Test_{row['run_name']}.csv")
        if not os.path.exists(ss_path):
            continue
        try:
            df_ss = pd.read_csv(ss_path)
            for j in range(n_classes):
                if j < len(df_ss):
                    sens_matrix[i, j] = df_ss.iloc[j]["sensitivity"]
                    spec_matrix[i, j] = df_ss.iloc[j]["specificity"]
        except Exception:
            pass

    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, n_combos * 1.2 + 2)))
    fig.suptitle(
        "Figure 5. Sensitivity and Specificity per Class (Test Set)",
        fontsize=12, fontweight="bold"
    )

    ylabels = df["combo_label"].tolist()
    xlabels = [f"C{i}\n{n[:6]}" for i, n in enumerate(CLASS_NAMES)]

    for ax, matrix, title in [
        (axes[0], sens_matrix, "Sensitivity (Recall)"),
        (axes[1], spec_matrix, "Specificity"),
    ]:
        sns.heatmap(
            matrix,
            annot=True, fmt=".3f",
            cmap="GnBu",
            xticklabels=xlabels,
            yticklabels=ylabels,
            ax=ax,
            vmin=0, vmax=1,
            linewidths=0.5,
            linecolor="white",
            cbar_kws={"label": title, "shrink": 0.8},
            annot_kws={"size": 9}
        )
        # Tandai cell dengan nilai < 0.7
        for y in range(n_combos):
            for x in range(n_classes):
                if matrix[y, x] < 0.70:
                    ax.add_patch(plt.Rectangle(
                        (x, y), 1, 1,
                        fill=False, edgecolor="black",
                        lw=2, clip_on=False
                    ))
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Class")
        ax.set_ylabel("")
        ax.tick_params(axis="x", rotation=0)
        ax.tick_params(axis="y", rotation=0)

    fig.text(0.5, -0.02,
             "Note: Bold border indicates value < 0.70 (requires clinical attention)",
             ha="center", fontsize=9, style="italic")

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, "fig5_sens_spec_comparison.png")
    plt.savefig(p, dpi=300); plt.close("all")
    LOG.info(f"Fig 5 → {os.path.basename(p)}")


# ══════════════════════════════════════════════════════
# SECTION 7 — Figure 6: ROC Curve semua kombinasi
# ══════════════════════════════════════════════════════

def _fig6_roc_curves(df):
    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    fig.suptitle(
        "Figure 6. ROC Curves (Test Set)",
        fontsize=12, fontweight="bold"
    )

    # Plot 1 — Macro-average ROC tiap kombinasi
    ax = axes[0]
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random Classifier")
    any_roc = False

    for idx, (_, row) in enumerate(df.iterrows()):
        prob_path = os.path.join(OUTPUT_DIR,
                                 f"yprob_Test_{row['run_name']}.npy")
        true_path = os.path.join(OUTPUT_DIR,
                                 f"ytrue_Test_{row['run_name']}.npy")
        if not (os.path.exists(prob_path) and os.path.exists(true_path)):
            continue

        y_prob = np.load(prob_path)
        y_true = np.load(true_path).astype(int)
        y_bin  = label_binarize(y_true, classes=list(range(len(CLASS_NAMES))))

        all_fpr = np.unique(np.concatenate([
            roc_curve(y_bin[:, i], y_prob[:, i])[0]
            for i in range(len(CLASS_NAMES))
        ]))
        mean_tpr = np.zeros_like(all_fpr)
        for i in range(len(CLASS_NAMES)):
            fpr_i, tpr_i, _ = roc_curve(y_bin[:, i], y_prob[:, i])
            mean_tpr += np.interp(all_fpr, fpr_i, tpr_i)
        mean_tpr /= len(CLASS_NAMES)
        macro_auc = auc(all_fpr, mean_tpr)

        ax.plot(all_fpr, mean_tpr,
                color=GS_PALETTE[idx % len(GS_PALETTE)],
                ls=GS_LINES[idx % len(GS_LINES)],
                lw=2,
                label=f"{row['combo_label']} (AUC={macro_auc:.4f})")
        any_roc = True

    ax.set_title("Macro-Average ROC per Combination", fontweight="bold")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    if any_roc:
        ax.legend(loc="lower right")
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])

    # Plot 2 — ROC per kelas untuk kombinasi terbaik
    ax2 = axes[1]
    ax2.plot([0, 1], [0, 1], "k--", lw=1, label="Random Classifier")

    best_row  = df.iloc[0]
    prob_path = os.path.join(OUTPUT_DIR,
                             f"yprob_Test_{best_row['run_name']}.npy")
    true_path = os.path.join(OUTPUT_DIR,
                             f"ytrue_Test_{best_row['run_name']}.npy")

    if os.path.exists(prob_path) and os.path.exists(true_path):
        y_prob = np.load(prob_path)
        y_true = np.load(true_path).astype(int)
        y_bin  = label_binarize(y_true, classes=list(range(len(CLASS_NAMES))))

        ls_cycle = ["-", "--", "-.", ":", (0, (3,1,1,1))]
        for i, name in enumerate(CLASS_NAMES):
            fpr_i, tpr_i, _ = roc_curve(y_bin[:, i], y_prob[:, i])
            auc_i = auc(fpr_i, tpr_i)
            ax2.plot(fpr_i, tpr_i,
                     color="black",
                     ls=ls_cycle[i % len(ls_cycle)],
                     lw=1.8,
                     label=f"C{i} {name[:8]} (AUC={auc_i:.4f})")

    ax2.set_title(
        f"Per-Class ROC — Best Config\n({best_row['combo_label']})",
        fontweight="bold"
    )
    ax2.set_xlabel("False Positive Rate")
    ax2.set_ylabel("True Positive Rate")
    ax2.legend(loc="lower right", fontsize=8)
    ax2.set_xlim([0, 1])
    ax2.set_ylim([0, 1.02])

    if not any_roc:
        for ax in axes:
            ax.text(0.5, 0.5,
                    "yprob_Test_*.npy tidak ditemukan\n"
                    "Jalankan Cell 12 terlebih dahulu",
                    ha="center", va="center", transform=ax.transAxes,
                    fontsize=9, color="gray", style="italic")

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, "fig6_roc_curves.png")
    plt.savefig(p, dpi=300); plt.close("all")
    LOG.info(f"Fig 6 → {os.path.basename(p)}")


# ══════════════════════════════════════════════════════
# SECTION 8 — Figure 7: Efficiency vs Performance
# ══════════════════════════════════════════════════════

def _fig7_efficiency_tradeoff(df):
    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    fig.suptitle(
        "Figure 7. Training Efficiency vs. Performance",
        fontsize=12, fontweight="bold"
    )

    # Plot 1 — Training time vs Test QWK
    ax = axes[0]
    for idx, (_, row) in enumerate(df.iterrows()):
        x = row.get("training_time_min", 0)
        y = row["test_qwk"]
        ax.scatter(x, y,
                   marker=GS_MARKERS[idx],
                   color=GS_PALETTE[idx],
                   s=120, zorder=5,
                   label=row["combo_label"],
                   edgecolors="black", linewidth=0.8)
        ax.annotate(row["combo_label"],
                    (x, y),
                    textcoords="offset points",
                    xytext=(8, 5), fontsize=8)

    ax.set_xlabel("Training Time (minutes)")
    ax.set_ylabel("Test QWK")
    ax.set_title("Training Time vs. Test QWK", fontweight="bold")
    ax.legend(fontsize=8)

    # Plot 2 — Best epoch per kombinasi
    ax2 = axes[1]
    combos = df["combo_label"].tolist()
    epochs = df["best_epoch"].fillna(0).astype(int).tolist()
    qwks   = df["test_qwk"].tolist()

    bars = ax2.barh(combos, epochs,
                    color=GS_PALETTE[:len(combos)],
                    edgecolor="black", linewidth=0.8,
                    hatch=GS_HATCHES[:len(combos)],
                    alpha=0.85)

    for bar, qwk in zip(bars, qwks):
        ax2.text(bar.get_width() + 0.3,
                 bar.get_y() + bar.get_height()/2,
                 f"QWK={qwk:.4f}",
                 va="center", fontsize=9)

    ax2.set_xlabel("Best Epoch")
    ax2.set_title("Convergence Speed per Combination", fontweight="bold")
    ax2.set_xlim(0, max(epochs) * 1.25 if epochs else 10)

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, "fig7_efficiency_tradeoff.png")
    plt.savefig(p, dpi=300); plt.close("all")
    LOG.info(f"Fig 7 → {os.path.basename(p)}")


# ══════════════════════════════════════════════════════
# SECTION 9 — Export Excel (Publication-ready)
# ══════════════════════════════════════════════════════

def _export_excel(df):
    def parse_params_to_million(params_str):
        if not params_str or not isinstance(params_str, str):
            return 0.0
        val_str = params_str.strip().upper()
        try:
            if val_str.endswith("M"):
                return float(val_str[:-1])
            elif val_str.endswith("B"):
                return float(val_str[:-1]) * 1000.0
            elif val_str.endswith("K"):
                return float(val_str[:-1]) / 1000.0
            return float(val_str)
        except ValueError:
            return 0.0

    # Sort df specifically for excel export to the user's requested order:
    # 1. CLAHE+Aug
    # 2. NoCLAHE+Aug
    # 3. CLAHE+NoAug
    # 4. NoCLAHE+NoAug
    order_dict = {
        "CLAHE+Aug": 0,
        "NoCLAHE+Aug": 1,
        "CLAHE+NoAug": 2,
        "NoCLAHE+NoAug": 3
    }
    df = df.copy()
    df["excel_sort_order"] = df["combo_label"].map(lambda x: order_dict.get(x, 999))
    df = df.sort_values("excel_sort_order").drop(columns=["excel_sort_order"]).reset_index(drop=True)

    wb   = Workbook()
    path = os.path.join(OUTPUT_DIR, "final_comparison_report.xlsx")

    # ── Style helpers ─────────────────────────────────
    def _hdr(bold=True, size=11, color="000000", bg="D9D9D9", center=True):
        return {
            "font"     : Font(bold=bold, size=size, color=color),
            "fill"     : PatternFill("solid", fgColor=bg),
            "alignment": Alignment(
                horizontal="center" if center else "left",
                vertical="center", wrap_text=True),
            "border"   : Border(
                left=Side(style="thin"), right=Side(style="thin"),
                top=Side(style="thin"), bottom=Side(style="thin"))
        }

    def _cell(ws, row, col, value, bold=False, center=True,
              bg=None, num_format=None, border=True):
        c = ws.cell(row=row, column=col, value=value)
        c.font      = Font(bold=bold, size=10,
                           name="Calibri")
        c.alignment = Alignment(
            horizontal="center" if center else "left",
            vertical="center")
        if bg:
            c.fill = PatternFill("solid", fgColor=bg)
        if num_format:
            c.number_format = num_format
        if border:
            c.border = Border(
                left=Side(style="thin"), right=Side(style="thin"),
                top=Side(style="thin"), bottom=Side(style="thin"))
        return c

    def _apply_hdr(ws, row, col, value, style):
        c = ws.cell(row=row, column=col, value=value)
        for k, v in style.items():
            setattr(c, k, v)
        return c

    # ══════════════════════════════════════════════════
    # Sheet 1 — Summary Comparison
    # ══════════════════════════════════════════════════
    ws1 = wb.active
    ws1.title = "Summary Comparison"
    ws1.sheet_view.showGridLines = False
    ws1.row_dimensions[1].height = 30

    hdr_style  = _hdr(bg="1F1F1F", color="FFFFFF")
    sub_style  = _hdr(bg="595959", color="FFFFFF")
    best_style = _hdr(bg="404040", color="FFFFFF", bold=True)

    # Title
    ws1.merge_cells("A1:U1")
    t = ws1.cell(row=1, column=1,
                 value=f"Diabetic Retinopathy Classification — Final Comparison Report")
    t.font      = Font(bold=True, size=14, name="Calibri")
    t.alignment = Alignment(horizontal="center", vertical="center")
    t.fill      = PatternFill("solid", fgColor="1F1F1F")
    t.font      = Font(bold=True, size=13, color="FFFFFF",
                       name="Calibri")

    # Sub-header baris 2
    ws1.merge_cells("A2:B2"); ws1.merge_cells("C2:E2")
    ws1.merge_cells("F2:K2"); ws1.merge_cells("L2:Q2")
    ws1.merge_cells("R2:U2")
    for col, val in [(1,"Config"), (3,"Model Info"),
                     (6,"Validation Metrics"),
                     (12,"Test Metrics"), (18,"Efficiency")]:
        _apply_hdr(ws1, 2, col, val, sub_style)
    ws1.row_dimensions[2].height = 22

    # Header baris 3
    headers3 = [
        "Rank", "Combination",
        "CLAHE", "Aug", "Parameter (Juta)",
        "Val QWK", "Val F1 (Macro)", "Val F1 (Weighted)", "Val Acc", "Val Macro-AUC", "Val Sens*",
        "Test QWK", "Test F1 (Macro)", "Test F1 (Weighted)", "Test Acc", "Test Macro-AUC", "Test Sens*",
        "Tr. Time (min)", "Best Epoch", "FLOPs", "Model Size (MB)"
    ]
    for col, h in enumerate(headers3, 1):
        _apply_hdr(ws1, 3, col, h, hdr_style)
    ws1.row_dimensions[3].height = 40

    # Data rows
    for ridx, (_, row) in enumerate(df.iterrows(), 4):
        is_best = row["rank"] == 1
        row_bg  = "F5F5F5" if ridx % 2 == 0 else "FFFFFF"
        bg      = "D0D0D0" if is_best else row_bg

        # Parse params to float million
        params_str = row.get("params", "-")
        params_val = parse_params_to_million(params_str) if params_str != "-" else 0.0

        vals = [
            int(row["rank"]),                         # 1
            row["combo_label"],                      # 2
            "Yes" if row["use_clahe"] else "No",     # 3
            "Yes" if row["use_aug"]   else "No",     # 4
            params_val,                              # 5: Parameter (Juta)
            row["val_qwk"],                          # 6
            row["val_f1_macro"],                     # 7: Val F1 (Macro)
            row.get("val_f1_weighted", 0.0),         # 8: Val F1 (Weighted)
            row["val_acc"],                          # 9
            row.get("val_macro_auc", 0.0),           # 10: Val Macro-AUC
            "",                                      # 11: Val Sens
            row["test_qwk"],                         # 12
            row["test_f1_macro"],                    # 13: Test F1 (Macro)
            row.get("test_f1_weighted", 0.0),        # 14: Test F1 (Weighted)
            row["test_acc"],                         # 15
            row.get("test_macro_auc", 0.0),          # 16: Test Macro-AUC
            "",                                      # 17: Test Sens
            round(row.get("training_time_min", 0), 2), # 18
            int(row.get("best_epoch", 0)),           # 19
            str(row.get("flops", "-")),              # 20
            round(row.get("model_size_mb", 0), 2),   # 21
        ]

        # Coba isi Sens dari CSV
        for split_col, split_name in [(11, "Val"), (17, "Test")]:
            ss_path = os.path.join(OUTPUT_DIR,
                                    f"sens_spec_{split_name}_{row['run_name']}.csv")
            if os.path.exists(ss_path):
                try:
                    df_ss = pd.read_csv(ss_path)
                    macro_sens = df_ss["sensitivity"].mean()
                    vals[split_col - 1] = round(macro_sens, 4)
                except Exception:
                    pass

        for col, val in enumerate(vals, 1):
            num_fmt = None
            if col in [6, 7, 8, 10, 11, 12, 13, 14, 16, 17]:
                num_fmt = "0.0000"
            elif col in [9, 15]:
                num_fmt = "0.00%"
            elif col in [5, 18, 21]:
                num_fmt = "0.00"
            elif col == 19:
                num_fmt = "0"
                
            c = _cell(ws1, ridx, col, val,
                      bold=is_best, bg=bg,
                      num_format=num_fmt)
            if is_best:
                c.font = Font(bold=True, size=10,
                              name="Calibri")

        ws1.row_dimensions[ridx].height = 18

    # Note row
    note_row = len(df) + 4
    ws1.merge_cells(f"A{note_row}:U{note_row}")
    nc = ws1.cell(row=note_row, column=1,
                  value="* Sens = Macro-average Sensitivity across all classes. "
                        "Bold row = Best configuration by Test QWK.")
    nc.font      = Font(italic=True, size=9, name="Calibri")
    nc.alignment = Alignment(horizontal="left")


    # Column widths sheet 1
    col_widths = [
        6,   # Rank (A)
        18,  # Combination (B)
        8,   # CLAHE (C)
        8,   # Aug (D)
        18,  # Parameter (Juta) (E)
        11,  # Val QWK (F)
        15,  # Val F1 (Macro) (G)
        16,  # Val F1 (Weighted) (H)
        11,  # Val Acc (I)
        15,  # Val Macro-AUC (J)
        11,  # Val Sens* (K)
        11,  # Test QWK (L)
        15,  # Test F1 (Macro) (M)
        16,  # Test F1 (Weighted) (N)
        11,  # Test Acc (O)
        15,  # Test Macro-AUC (P)
        11,  # Test Sens* (Q)
        14,  # Tr. Time (min) (R)
        12,  # Best Epoch (S)
        12,  # FLOPs (T)
        16,  # Model Size (MB) (U)
    ]
    for i, w in enumerate(col_widths, 1):
        ws1.column_dimensions[get_column_letter(i)].width = w

# ══════════════════════════════════════════════════
    # Sheet 2 — Per-Class Breakdown per kombinasi
    # ══════════════════════════════════════════════════
    ws2 = wb.create_sheet("Per-Class Breakdown")
    ws2.sheet_view.showGridLines = False

    # Header
    hdrs2 = ["Combination", "Metric"] + CLASS_NAMES + ["Macro Avg"]
    for col, h in enumerate(hdrs2, 1):
        _apply_hdr(ws2, 1, col, h, hdr_style)
    ws2.row_dimensions[1].height = 35

    current_row = 2
    metrics_pc  = ["precision", "recall", "f1-score"]

    for cidx, (_, row) in enumerate(df.iterrows()):
        bg = "F0F0F0" if cidx % 2 == 0 else "FFFFFF"
        rpt_path = os.path.join(OUTPUT_DIR,
                                f"report_Test_{row['run_name']}.csv")
        df_rpt = None
        if os.path.exists(rpt_path):
            try:
                df_rpt = pd.read_csv(rpt_path, index_col=0)
            except Exception:
                pass

        for midx, metric in enumerate(metrics_pc):
            # Kolom A — nama kombinasi, tulis di setiap baris (tanpa merge)
            c = ws2.cell(row=current_row, column=1,
                         value=row["combo_label"] if midx == 0 else "")
            c.font      = Font(bold=True, size=10, name="Calibri")
            c.alignment = Alignment(horizontal="center", vertical="center")
            c.fill      = PatternFill("solid", fgColor=bg)
            c.border    = Border(
                left=Side(style="thin"), right=Side(style="thin"),
                top=Side(style="thin"),  bottom=Side(style="thin"))

            # Kolom B — nama metrik
            _cell(ws2, current_row, 2, metric.title(), bold=True, bg=bg)

            # Kolom C dst — nilai per kelas
            vals_row = []
            for j, name in enumerate(CLASS_NAMES):
                val = 0.0
                if df_rpt is not None and name in df_rpt.index:
                    try:
                        val = float(df_rpt.loc[name, metric])
                    except Exception:
                        pass
                _cell(ws2, current_row, j + 3, round(val, 4),
                      bg=bg, num_format="0.0000")
                vals_row.append(val)

            # Kolom terakhir — macro avg
            macro = np.mean(vals_row) if vals_row else 0.0
            _cell(ws2, current_row, len(CLASS_NAMES) + 3,
                  round(macro, 4), bold=True, bg=bg,
                  num_format="0.0000")

            ws2.row_dimensions[current_row].height = 18
            current_row += 1

        # Baris pemisah antar kombinasi
        current_row += 1

    col_widths2 = [18, 12] + [12] * (len(CLASS_NAMES) + 1)
    for i, w in enumerate(col_widths2, 1):
        ws2.column_dimensions[get_column_letter(i)].width = w

# ══════════════════════════════════════════════════
    # Sheet 3 — Sensitivity & Specificity
    # ══════════════════════════════════════════════════
    ws3 = wb.create_sheet("Sensitivity & Specificity")
    ws3.sheet_view.showGridLines = False

    hdrs3 = ["Combination", "Class", "Sensitivity",
             "Specificity", "PPV", "NPV", "TP", "TN", "FP", "FN"]
    for col, h in enumerate(hdrs3, 1):
        _apply_hdr(ws3, 1, col, h, hdr_style)
    ws3.row_dimensions[1].height = 30

    current_row = 2
    for cidx, (_, row) in enumerate(df.iterrows()):
        bg      = "F0F0F0" if cidx % 2 == 0 else "FFFFFF"
        ss_path = os.path.join(OUTPUT_DIR,
                               f"sens_spec_Test_{row['run_name']}.csv")
        if not os.path.exists(ss_path):
            continue
        try:
            df_ss = pd.read_csv(ss_path)
        except Exception:
            continue

        for ridx, ss_row in enumerate(df_ss.itertuples(index=False)):
            # Kolom A — nama kombinasi hanya di baris pertama, sisanya kosong
            combo_val = row["combo_label"] if ridx == 0 else ""
            c = ws3.cell(row=current_row, column=1, value=combo_val)
            c.font      = Font(bold=True, size=10, name="Calibri")
            c.alignment = Alignment(horizontal="center", vertical="center")
            c.fill      = PatternFill("solid", fgColor=bg)
            c.border    = Border(
                left=Side(style="thin"), right=Side(style="thin"),
                top=Side(style="thin"),  bottom=Side(style="thin"))

            # Kolom B — nama kelas
            _cell(ws3, current_row, 2,
                  getattr(ss_row, "class_name", ""), bg=bg)

            # Kolom C-J — nilai metrik
            for col, key in enumerate(
                    ["sensitivity", "specificity", "ppv", "npv",
                     "TP", "TN", "FP", "FN"], 3):
                val     = getattr(ss_row, key, 0)
                num_fmt = "0.0000" if col <= 6 else "0"
                _cell(ws3, current_row, col, val,
                      bg=bg, num_format=num_fmt)

            ws3.row_dimensions[current_row].height = 18
            current_row += 1

        # Baris macro average
        try:
            avg_sens = df_ss["sensitivity"].mean()
            avg_spec = df_ss["specificity"].mean()
            avg_ppv  = df_ss["ppv"].mean()
            avg_npv  = df_ss["npv"].mean()
        except Exception:
            avg_sens = avg_spec = avg_ppv = avg_npv = 0.0

        _cell(ws3, current_row, 1, "", bg="D0D0D0")
        _cell(ws3, current_row, 2, "Macro Average",
              bold=True, bg="D0D0D0")
        _cell(ws3, current_row, 3, round(avg_sens, 4),
              bold=True, bg="D0D0D0", num_format="0.0000")
        _cell(ws3, current_row, 4, round(avg_spec, 4),
              bold=True, bg="D0D0D0", num_format="0.0000")
        _cell(ws3, current_row, 5, round(avg_ppv, 4),
              bold=True, bg="D0D0D0", num_format="0.0000")
        _cell(ws3, current_row, 6, round(avg_npv, 4),
              bold=True, bg="D0D0D0", num_format="0.0000")
        for col in range(7, 11):
            _cell(ws3, current_row, col, "", bg="D0D0D0")
        ws3.row_dimensions[current_row].height = 18
        current_row += 1

        # Baris pemisah
        current_row += 1

    col_widths3 = [18, 18, 13, 13, 10, 10, 8, 8, 8, 8]
    for i, w in enumerate(col_widths3, 1):
        ws3.column_dimensions[get_column_letter(i)].width = w


    # ══════════════════════════════════════════════════
    # Sheet 4 — Training History
    # ══════════════════════════════════════════════════
    ws4 = wb.create_sheet("Training History")
    ws4.sheet_view.showGridLines = False

    hist_hdrs = ["Combination", "Epoch",
                 "Train Loss", "Val Loss",
                 "Train Acc",  "Val Acc",
                 "Train F1",   "Val F1",
                 "Train QWK",  "Val QWK",
                 "LR"]
    for col, h in enumerate(hist_hdrs, 1):
        _apply_hdr(ws4, 1, col, h, hdr_style)
    ws4.row_dimensions[1].height = 30

    current_row = 2
    for cidx, (_, row) in enumerate(df.iterrows()):
        bg        = "F5F5F5" if cidx % 2 == 0 else "FFFFFF"
        hist_path = os.path.join(OUTPUT_DIR,
                                 f"history_{row['run_name']}.csv")
        if not os.path.exists(hist_path):
            continue

        try:
            dh = pd.read_csv(hist_path)
        except Exception:
            continue

        for epoch_idx, hr in enumerate(dh.itertuples(index=False)):
            # Kolom A — nama kombinasi hanya di baris pertama, sisanya kosong
            combo_val = row["combo_label"] if epoch_idx == 0 else ""
            c = ws4.cell(row=current_row, column=1, value=combo_val)
            c.font      = Font(bold=True, size=10, name="Calibri")
            c.alignment = Alignment(horizontal="center", vertical="center")
            c.fill      = PatternFill("solid", fgColor=bg)
            c.border    = Border(
                left=Side(style="thin"), right=Side(style="thin"),
                top=Side(style="thin"),  bottom=Side(style="thin"))

            # Kolom B — epoch
            _cell(ws4, current_row, 2,
                  int(getattr(hr, "epoch", epoch_idx + 1)),
                  bg=bg, num_format="0")

            # Kolom C-K — metrik per epoch
            keys = ["train_loss", "val_loss",
                    "train_acc",  "val_acc",
                    "train_f1",   "val_f1",
                    "train_qwk",  "val_qwk",
                    "lr"]
            for col, key in enumerate(keys, 3):
                val = getattr(hr, key, 0)
                fmt = "0.00E+00" if key == "lr" else "0.0000"
                _cell(ws4, current_row, col, val,
                      bg=bg, num_format=fmt)

            ws4.row_dimensions[current_row].height = 16
            current_row += 1

        # Baris pemisah antar kombinasi
        current_row += 1

    col_widths4 = [18, 7] + [12] * 9
    for i, w in enumerate(col_widths4, 1):
        ws4.column_dimensions[get_column_letter(i)].width = w

    # ══════════════════════════════════════════════════
    # Sheet 5 — Ablation Effect Summary
    # ══════════════════════════════════════════════════
    ws5 = wb.create_sheet("Ablation Effect")
    ws5.sheet_view.showGridLines = False

    metrics_abl = ["test_qwk", "test_f1_macro",
                   "test_acc", "test_macro_auc"]
    m_labels_abl = ["Test QWK", "Test F1-Macro",
                    "Test Accuracy", "Test Macro-AUC"]

    ws5.merge_cells("A1:F1")
    t5 = ws5.cell(row=1, column=1,
                  value="Ablation Study: Effect of CLAHE and Augmentation")
    t5.font      = Font(bold=True, size=12, name="Calibri")
    t5.alignment = Alignment(horizontal="center")
    t5.fill      = PatternFill("solid", fgColor="1F1F1F")
    t5.font      = Font(bold=True, size=12, color="FFFFFF",
                        name="Calibri")
    ws5.row_dimensions[1].height = 28

    # Sub-tabel CLAHE
    ws5.merge_cells("A2:F2")
    sc = ws5.cell(row=2, column=1, value="Effect of CLAHE Preprocessing")
    sc.font      = Font(bold=True, size=11, name="Calibri")
    sc.alignment = Alignment(horizontal="center")
    sc.fill      = PatternFill("solid", fgColor="595959")
    sc.font      = Font(bold=True, color="FFFFFF", name="Calibri")
    ws5.row_dimensions[2].height = 22

    abl_hdrs = ["Metric", "CLAHE ON (mean)", "CLAHE OFF (mean)",
                "Difference", "Effect", "Verdict"]
    for col, h in enumerate(abl_hdrs, 1):
        _apply_hdr(ws5, 3, col, h, hdr_style)
    ws5.row_dimensions[3].height = 22

    for ridx, (metric, label) in enumerate(
            zip(metrics_abl, m_labels_abl), 4):
        on_val  = df[df["use_clahe"] == True ][metric].mean()
        off_val = df[df["use_clahe"] == False][metric].mean()
        diff    = on_val - off_val
        effect  = "Positive" if diff > 0.001 else (
                  "Negative" if diff < -0.001 else "Neutral")
        verdict = "CLAHE helps ✓" if diff > 0.001 else (
                  "CLAHE hurts ✗" if diff < -0.001 else
                  "No significant effect")
        bg = "F0F0F0" if ridx % 2 == 0 else "FFFFFF"
        for col, val, fmt in [
            (1, label,   None),
            (2, round(on_val,  4), "0.0000"),
            (3, round(off_val, 4), "0.0000"),
            (4, round(diff,    4), "+0.0000;-0.0000;0.0000"),
            (5, effect,  None),
            (6, verdict, None),
        ]:
            c = _cell(ws5, ridx, col, val, bg=bg, num_format=fmt)
            if col == 4:
                c.font = Font(
                    bold=True, size=10, name="Calibri",
                    color="000000" if abs(diff) < 0.001 else "000000")
        ws5.row_dimensions[ridx].height = 18

    # Sub-tabel Augmentasi
    aug_start = len(metrics_abl) + 6
    ws5.merge_cells(f"A{aug_start}:F{aug_start}")
    sa = ws5.cell(row=aug_start, column=1,
                  value="Effect of Data Augmentation")
    sa.font      = Font(bold=True, color="FFFFFF",
                        name="Calibri")
    sa.alignment = Alignment(horizontal="center")
    sa.fill      = PatternFill("solid", fgColor="595959")
    ws5.row_dimensions[aug_start].height = 22

    for col, h in enumerate(abl_hdrs, 1):
        _apply_hdr(ws5, aug_start+1, col, h, hdr_style)
    ws5.row_dimensions[aug_start+1].height = 22

    for ridx2, (metric, label) in enumerate(
            zip(metrics_abl, m_labels_abl), aug_start+2):
        on_val  = df[df["use_aug"] == True ][metric].mean()
        off_val = df[df["use_aug"] == False][metric].mean()
        diff    = on_val - off_val
        effect  = "Positive" if diff > 0.001 else (
                  "Negative" if diff < -0.001 else "Neutral")
        verdict = "Aug helps ✓" if diff > 0.001 else (
                  "Aug hurts ✗" if diff < -0.001 else
                  "No significant effect")
        bg = "F0F0F0" if ridx2 % 2 == 0 else "FFFFFF"
        for col, val, fmt in [
            (1, label,   None),
            (2, round(on_val,  4), "0.0000"),
            (3, round(off_val, 4), "0.0000"),
            (4, round(diff,    4), "+0.0000;-0.0000;0.0000"),
            (5, effect,  None),
            (6, verdict, None),
        ]:
            _cell(ws5, ridx2, col, val, bg=bg, num_format=fmt)
        ws5.row_dimensions[ridx2].height = 18

    col_widths5 = [18, 16, 17, 14, 12, 22]
    for i, w in enumerate(col_widths5, 1):
        ws5.column_dimensions[get_column_letter(i)].width = w

    # ── Simpan ────────────────────────────────────────
    wb.save(path)
    LOG.info(f"Excel report → {os.path.basename(path)}")
    print(f"\n  📊 Excel disimpan → {path}")
    print(f"  Sheet: Summary Comparison | Per-Class Breakdown | "
          f"Sensitivity & Specificity | Training History | Ablation Effect")


# ══════════════════════════════════════════════════════
# MAIN — Jalankan semua
# ══════════════════════════════════════════════════════

def run_final_comparison(all_results):
    df = _load_results(all_results)
    if df is None:
        return None

    print("\n" + "█"*110)
    print("  CELL 13 — FINAL COMPARISON & PUBLICATION REPORT")
    print("█"*110)

    _print_summary(df)

    LOG.info("Generating Fig 1: Performance Overview...")
    _fig1_performance_overview(df)

    LOG.info("Generating Fig 2: Ablation Study...")
    _fig2_ablation_study(df)

    LOG.info("Generating Fig 3: Training Curves...")
    _fig3_training_curves(df)

    LOG.info("Generating Fig 4: Per-class Heatmap...")
    _fig4_perclass_heatmap(df)

    LOG.info("Generating Fig 5: Sensitivity & Specificity...")
    _fig5_sens_spec_comparison(df)

    LOG.info("Generating Fig 6: ROC Curves...")
    _fig6_roc_curves(df)

    LOG.info("Generating Fig 7: Efficiency Tradeoff...")
    _fig7_efficiency_tradeoff(df)

    LOG.info("Exporting Excel report...")
    _export_excel(df)

    print("\n" + "═"*60)
    print(f"  ✅ Cell 13 selesai")
    print(f"  📁 Semua output di: {OUTPUT_DIR}")
    print(f"  📊 7 Figures (PNG 300dpi) + 1 Excel (5 sheets)")
    print("═"*60)

    return df


# ── Jalankan ──────────────────────────────────────────
df_final = run_final_comparison(all_experiment_results)

plt.ion()

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║ CELL 14 — Ablation Study Heatmap                     ║
# ╚══════════════════════════════════════════════════════╝

def generate_ablation_heatmap():
    csv_files = glob.glob(os.path.join(OUTPUT_DIR, "results_*.csv"))
    if not csv_files:
        LOG.warn("Belum ada results_*.csv")
        return None

    df_all = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
    if "test_qwk" in df_all.columns:
        df_all = df_all.sort_values("test_qwk", ascending=False).reset_index(drop=True)

    metric_cols = [c for c in ["val_qwk","val_f1_macro","val_macro_auc","test_qwk","test_f1_macro","test_macro_auc","test_acc"] if c in df_all.columns]
    if not metric_cols:
        LOG.warn("Kolom metrik tidak ditemukan.")
        return None

    fig, ax = plt.subplots(figsize=(13, max(4, len(df_all) * 1.4)))
    sns.heatmap(df_all.set_index("run_name")[metric_cols], annot=True, fmt=".4f", cmap="YlGn", linewidths=0.5, ax=ax, vmin=0, vmax=1)
    ax.set_title("Ablation Study Heatmap — CLAHE × Augmentation", fontweight="bold", fontsize=13)
    plt.tight_layout()

    p = os.path.join(OUTPUT_DIR, "ablation_heatmap.png")
    plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")
    LOG.info(f"Ablation heatmap → {os.path.basename(p)}")
    return df_all

df_ablation = generate_ablation_heatmap()

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║ CELL 15 — 🏆 Hall of Fame                           ║
# ║  Ranking berdasarkan QWK, F1, Acc, AUC              ║
# ╚══════════════════════════════════════════════════════╝
def print_hall_of_fame(all_results):
    if not all_results:
        LOG.warn("Belum ada data eksperimen."); return
    df = pd.DataFrame(all_results)
    if df.empty or "test_qwk" not in df.columns:
        LOG.warn("DataFrame kosong atau kolom test_qwk tidak ada."); return
    df = df.dropna(subset=["test_qwk"]).reset_index(drop=True)
    categories = {
        "🥇 Best QWK"       : ("test_qwk",       "Quadratic Weighted Kappa"),
        "🎯 Best F1-Macro"  : ("test_f1_macro",   "F1-Score Macro"),
        "📈 Best Accuracy"  : ("test_acc",        "Test Accuracy"),
        "📉 Best AUC"       : ("test_macro_auc",  "Macro ROC-AUC"),
        "⚡ Fastest Training": ("training_time_min","Training Time (menit) — ascending"),
    }

    print("\n" + "═"*65)
    print("\n"+"="*80)
    print("  [CELL 15]  HALL OF FAME -- BEST OF THE BEST")
    model_name_dinamis = df["model_name"].iloc[0] if ("model_name" in df.columns and len(df) > 0) else "VGG16"
    print(f"  {model_name_dinamis} | 4-Kombinasi CLAHE x Aug | APTOS 2019")

    print("="*80)
    for title,(col,desc) in categories.items():
        if col not in df.columns: continue
        asc="ascending" in desc.lower()
        row=df.loc[df[col].idxmin() if asc else df[col].idxmax()]
        fmt=f"{row[col]*100:.2f}%" if "acc" in col else (f"{row[col]:.2f}min" if "time" in col.lower() else f"{row[col]:.6f}")
        cs="ON" if row["use_clahe"] else "OFF"; as_="ON" if row["use_aug"] else "OFF"
        print(f"  [{title}]")
        print(f"    Pemenang : {row['run_name']}  |  CLAHE={cs}  Aug={as_}")
        print(f"    {desc}: {fmt}")
    print("\n  RANKING (Test QWK desc):")
    print(f"  {'#':<4}{'Run':<28}{'QWK':>8}{'F1':>8}{'Acc':>8}{'AUC':>8}{'BestEp':>7}")
    print("  "+"-"*66)
    for i,row in df.sort_values("test_qwk",ascending=False).reset_index(drop=True).iterrows():
        print(f"  {i+1:<4}{row['run_name']:<28}{row['test_qwk']:>8.4f}{row['test_f1_macro']:>8.4f}"
              f"{row['test_acc']*100:>7.2f}%{row['test_macro_auc']:>8.4f}{int(row.get('best_epoch',0)):>7}")
    print("="*80)

print_hall_of_fame(all_experiment_results)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CELL 16 — GradCAM & GradCAM++ Universal (Zero Hardcode Architecture)   ║
# ║                                                                          ║
# ║  Strategi: Pure Auto-Detect — tidak ada if/else per nama model.         ║
# ║  Bekerja otomatis untuk arsitektur apapun yang punya Conv2d,            ║
# ║  termasuk yang belum ada saat kode ini ditulis.                         ║
# ║                                                                          ║
# ║  Prinsip deteksi target layer:                                           ║
# ║    1. Kumpulkan SEMUA Conv2d di seluruh model via named_modules()       ║
# ║    2. Filter: buang layer milik classifier/head (fc, logit, score)      ║
# ║    3. Pilih Conv2d terakhir yang punya output spatial (H×W > 1×1)       ║
# ║    4. Validasi: pastikan layer bisa menghasilkan gradient                ║
# ║    5. Fallback bertingkat jika validasi gagal                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# FIX: Import duplikat dihapus (os, glob, gc, warnings, np, cv2, torch, nn, tqdm, plt
# sudah diimport di Cell 2). Hanya import baru yang tidak ada di Cell 2 yang dipertahankan.
import warnings
warnings.filterwarnings("ignore")

warnings.filterwarnings("ignore")


# ════════════════════════════════════════════════════════════════════════════
# BAGIAN 1 — AUTO-DETECT TARGET LAYER
# ════════════════════════════════════════════════════════════════════════════

# Kata kunci yang menandai layer milik classifier/head — bukan feature map
_HEAD_KEYWORDS = (
    "classifier", "fc", "head", "logit", "score",
    "linear", "out", "pred", "output_layer",
)

def _is_head_layer(name: str) -> bool:
    """True jika path nama modul mengandung keyword classifier/head."""
    n = name.lower()
    return any(kw in n for kw in _HEAD_KEYWORDS)


def _get_output_spatial_size(layer: nn.Module, img_size: int, device) -> tuple:
    """
    Jalankan forward pass dummy untuk mengukur spatial output (H, W).
    Mengembalikan (H, W) atau (0, 0) jika gagal.
    """
    dummy = torch.zeros(1, layer.in_channels, img_size, img_size,
                        device=device, requires_grad=False)
    try:
        with torch.no_grad():
            out = layer(dummy)
        return out.shape[-2], out.shape[-1]
    except Exception:
        return 0, 0


def _collect_all_convs(model: nn.Module) -> list:
    """
    Kembalikan list of (name, module) untuk SEMUA Conv2d di dalam model,
    urutan sesuai forward pass (named_modules() berjalan DFS pre-order).
    """
    return [
        (name, module)
        for name, module in model.named_modules()
        if isinstance(module, nn.Conv2d)
    ]


def _probe_layer_gradient(model: nn.Module, layer: nn.Module,
                           img_size: int, num_classes: int, device) -> bool:
    """
    Cek apakah layer menghasilkan gradient saat backward pass.
    Ini memastikan GradCAM bisa bekerja pada layer tersebut.
    """
    handles = []
    grad_received = [False]

    def hook(grad):
        grad_received[0] = True

    try:
        dummy = torch.zeros(1, 3, img_size, img_size,
                            device=device, requires_grad=True)
        # Register forward hook untuk menangkap aktivasi
        activation = {}
        def fwd_hook(mod, inp, out):
            activation['feat'] = out
            out.retain_grad()

        h = layer.register_forward_hook(fwd_hook)
        handles.append(h)

        model.train()   # perlu train mode agar gradient mengalir
        out = model(dummy)
        loss = out[:, 0].sum()
        loss.backward()

        feat = activation.get('feat')
        result = (feat is not None and feat.grad is not None)
        return result

    except Exception:
        return False
    finally:
        model.eval()
        for h in handles:
            h.remove()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def get_target_layers(model: nn.Module, img_size: int = 224,
                      verbose: bool = True) -> list:
    """
    Auto-detect target layer untuk GradCAM tanpa hardcode nama arsitektur.

    Algoritma (4 langkah berurutan, berhenti di langkah pertama yang berhasil):

    Langkah 1 — Seleksi Kandidat
        Kumpulkan semua Conv2d, buang yang ada di path classifier/head.

    Langkah 2 — Pilih berdasarkan posisi
        Ambil Conv2d terakhir (paling dekat output) yang bukan head.
        Ini sudah benar untuk ~95% arsitektur CNN.

    Langkah 3 — Validasi gradient (opsional, aktifkan verbose)
        Pastikan layer kandidat bisa menghasilkan gradient.
        Jika tidak, mundur ke Conv2d sebelumnya.

    Langkah 4 — Emergency fallback
        Jika semua kandidat gagal gradient, pakai Conv2d terakhir
        tanpa filter (termasuk yang mungkin di head).

    Parameters
    ----------
    model    : nn.Module — model yang sudah di-build dan .eval()
    img_size : int       — ukuran input gambar (dari cfg["img_size"])
    verbose  : bool      — cetak info layer yang dipilih

    Returns
    -------
    List[nn.Module]  — list berisi 1 layer (GradCAM expects a list)
    """
    device = next(model.parameters()).device
    num_classes = None
    # Deteksi num_classes dari layer linear terakhir
    for _, m in reversed(list(model.named_modules())):
        if isinstance(m, nn.Linear):
            num_classes = m.out_features
            break
    num_classes = num_classes or 5

    # ── Langkah 1: Kumpulkan semua Conv2d bukan head ──────────────────────
    all_convs = _collect_all_convs(model)
    candidate_convs = [
        (name, mod) for name, mod in all_convs
        if not _is_head_layer(name)
    ]

    if not candidate_convs:
        # Tidak ada Conv2d sama sekali (misal pure transformer)
        # → coba cari LayerNorm terakhir untuk ViT/Swin
        layer_norms = [
            (name, mod) for name, mod in model.named_modules()
            if isinstance(mod, nn.LayerNorm)
        ]
        if layer_norms:
            chosen_name, chosen_layer = layer_norms[-1]
            if verbose:
                LOG.info(f"[AutoCAM] Tidak ada Conv2d — pakai LayerNorm: {chosen_name}")
            return [chosen_layer]
        raise RuntimeError(
            "Model tidak memiliki Conv2d maupun LayerNorm. "
            "Tidak bisa menentukan target layer untuk GradCAM."
        )

    # ── Langkah 2: Mulai dari Conv2d terakhir (non-head) ──────────────────
    # Iterasi mundur: dari posisi paling dekat output
    model.eval()
    chosen_name, chosen_layer = None, None

    for name, mod in reversed(candidate_convs):
        # Validasi ringan: skip Conv2d dengan kernel 1×1 yang merupakan
        # projection/squeeze layer (kurang informatif secara spasial),
        # KECUALI itu satu-satunya pilihan
        if mod.kernel_size == (1, 1) and len(candidate_convs) > 1:
            continue
        chosen_name, chosen_layer = name, mod
        break

    # Jika semua 1×1, ambil yang terakhir saja
    if chosen_layer is None:
        chosen_name, chosen_layer = candidate_convs[-1]

    # ── Langkah 3: Validasi gradient ──────────────────────────────────────
    grad_ok = _probe_layer_gradient(model, chosen_layer,
                                    img_size, num_classes, device)
    model.eval()

    if not grad_ok and verbose:
        LOG.warn(
            f"[AutoCAM] Layer '{chosen_name}' tidak menghasilkan gradient. "
            f"Mencoba layer sebelumnya..."
        )
        # Mundur satu per satu
        idx = next(
            (i for i, (n, _) in enumerate(candidate_convs) if n == chosen_name),
            len(candidate_convs) - 1
        )
        for fallback_name, fallback_layer in reversed(candidate_convs[:idx]):
            grad_ok = _probe_layer_gradient(
                model, fallback_layer, img_size, num_classes, device
            )
            model.eval()
            if grad_ok:
                chosen_name, chosen_layer = fallback_name, fallback_layer
                break

    # ── Langkah 4: Emergency fallback ─────────────────────────────────────
    if not grad_ok:
        # Pakai Conv2d paling akhir tanpa filter apapun
        chosen_name, chosen_layer = all_convs[-1]
        if verbose:
            LOG.warn(
                f"[AutoCAM] Emergency fallback → last Conv2d: {chosen_name}"
            )

    if verbose:
        LOG.info(
            f"[AutoCAM] Target layer  : {chosen_name}\n"
            f"          Type          : {type(chosen_layer).__name__}\n"
            f"          Out channels  : {chosen_layer.out_channels}\n"
            f"          Kernel size   : {chosen_layer.kernel_size}\n"
            f"          Gradient OK   : {grad_ok}"
        )

    return [chosen_layer]


# ════════════════════════════════════════════════════════════════════════════
# BAGIAN 2 — RESHAPE TRANSFORM (untuk ViT / Swin Transformer)
# ════════════════════════════════════════════════════════════════════════════

def get_reshape_transform(model: nn.Module) -> object:
    """
    Deteksi otomatis apakah model butuh reshape transform.

    Cara deteksi: cek apakah model memiliki atribut yang khas transformer
    (patch_embed, cls_token, blocks dengan attn). Jika ya, buat reshape fn.
    Jika tidak (CNN biasa), kembalikan None.
    """
    has_patch_embed = hasattr(model, "patch_embed")
    has_cls_token   = hasattr(model, "cls_token")

    if not (has_patch_embed or has_cls_token):
        return None   # CNN biasa — tidak perlu reshape

    # Deteksi jenis transformer dari atribut
    has_cls = hasattr(model, "cls_token") and model.cls_token is not None

    def _reshape_vit(tensor):
        """ViT: output shape (B, L+1, C) → (B, C, H, W)"""
        result = tensor[:, 1:, :]           # buang CLS token
        B, L, C = result.shape
        H = W = int(L ** 0.5)
        return result.transpose(1, 2).reshape(B, C, H, W)

    def _reshape_swin(tensor):
        """Swin: output shape (B, L, C) → (B, C, H, W)"""
        B, L, C = tensor.shape
        H = W = int(L ** 0.5)
        return tensor.transpose(1, 2).reshape(B, C, H, W)

    if has_cls:
        LOG.info("[AutoCAM] Transformer terdeteksi: ViT — reshape transform aktif")
        return _reshape_vit
    else:
        LOG.info("[AutoCAM] Transformer terdeteksi: Swin — reshape transform aktif")
        return _reshape_swin


# ════════════════════════════════════════════════════════════════════════════
# BAGIAN 3 — PREPROCESSING GAMBAR
# ════════════════════════════════════════════════════════════════════════════

def preprocess_for_gradcam(img_path: str, img_size: int, cfg: dict):
    """
    Load + preprocess gambar untuk GradCAM.

    Menggunakan load_image() dari Cell 4 agar CLAHE / pipeline
    konsisten dengan proses training.

    Returns
    -------
    input_tensor : torch.Tensor  (1, 3, H, W)  — normalized, siap masuk model
    rgb_float    : np.ndarray    (H, W, 3)      — float32 [0,1] untuk overlay
    """
    # Gunakan pipeline preprocessing yang sama dengan training
    img_rgb = load_image(img_path, use_clahe=cfg.get("use_clahe", True), cfg=cfg)
    img_resized = cv2.resize(img_rgb, (img_size, img_size))

    rgb_float = img_resized.astype(np.float32) / 255.0

    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    normalized = (rgb_float - mean) / std

    input_tensor = (
        torch.from_numpy(normalized.transpose(2, 0, 1))
        .unsqueeze(0).float()
    )
    return input_tensor, rgb_float


# ════════════════════════════════════════════════════════════════════════════
# BAGIAN 4 — CORE GRADCAM ENGINE
# ════════════════════════════════════════════════════════════════════════════

def compute_cam(
    model,
    input_tensor   : torch.Tensor,
    rgb_float      : np.ndarray,
    target_layers  : list,
    target_class   : int  = None,
    use_plusplus   : bool = False,
    reshape_transform    = None,
    aug_smooth     : bool = False,
    eigen_smooth   : bool = False,
):
    """
    Jalankan GradCAM atau GradCAM++ dan hasilkan overlay + raw map.

    Parameters
    ----------
    aug_smooth   : rata-rata CAM dari beberapa augmentasi ringan
                   (sedikit lebih robust, sedikit lebih lambat)
    eigen_smooth : kurangi noise dengan dekomposisi eigen pada feature map

    Returns
    -------
    cam_overlay  : np.ndarray (H, W, 3) uint8   — heatmap di atas gambar asli
    raw_cam      : np.ndarray (H, W)    float32  — CAM murni [0, 1]
    pred_class   : int
    pred_conf    : float
    all_probs    : np.ndarray (num_classes,)
    """
    CAMClass = GradCAMPlusPlus if use_plusplus else GradCAM

    input_gpu = input_tensor.to(DEVICE)

    with CAMClass(
        model             = model,
        target_layers     = target_layers,
        reshape_transform = reshape_transform,
    ) as cam_obj:

        targets = (
            [ClassifierOutputTarget(target_class)]
            if target_class is not None else None
        )

        raw_cam = cam_obj(
            input_tensor  = input_gpu,
            targets       = targets,
            aug_smooth    = aug_smooth,
            eigen_smooth  = eigen_smooth,
        )[0]   # (H, W)

    # Prediksi
    model.eval()
    with torch.no_grad():
        logits    = model(input_gpu)
        all_probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

    pred_class = int(np.argmax(all_probs))
    pred_conf  = float(all_probs[pred_class])

    # Resize raw_cam ke ukuran rgb_float jika berbeda (misal transformer)
    H, W = rgb_float.shape[:2]
    if raw_cam.shape != (H, W):
        raw_cam = cv2.resize(raw_cam, (W, H))

    cam_overlay = show_cam_on_image(rgb_float, raw_cam, use_rgb=True)
    return cam_overlay, raw_cam, pred_class, pred_conf, all_probs


# ════════════════════════════════════════════════════════════════════════════
# BAGIAN 5 — HELPER VISUAL
# ════════════════════════════════════════════════════════════════════════════

def _colorize(raw: np.ndarray, cmap: str = "jet") -> np.ndarray:
    """Konversi raw CAM [0,1] ke (H,W,3) uint8 dengan colormap."""
    norm = (raw - raw.min()) / (raw.max() - raw.min() + 1e-8)
    colored = plt.get_cmap(cmap)(norm)[:, :, :3]  # FIX: mpl_cm.get_cmap deprecated sejak Matplotlib 3.7
    return (colored * 255).astype(np.uint8)


def _confidence_bar(probs: np.ndarray, ax, title: str = ""):
    """Bar chart horizontal confidence per kelas."""
    colors = [PALETTE[i % len(PALETTE)] for i in range(len(CLASS_NAMES))]
    ax.barh(CLASS_NAMES, probs, color=colors, edgecolor="white", height=0.6)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Confidence")
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.3)
    for i, p in enumerate(probs):
        ax.text(min(p + 0.02, 0.95), i, f"{p:.1%}", va="center", fontsize=8)


# ════════════════════════════════════════════════════════════════════════════
# BAGIAN 6 — VISUALISASI: SIDE-BY-SIDE COMPARISON (GradCAM vs GradCAM++)
# ════════════════════════════════════════════════════════════════════════════

def visualize_gradcam_comparison(
    model,
    cfg          : dict,
    target_layers: list,
    n_samples    : int  = 6,
    split        : str  = "test",
    reshape_transform   = None,
    aug_smooth   : bool = False,
    save_dir     : str  = None,
):
    """
    Layout per baris (1 gambar, 7 panel):
      [Original] [GradCAM] [GradCAM++] [HeatV1] [HeatV2] [Diff|V2-V1|] [Confidence]
    """
    save_dir  = save_dir or OUTPUT_DIR
    img_size  = cfg["img_size"]
    run_name  = cfg["run_name"]

    import pandas as pd
    df      = pd.read_csv(cfg[f"{split}_csv"])
    img_dir = cfg[f"{split}_dir"]

    sampled  = df.sample(n=min(n_samples, len(df)), random_state=77)
    n_actual = len(sampled)
    n_cols   = 7   # [Orig][V1][V2][H1][H2][Diff][Conf]

    fig, axes = plt.subplots(
        n_actual, n_cols,
        figsize=(n_cols * 3.6, n_actual * 3.8),
        gridspec_kw={"width_ratios": [1, 1, 1, 1, 1, 1, 0.8]}
    )
    if n_actual == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(
        f"GradCAM vs GradCAM++ — {run_name} | {split.title()} Split",
        fontsize=13, fontweight="bold", y=1.01
    )

    col_headers = [
        "Original", "GradCAM", "GradCAM++",
        "Heatmap (V1)", "Heatmap (V2)", "Diff |V2-V1|", "Confidence"
    ]
    for col, h in enumerate(col_headers):
        axes[0, col].set_title(h, fontsize=9, fontweight="bold")

    for row_idx, (_, row_data) in enumerate(
            tqdm(sampled.iterrows(), total=n_actual, desc="Comparison")):

        img_path   = os.path.join(img_dir, row_data["id_code"] + ".png")
        true_label = int(row_data["diagnosis"])

        try:
            input_tensor, rgb_float = preprocess_for_gradcam(img_path, img_size, cfg)

            # GradCAM (V1)
            ov1, raw1, pred_cls, pred_conf, probs = compute_cam(
                model, input_tensor, rgb_float, target_layers,
                use_plusplus=False, reshape_transform=reshape_transform,
                aug_smooth=aug_smooth,
            )

            # GradCAM++ (V2) — paksa kelas sama agar diff bermakna
            ov2, raw2, _, _, _ = compute_cam(
                model, input_tensor, rgb_float, target_layers,
                target_class=pred_cls, use_plusplus=True,
                reshape_transform=reshape_transform,
                aug_smooth=aug_smooth,
            )

            def _norm(r):
                return (r - r.min()) / (r.max() - r.min() + 1e-8)

            diff = np.abs(_norm(raw2) - _norm(raw1))

            panels = [
                (rgb_float * 255).astype(np.uint8),
                ov1, ov2,
                _colorize(_norm(raw1), "jet"),
                _colorize(_norm(raw2), "jet"),
                _colorize(diff, "inferno"),
                None,   # confidence bar — ditangani terpisah
            ]

            correct = pred_cls == true_label
            y_label = (
                f"True: {CLASS_NAMES[true_label]}\n"
                f"Pred: {CLASS_NAMES[pred_cls]}\n"
                f"Conf: {pred_conf:.1%}"
            )

            for col, panel in enumerate(panels):
                ax = axes[row_idx, col]
                if panel is None:
                    _confidence_bar(probs, ax, title="Per-Class Prob")
                else:
                    ax.imshow(panel)
                    ax.axis("off")

            axes[row_idx, 0].set_ylabel(
                y_label, fontsize=8,
                color="green" if correct else "red",
                rotation=0, labelpad=130, va="center"
            )

        except Exception as e:
            LOG.warn(f"Baris {row_idx} gagal: {e}")
            for col in range(n_cols):
                axes[row_idx, col].axis("off")

    plt.tight_layout()
    fname = f"gradcam_comparison_{split}_{run_name}.png"
    plt.savefig(os.path.join(save_dir, fname), dpi=150, bbox_inches="tight")
    plt.show(); plt.close("all")
    LOG.info(f"Comparison disimpan → {fname}")


# ════════════════════════════════════════════════════════════════════════════
# BAGIAN 7 — VISUALISASI: GRID PER KELAS
# ════════════════════════════════════════════════════════════════════════════

def visualize_gradcam_grid(
    model,
    cfg          : dict,
    target_layers: list,
    n_per_class  : int  = 2,
    split        : str  = "test",
    use_plusplus : bool = False,
    reshape_transform   = None,
    save_dir     : str  = None,
):
    """
    Grid n_per_class × len(CLASS_NAMES) sampel.
    Tiap baris: [Original | CAM Overlay | Heatmap Jet | Heatmap Hot]
    """
    save_dir  = save_dir or OUTPUT_DIR
    img_size  = cfg["img_size"]
    run_name  = cfg["run_name"]
    method    = "GradCAMpp" if use_plusplus else "GradCAM"

    import pandas as pd
    df      = pd.read_csv(cfg[f"{split}_csv"])
    img_dir = cfg[f"{split}_dir"]

    samples = []
    for cls_id in range(len(CLASS_NAMES)):
        rows = df[df["diagnosis"] == cls_id]
        sel  = rows.sample(n=min(n_per_class, len(rows)), random_state=42)
        for _, r in sel.iterrows():
            p = os.path.join(img_dir, r["id_code"] + ".png")
            if os.path.exists(p):
                samples.append((p, cls_id))

    if not samples:
        LOG.warn("Tidak ada gambar untuk GradCAM grid."); return

    n_rows = len(samples)
    n_cols = 4   # [Orig][Overlay][Heatmap Jet][Heatmap Hot]

    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(n_cols * 4, n_rows * 3.8))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(
        f"{method} Grid — {run_name} | {split.title()}",
        fontsize=13, fontweight="bold"
    )
    for col, h in enumerate(["Original", f"{method} Overlay",
                              "Heatmap (Jet)", "Heatmap (Hot)"]):
        axes[0, col].set_title(h, fontsize=10, fontweight="bold")

    for row_idx, (img_path, true_label) in enumerate(
            tqdm(samples, desc=f"{method} Grid")):
        try:
            input_tensor, rgb_float = preprocess_for_gradcam(img_path, img_size, cfg)
            overlay, raw, pred_cls, pred_conf, _ = compute_cam(
                model, input_tensor, rgb_float, target_layers,
                use_plusplus=use_plusplus,
                reshape_transform=reshape_transform,
            )

            def _norm(r):
                return (r - r.min()) / (r.max() - r.min() + 1e-8)

            panels = [
                (rgb_float * 255).astype(np.uint8),
                overlay,
                _colorize(_norm(raw), "jet"),
                _colorize(_norm(raw), "hot"),
            ]

            correct = pred_cls == true_label
            for col, panel in enumerate(panels):
                axes[row_idx, col].imshow(panel)
                axes[row_idx, col].axis("off")

            axes[row_idx, 0].set_ylabel(
                f"True: {CLASS_NAMES[true_label]}\n"
                f"Pred: {CLASS_NAMES[pred_cls]} ({pred_conf:.1%})",
                fontsize=8,
                color="green" if correct else "red",
                rotation=0, labelpad=110, va="center"
            )

        except Exception as e:
            LOG.warn(f"Row {row_idx} gagal: {e}")
            for col in range(n_cols):
                axes[row_idx, col].axis("off")

    plt.tight_layout()
    fname = f"gradcam_{method}_grid_{split}_{run_name}.png"
    plt.savefig(os.path.join(save_dir, fname), dpi=150, bbox_inches="tight")
    plt.show(); plt.close("all")
    LOG.info(f"Grid disimpan → {fname}")


# ════════════════════════════════════════════════════════════════════════════
# BAGIAN 8 — VISUALISASI: MEAN CAM PER KELAS
# ════════════════════════════════════════════════════════════════════════════

def plot_mean_cam_per_class(
    model,
    cfg          : dict,
    target_layers: list,
    n_per_class  : int  = 8,
    split        : str  = "test",
    use_plusplus : bool = True,
    reshape_transform   = None,
    save_dir     : str  = None,
):
    """
    Rata-rata CAM seluruh sampel per kelas → visualisasi region penting per grade.

    Layout 3 baris:
      Baris 0 : Mean heatmap per kelas (Jet)
      Baris 1 : Histogram distribusi aktivasi
      Baris 2 : Mean heatmap superimposed pada mean gambar asli
    """
    save_dir  = save_dir or OUTPUT_DIR
    img_size  = cfg["img_size"]
    run_name  = cfg["run_name"]
    method    = "GradCAM++" if use_plusplus else "GradCAM"

    import pandas as pd
    df      = pd.read_csv(cfg[f"{split}_csv"])
    img_dir = cfg[f"{split}_dir"]

    n_cls   = len(CLASS_NAMES)
    n_rows  = 3

    fig, axes = plt.subplots(n_rows, n_cls,
                              figsize=(n_cls * 4, n_rows * 4))
    fig.suptitle(
        f"Mean {method} per Kelas DR — {run_name}",
        fontsize=14, fontweight="bold"
    )

    for cls_id in tqdm(range(n_cls), desc="Mean CAM"):
        cls_rows = df[df["diagnosis"] == cls_id]
        selected = cls_rows.sample(
            n=min(n_per_class, len(cls_rows)), random_state=42
        )

        cam_stack  = []
        img_stack  = []

        for _, row_data in selected.iterrows():
            img_path = os.path.join(img_dir, row_data["id_code"] + ".png")
            if not os.path.exists(img_path):
                continue
            try:
                input_tensor, rgb_float = preprocess_for_gradcam(
                    img_path, img_size, cfg
                )
                _, raw, _, _, _ = compute_cam(
                    model, input_tensor, rgb_float, target_layers,
                    target_class=cls_id, use_plusplus=use_plusplus,
                    reshape_transform=reshape_transform,
                )
                norm = (raw - raw.min()) / (raw.max() - raw.min() + 1e-8)
                cam_stack.append(norm)
                img_stack.append(rgb_float)
            except Exception as e:
                LOG.warn(f"Gagal {img_path}: {e}")

        if not cam_stack:
            for r in range(n_rows):
                axes[r, cls_id].axis("off")
            continue

        mean_cam = np.mean(cam_stack, axis=0)     # (H, W)
        mean_img = np.mean(img_stack, axis=0)     # (H, W, 3)

        # Baris 0: heatmap mean (jet)
        im = axes[0, cls_id].imshow(mean_cam, cmap="jet", vmin=0, vmax=1)
        axes[0, cls_id].set_title(
            CLASS_NAMES[cls_id], fontsize=11, fontweight="bold",
            color=PALETTE[cls_id]
        )
        axes[0, cls_id].axis("off")
        plt.colorbar(im, ax=axes[0, cls_id], fraction=0.046, pad=0.04)

        # Baris 1: histogram distribusi
        axes[1, cls_id].hist(
            mean_cam.flatten(), bins=40,
            color=PALETTE[cls_id], alpha=0.85, edgecolor="white"
        )
        axes[1, cls_id].set_xlabel("Intensitas CAM", fontsize=8)
        axes[1, cls_id].set_ylabel("Frekuensi",       fontsize=8)
        mean_val = mean_cam.mean()
        axes[1, cls_id].axvline(mean_val, color="black", lw=1.5,
                                 linestyle="--", label=f"μ={mean_val:.2f}")
        axes[1, cls_id].legend(fontsize=8)
        axes[1, cls_id].grid(alpha=0.3)

        # Baris 2: overlay pada mean gambar asli
        cam_on_mean = show_cam_on_image(
            mean_img.astype(np.float32), mean_cam, use_rgb=True
        )
        axes[2, cls_id].imshow(cam_on_mean)
        axes[2, cls_id].axis("off")
        axes[2, cls_id].set_title("Mean Overlay", fontsize=8)

    plt.tight_layout()
    fname = f"mean_cam_perclass_{split}_{run_name}.png"
    plt.savefig(os.path.join(save_dir, fname), dpi=150, bbox_inches="tight")
    plt.show(); plt.close("all")
    LOG.info(f"Mean CAM per kelas → {fname}")


# ════════════════════════════════════════════════════════════════════════════
# BAGIAN 9 — VISUALISASI: MISCLASSIFIED GRADCAM
# ════════════════════════════════════════════════════════════════════════════

def visualize_misclassified_gradcam(
    model,
    cfg          : dict,
    target_layers: list,
    n_samples    : int  = 8,
    split        : str  = "test",
    reshape_transform   = None,
    save_dir     : str  = None,
):
    """
    Khusus menampilkan GradCAM++ pada sampel yang salah diprediksi.
    Berguna untuk memahami apa yang membingungkan model.

    Layout per baris: [Original] [GradCAM++] [Conf True Class] [Conf Pred Class]
    """
    save_dir  = save_dir or OUTPUT_DIR
    img_size  = cfg["img_size"]
    run_name  = cfg["run_name"]

    import pandas as pd
    df      = pd.read_csv(cfg[f"{split}_csv"])
    img_dir = cfg[f"{split}_dir"]

    # Kumpulkan misclassified samples (limit ke n_samples pertama)
    misclassified = []
    for _, row_data in tqdm(df.iterrows(), total=len(df), desc="Scan misclassified"):
        if len(misclassified) >= n_samples:
            break
        img_path = os.path.join(img_dir, row_data["id_code"] + ".png")
        if not os.path.exists(img_path):
            continue
        try:
            input_tensor, rgb_float = preprocess_for_gradcam(img_path, img_size, cfg)
            _, _, pred_cls, _, probs = compute_cam(
                model, input_tensor, rgb_float, target_layers,
                use_plusplus=True, reshape_transform=reshape_transform,
            )
            true_label = int(row_data["diagnosis"])
            if pred_cls != true_label:
                misclassified.append({
                    "img_path"  : img_path,
                    "true_label": true_label,
                    "pred_cls"  : pred_cls,
                    "probs"     : probs,
                })
        except Exception:
            continue

    if not misclassified:
        LOG.info("Tidak ada misclassified sample ditemukan (akurasi 100%?).")
        return

    n_actual = len(misclassified)
    n_cols   = 5   # [Orig][GradCAM++][HeatJet][ConfTrue][ConfPred]

    fig, axes = plt.subplots(
        n_actual, n_cols,
        figsize=(n_cols * 3.5, n_actual * 3.8),
        gridspec_kw={"width_ratios": [1, 1, 1, 0.7, 0.7]}
    )
    if n_actual == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(
        f"Misclassified GradCAM++ — {run_name} | {split.title()}",
        fontsize=13, fontweight="bold"
    )
    for col, h in enumerate(["Original", "GradCAM++", "Heatmap",
                              "True Class\nConf", "Pred Class\nConf"]):
        axes[0, col].set_title(h, fontsize=9, fontweight="bold")

    for row_idx, info in enumerate(misclassified):
        img_path   = info["img_path"]
        true_label = info["true_label"]
        pred_cls   = info["pred_cls"]
        probs      = info["probs"]

        try:
            input_tensor, rgb_float = preprocess_for_gradcam(img_path, img_size, cfg)

            # GradCAM++ targeting TRUE class (apa yang model HARUSNYA lihat)
            ov_true, raw_true, _, _, _ = compute_cam(
                model, input_tensor, rgb_float, target_layers,
                target_class=true_label, use_plusplus=True,
                reshape_transform=reshape_transform,
            )

            # GradCAM++ targeting PRED class (apa yang model SALAH lihat)
            ov_pred, raw_pred, _, _, _ = compute_cam(
                model, input_tensor, rgb_float, target_layers,
                target_class=pred_cls, use_plusplus=True,
                reshape_transform=reshape_transform,
            )

            def _norm(r):
                return (r - r.min()) / (r.max() - r.min() + 1e-8)

            heat_true = _colorize(_norm(raw_true), "jet")

            # Panel: Orig / GradCAM++ pred / Heatmap true
            for col, img_disp in enumerate([
                (rgb_float * 255).astype(np.uint8),
                ov_pred,
                heat_true,
            ]):
                axes[row_idx, col].imshow(img_disp)
                axes[row_idx, col].axis("off")

            # Panel confidence: kelas true
            _confidence_bar(
                probs, axes[row_idx, 3],
                title=f"True: {CLASS_NAMES[true_label]}\n({probs[true_label]:.1%})"
            )
            axes[row_idx, 3].axhline(
                y=true_label, color="blue", lw=2, linestyle="--"
            )

            # Panel confidence: kelas pred
            _confidence_bar(
                probs, axes[row_idx, 4],
                title=f"Pred: {CLASS_NAMES[pred_cls]}\n({probs[pred_cls]:.1%})"
            )
            axes[row_idx, 4].axhline(
                y=pred_cls, color="red", lw=2, linestyle="--"
            )

            axes[row_idx, 0].set_ylabel(
                f"#{row_idx+1}", fontsize=9, fontweight="bold",
                rotation=0, labelpad=20, va="center"
            )

        except Exception as e:
            LOG.warn(f"Misclassified row {row_idx} gagal: {e}")
            for col in range(n_cols):
                axes[row_idx, col].axis("off")

    plt.tight_layout()
    fname = f"gradcam_misclassified_{split}_{run_name}.png"
    plt.savefig(os.path.join(save_dir, fname), dpi=150, bbox_inches="tight")
    plt.show(); plt.close("all")
    LOG.info(f"Misclassified GradCAM → {fname}")


# ════════════════════════════════════════════════════════════════════════════
# BAGIAN 10 — MAIN RUNNER
# ════════════════════════════════════════════════════════════════════════════

def run_gradcam_visualization(
    n_grid_per_class   : int  = 2,
    n_comparison       : int  = 6,
    n_mean_per_class   : int  = 8,
    n_misclassified    : int  = 8,
    split              : str  = "test",
    aug_smooth         : bool = False,
    save_dir           : str  = None,
):
    """
    Entry point — jalankan semua visualisasi GradCAM secara otomatis.

    Alur:
      1. Scan semua best_*.pth di OUTPUT_DIR
      2. Untuk tiap model:
         a. Load checkpoint + rebuild model
         b. Auto-detect target layer (tanpa hardcode)
         c. Auto-detect reshape transform (CNN vs Transformer)
         d. Jalankan 4 jenis visualisasi
         e. Bersihkan VRAM

    Parameter
    ---------
    n_grid_per_class : sampel per kelas pada GradCAM/GradCAM++ grid
    n_comparison     : gambar pada side-by-side comparison panel
    n_mean_per_class : sampel untuk mean CAM per kelas
    n_misclassified  : sampel salah prediksi untuk analisis khusus
    split            : "test" / "val" / "train"
    save_dir         : direktori simpan gambar (default = OUTPUT_DIR)
    """
    save_dir  = save_dir or OUTPUT_DIR
    pth_files = glob.glob(os.path.join(OUTPUT_DIR, "best_*.pth"))

    if not pth_files:
        LOG.warn(
            "Tidak ada file best_*.pth di OUTPUT_DIR.\n"
            "Jalankan Cell 12 (training) terlebih dahulu."
        )
        return

    LOG.section(
        f"CELL 16 — GradCAM Universal | {len(pth_files)} model | split={split}"
    )

    for pth_path in pth_files:
        run_name = (
            os.path.basename(pth_path)
            .replace("best_", "").replace(".pth", "")
        )
        LOG.info(f"▶ Memproses: {run_name}")

        # ── Load checkpoint ───────────────────────────────────────────────
        try:
            ckpt = torch.load(pth_path, map_location=DEVICE, weights_only=False)
        except Exception as e:
            LOG.warn(f"Gagal load {pth_path}: {e}"); continue

        cfg = ckpt.get("cfg")
        if cfg is None:
            LOG.warn(f"Config tidak ditemukan di {pth_path}. Skip."); continue

        # ── Rebuild model ─────────────────────────────────────────────────
        try:
            eval_model = build_model(cfg)
            clean_sd   = {
                k: v for k, v in ckpt["state_dict"].items()
                if not k.endswith(("total_ops", "total_params"))
            }
            eval_model.load_state_dict(clean_sd, strict=True)
            eval_model.eval().to(DEVICE)
        except Exception as e:
            LOG.warn(f"Gagal build model {run_name}: {e}"); continue

        # ── Auto-detect target layer ──────────────────────────────────────
        try:
            target_layers = get_target_layers(
                eval_model,
                img_size = cfg["img_size"],
                verbose  = True,
            )
        except Exception as e:
            LOG.warn(f"Gagal resolve target layer untuk {run_name}: {e}")
            del eval_model; gc.collect(); torch.cuda.empty_cache(); continue

        # ── Auto-detect reshape transform ─────────────────────────────────
        reshape_fn = get_reshape_transform(eval_model)

        print(f"\n{'═'*65}")
        print(f"  {run_name}")
        print(f"  Arsitektur   : {cfg['model_name']}")
        print(f"  Target Layer : {[type(l).__name__ for l in target_layers]}")
        print(f"  Transformer  : {'Ya' if reshape_fn else 'Tidak'}")
        print(f"{'═'*65}\n")

        # ── (1) GradCAM Grid ──────────────────────────────────────────────
        for use_pp in [False, True]:
            visualize_gradcam_grid(
                model             = eval_model,
                cfg               = cfg,
                target_layers     = target_layers,
                n_per_class       = n_grid_per_class,
                split             = split,
                use_plusplus      = use_pp,
                reshape_transform = reshape_fn,
                save_dir          = save_dir,
            )

        # ── (2) Comparison Panel ──────────────────────────────────────────
        visualize_gradcam_comparison(
            model             = eval_model,
            cfg               = cfg,
            target_layers     = target_layers,
            n_samples         = n_comparison,
            split             = split,
            reshape_transform = reshape_fn,
            aug_smooth        = aug_smooth,
            save_dir          = save_dir,
        )

        # ── (3) Mean CAM per Kelas ────────────────────────────────────────
        plot_mean_cam_per_class(
            model             = eval_model,
            cfg               = cfg,
            target_layers     = target_layers,
            n_per_class       = n_mean_per_class,
            split             = split,
            use_plusplus      = True,
            reshape_transform = reshape_fn,
            save_dir          = save_dir,
        )

        # ── (4) Misclassified Analysis ────────────────────────────────────
        visualize_misclassified_gradcam(
            model             = eval_model,
            cfg               = cfg,
            target_layers     = target_layers,
            n_samples         = n_misclassified,
            split             = split,
            reshape_transform = reshape_fn,
            save_dir          = save_dir,
        )

        # ── Cleanup ───────────────────────────────────────────────────────
        del eval_model
        gc.collect()
        torch.cuda.empty_cache()
        LOG.info(f"✅ Selesai: {run_name}\n")

    LOG.info("CELL 16 selesai — semua visualisasi GradCAM tersimpan.")


# ════════════════════════════════════════════════════════════════════════════
# EKSEKUSI
# ════════════════════════════════════════════════════════════════════════════
# Membaca config secara otomatis dari checkpoint terbaik jika ada
pth_files = glob.glob(os.path.join(OUTPUT_DIR, "best_*.pth"))
cfg_gc = {}
if pth_files:
    try:
        ckpt = torch.load(pth_files[0], map_location=DEVICE, weights_only=False)
        cfg_gc = ckpt.get("cfg", {})
    except Exception:
        pass

run_gradcam_visualization(
    n_grid_per_class = cfg_gc.get("gradcam_grid_per_class", 2),      # jumlah gambar per kelas di grid
    n_comparison     = cfg_gc.get("gradcam_comparison_samples", 6),  # jumlah gambar di panel perbandingan
    n_mean_per_class = cfg_gc.get("gradcam_mean_per_class", 8),      # jumlah gambar untuk rata-rata CAM
    n_misclassified  = cfg_gc.get("gradcam_misclassified_samples", 8), # jumlah sampel salah prediksi untuk analisis
    split            = cfg_gc.get("gradcam_split", "test"),
    aug_smooth       = cfg_gc.get("gradcam_aug_smooth", False),      # True = lebih robust tapi ~3× lebih lambat
)
